In [1]:
# 1 - CREATE - 1_INSTRUCTION_ALL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import glob, os, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '2_INPUT/riscv-unified-db/backends/instructions_appendix/all_instructions.golden.adoc'
FOLDER_INPUT = '2_INPUT/riscv-opcodes/extensions'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/1_INSTRUCTION_ALL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_BLOCK_SORT_KEY(STRING_BLOCK: str):
    MATCH = re.search(r'^REGISTER:\s*([^\n]+)', STRING_BLOCK, flags=re.M)
    STRING_NAME = MATCH.group(1).strip() if MATCH else STRING_BLOCK
    return FUNCTION_NATURAL_SORT_KEY(STRING_NAME)

def FUNCTION_NORMALIZE_SUFFIX_FORM(STRING_TOKEN):
    for EACH_PATTERN_SUFFIX, EACH_STRING_KIND in ((r'([a-z_]*imm)(\d+)(?:sp)?(hi|lo)', None), (r'([a-z_]*imm)(\d+)sp(?:_s)?', 'sp'), (r'([a-z_]*imm)(\d+)(hi|lo)', None)):
        MATCH_SUFFIX = re.fullmatch(EACH_PATTERN_SUFFIX, STRING_TOKEN, re.I)
        if MATCH_SUFFIX:
            return (f'{MATCH_SUFFIX.group(1)}{MATCH_SUFFIX.group(2)}', (EACH_STRING_KIND or MATCH_SUFFIX.group(3).lower()))
    if re.fullmatch(r'[a-z_]*imm\d+$', STRING_TOKEN, re.I):
        return STRING_TOKEN, 'plain'
    return None, None

def FUNCTION_CHOOSE_IMM(LIST_OPERAND):
    LIST_IMM = [EACH_TOKEN for EACH_TOKEN in LIST_OPERAND if REGEX_TOKEN_IMM.fullmatch(EACH_TOKEN)]
    if not LIST_IMM:
        return None
    MAP_GROUP = {}
    for EACH_TOKEN in LIST_IMM:
        STRING_BASE, STRING_KIND = FUNCTION_NORMALIZE_SUFFIX_FORM(EACH_TOKEN)
        if STRING_BASE:
            MAP_GROUP.setdefault(STRING_BASE, set()).add(STRING_KIND)
    for EACH_STRING_PREFERENCE in ('hi lo', 'sp', 'plain'):
        for EACH_STRING_BASE, EACH_SET_KIND in MAP_GROUP.items():
            if all(EACH_PREF in EACH_SET_KIND for EACH_PREF in EACH_STRING_PREFERENCE.split()):
                return EACH_STRING_BASE
    STRING_BASE, _ = FUNCTION_NORMALIZE_SUFFIX_FORM(LIST_IMM[0])
    return STRING_BASE or LIST_IMM[0]

def FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN):
    return [EACH_TOKEN for EACH_TOKEN in LIST_TOKEN if not REGEX_TOKEN_ENCODING.match(EACH_TOKEN) and '=' not in EACH_TOKEN and '..' not in EACH_TOKEN and '::' not in EACH_TOKEN]

def FUNCTION_PROMPT(MAP_IMM, STRING_MNEMONIC, LIST_OPERAND):
    STRING_IMM = FUNCTION_CHOOSE_IMM(LIST_OPERAND)
    if STRING_IMM and STRING_MNEMONIC not in MAP_IMM:
        MAP_IMM[STRING_MNEMONIC] = STRING_IMM
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_TOKEN_IMM = re.compile(r'(?:[a-z_]*imm\d+(?:hi|lo|sphi|splo|sp(?:_s)?)?|zimm(?:\d+)?|shamt\d+)$', re.I)
REGEX_TOKEN_ENCODING = re.compile(r'(?:^\d+\.\.\d+=|^\d+=|^0x[0-9a-f]+$|^[01]+$|^\d+\.\.\d+$)', re.I)
MAP_IMM = {}
LIST_BLOCK_FINAL = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_LINE = FILE_HANDLE.readlines()

LIST_BLOCK = [(f"INSTRUCTION: {EACH_LINE_NEXT.split()[0].upper()}\nFORMAT: " f"{EACH_LINE_NEXT.strip().replace('xd','rd').replace('xs1','rs1').replace('xs2','rs2')}") for EACH_LINE_CURRENT, EACH_LINE_NEXT in zip(LIST_LINE, LIST_LINE[1:]) if EACH_LINE_CURRENT.startswith('Assembly::')]

for EACH_FILE_PATH in (EACH_PATH for EACH_PATH in glob.glob(os.path.join(FOLDER_INPUT, 'rv*')) if os.path.isfile(EACH_PATH)):
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        for EACH_RAW_LINE in FILE_HANDLE:
            STRING_LINE = EACH_RAW_LINE.strip()
            if not STRING_LINE:
                continue
            if '->' in STRING_LINE:
                try:
                    _, STRING_RIGHT = STRING_LINE.split('->', 1)
                    LIST_TOKEN = STRING_RIGHT.split()
                    if LIST_TOKEN:
                        FUNCTION_PROMPT(MAP_IMM, LIST_TOKEN[0].split(':')[-1], FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN[1:]))
                except ValueError:
                    pass
                continue
            LIST_TOKEN = STRING_LINE.split()
            if not LIST_TOKEN:
                continue
            if LIST_TOKEN[0] == '$pseudo_op' and len(LIST_TOKEN) >= 3:
                FUNCTION_PROMPT(MAP_IMM, LIST_TOKEN[2], FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN[3:]))
                continue
            FUNCTION_PROMPT(MAP_IMM, LIST_TOKEN[0].split(':')[-1], FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN[1:]))

for EACH_STRING_BLOCK in LIST_BLOCK:
    STRING_INSTRUCTION = EACH_STRING_BLOCK.splitlines()[0].split(':', 1)[1].strip()
    STRING_FORMAT = EACH_STRING_BLOCK.splitlines()[1].split(':', 1)[1].strip()
    STRING_IMM = MAP_IMM.get(STRING_INSTRUCTION.lower())
    if STRING_IMM:
        STRING_FORMAT = re.sub(r'\bimm\b(?=\()', STRING_IMM, STRING_FORMAT, flags=re.I)
        STRING_FORMAT = re.sub(r'\bimm\b(?!\()', STRING_IMM, STRING_FORMAT, flags=re.I)
    for EACH_PATTERN_LABEL in (r'\bbimm\d+(hi|lo)?\b', r'\bjimm\d+\b', r'\bc_bimm\d+(hi|lo)?\b', r'\bc_imm12\b'):
        STRING_FORMAT = re.sub(EACH_PATTERN_LABEL, 'label', STRING_FORMAT, flags=re.I)
    if STRING_INSTRUCTION.upper() in ('AUIPC', 'LUI'):
        STRING_FORMAT = STRING_FORMAT.replace('imm20', 'uimm20')
    LIST_BLOCK_FINAL.append(f'INSTRUCTION: {STRING_INSTRUCTION.upper()}\nFORMAT: {STRING_FORMAT}')

LIST_BLOCK_FINAL = [EACH_BLOCK.replace('ssamoswap.d rd, rs2, rs1', 'ssamoswap.d rd, rs2, (rs1)').replace('ssamoswap.w rd, rs2, rs1', 'ssamoswap.w rd, rs2, (rs1)') for EACH_BLOCK in LIST_BLOCK_FINAL]

LIST_BLOCK_FINAL.sort(key=FUNCTION_BLOCK_SORT_KEY)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_BLOCK_FINAL))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 1_INSTRUCTION_ALL.txt - END

0.06 second(s)
====
DONE
====


In [2]:
# 2 - CREATE - 2_INSTRUCTION_SOME.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/1_INSTRUCTION_ALL.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/2_INSTRUCTION_SOME.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]
# FUNCTION - FINISH

# VARIABLE - BEGIN
LIST_INSTRUCTION_WANTED = ['FCLASS.D', 'FADD.D', 'FCVT.D.H', 'FMIN.D', 'FSUB.D', 'VAADD.VV', 'VADC.VIM', 'VADD.VI', 'VSUB.VV', 'VAND.VI']
# LIST_INSTRUCTION_WANTED = ['ADDIW']
IS_USING_EVERYTHING = False
# VARIABLE - FINISH

# CODE - BEGIN
LIST_INSTRUCTION_WANTED.sort(key=FUNCTION_NATURAL_SORT_KEY)

if IS_USING_EVERYTHING:
    with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
        STRING_CONTENT = FILE_HANDLE.read().strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_CONTENT)
else:
    with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
        LIST_LINE = FILE_HANDLE.readlines()
    LIST_WANTED = sorted({EACH_INSTRUCTION.upper() for EACH_INSTRUCTION in LIST_INSTRUCTION_WANTED}, key=FUNCTION_NATURAL_SORT_KEY)
    LIST_SELECTED = []
    for EACH_STRING_WANTED in LIST_WANTED:
        for EACH_INDEX_LINE, EACH_STRING_LINE in enumerate(LIST_LINE):
            if EACH_STRING_LINE.strip() == f'INSTRUCTION: {EACH_STRING_WANTED}':
                LIST_SELECTED.append(f'{EACH_STRING_LINE.strip()}\n{LIST_LINE[EACH_INDEX_LINE + 1].strip()}')
                break
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write('\n\n'.join(LIST_SELECTED))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 2 - CREATE - 2_INSTRUCTION_SOME.txt - END

0.01 second(s)
====
DONE
====


In [3]:
# 3 - CREATE - 3_INSTRUCTION_BOOLEAN.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FOLDER_INPUT = '3_SPECIFICATION/2_SPLIT'
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/2_INSTRUCTION_SOME.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/3_INSTRUCTION_BOOLEAN.txt'
FILE_PROMPT = '4_INSTRUCTION/2_HISTORY/1_HISTORY_INSTRUCTION_BOOLEAN.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 16384
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
WIDTH = 120
DIVIDER_LINE = ('#' * WIDTH + '\n') * 3
DIVIDER_LINE = DIVIDER_LINE.rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
LIST_OUTPUT_LINE = []
LIST_HISTORY = []
GLOBAL_NUMBER = 1
IS_LOCAL = False
SET_COMPRESS = {'add', 'addi', 'addi16sp', 'addi4spn', 'addiw', 'addw', 'and', 'andi', 'beqz', 'bnez', 'ebreak', 'fld', 'fldsp', 'flw', 'flwsp', 'fsd', 'fsdsp', 'fsw', 'fswsp', 'j', 'jal', 'jalr', 'jr', 'lbu', 'ld', 'ldsp', 'lh', 'lhu', 'li', 'lui', 'lw', 'lwsp', 'mop.n', 'mul', 'mv', 'nop', 'not', 'ntl.all', 'ntl.p1', 'ntl.pall', 'ntl.s1', 'or', 'sb', 'sd', 'sdsp', 'sext.b', 'sext.h', 'sh', 'slli', 'srai', 'srli', 'sub', 'subw', 'sw', 'swsp', 'xor', 'zext.b', 'zext.h', 'zext.w'}
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_INSTRUCTION = [EACH_LINE.split(': ')[1].strip() for EACH_LINE in FILE_HANDLE if EACH_LINE.startswith('INSTRUCTION: ')]

for EACH_INSTRUCTION in LIST_INSTRUCTION:
    SET_KEY = {EACH_INSTRUCTION, EACH_INSTRUCTION.lower()}
    if EACH_INSTRUCTION.startswith('AMO'):
        SET_KEY.add('AMO')
    elif '.' in EACH_INSTRUCTION and EACH_INSTRUCTION.index('.') >= 3:
        SET_KEY.add(EACH_INSTRUCTION.split('.', 1)[0])
    LIST_PATH = []
    for EACH_FILE_NAME in sorted(os.listdir(FOLDER_INPUT), key=FUNCTION_NATURAL_SORT_KEY):
        if not EACH_FILE_NAME.endswith('.txt'):
            continue
        FILE_PATH = os.path.join(FOLDER_INPUT, EACH_FILE_NAME)
        with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            if any(any(EACH_KEY in EACH_LINE for EACH_KEY in SET_KEY) for EACH_LINE in FILE_HANDLE):
                LIST_PATH.append(FILE_PATH)
    LIST_PATH.sort(key=lambda EACH_PATH_VALUE: int(EACH_PATH_VALUE.split('SPECIFICATION_')[1].split('.txt')[0]))
    LOCAL_NUMBER = 1
    for EACH_FILE_PATH in LIST_PATH:
        with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            STRING_SPEC = FILE_HANDLE.read()
        STRING_EXCEPTION = (f'Remember, {EACH_INSTRUCTION}, not C.{EACH_INSTRUCTION}! ' f'If the text contains any information about C.{EACH_INSTRUCTION}, output NO! ' if EACH_INSTRUCTION.lower() in SET_COMPRESS else '')
        STRING_PROMPT = (f'For RISC-V, does this text below contain any information about ' f'{EACH_INSTRUCTION} instruction? {STRING_EXCEPTION}' f'If the text contains any information about HINT instructions, ' f'output NO! Output YES or NO only.\n\nThe text:\n\n{STRING_SPEC}')
        if IS_LOCAL:
            STRING_RESPONSE = chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
        else:
            STRING_RESPONSE = CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
        STRING_RESPONSE = FUNCTION_CLEAN_TEXT(STRING_RESPONSE)
        STRING_INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {EACH_INSTRUCTION}
{'=' * (WIDTH // 2)}
=======
PROMPT:
=======
{'=' * (WIDTH // 2)}
'''
        STRING_ENDING = f'''
{'=' * (WIDTH // 2)}
=========
RESPONSE:
=========
{'=' * (WIDTH // 2)}
{STRING_RESPONSE}
'''
        LIST_HISTORY.append(f"{f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')}\n" f"{STRING_INTRO}{STRING_PROMPT}{STRING_ENDING}" f"{f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')}{SUFFIX}")
        if 'YES' in STRING_RESPONSE:
            LIST_OUTPUT_LINE.append(f'{EACH_INSTRUCTION} - {EACH_FILE_PATH}')
        GLOBAL_NUMBER += 1
        LOCAL_NUMBER += 1

STRING_PROMPT_HISTORY = ''.join(LIST_HISTORY).removesuffix(SUFFIX)

with open(FILE_PROMPT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n'.join(LIST_OUTPUT_LINE))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 3 - CREATE - 3_INSTRUCTION_BOOLEAN.txt - END

1 minute(s) 8.87 second(s)
====
DONE
====


In [4]:
# 4 - CREATE - 4_INSTRUCTION_RULE.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from collections import defaultdict
from ollama import chat, Client
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/3_INSTRUCTION_BOOLEAN.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/4_INSTRUCTION_RULE.txt'
FILE_PROMPT_1 = '4_INSTRUCTION/2_HISTORY/2_HISTORY_INSTRUCTION_RULE_1.txt'
FILE_PROMPT_2 = '4_INSTRUCTION/2_HISTORY/3_HISTORY_INSTRUCTION_RULE_2.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()

def FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE, STRING_SUFFIX):
    HALF_WIDTH = WIDTH // 2
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + STRING_SUFFIX
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
CONTEXT = 65536
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
LIST_OUTPUT_CONTENT = []
MAP_INSTRUCTION_RULE = defaultdict(list)
LIST_OUTPUT_FINAL = []
LIST_OUTPUT_SUPER_FINAL = []
STRING_PROMPT_HISTORY_1 = ''
STRING_PROMPT_HISTORY_2 = ''
GLOBAL_NUMBER_1 = 1
LOCAL_NUMBER_1 = 1
GLOBAL_NUMBER_2 = 1
LOCAL_NUMBER_2 = 1
INSTRUCTION_INITIAL = None
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_INSTRUCTION, EACH_FILE_PATH in LIST_PAIR:
    if EACH_INSTRUCTION not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_INSTRUCTION)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_INSTRUCTION].append(FILE_HANDLE.read())

for EACH_INSTRUCTION in LIST_REGISTER_ORDER:
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[EACH_INSTRUCTION]).strip()
    LOCAL_COUNTER_1 = 1
    LOCAL_COUNTER_2 = 1
    INSTRUCTION_NAME = EACH_INSTRUCTION
    
    # PROMPT 1 - BEGIN
    STRING_PROMPT_1 = f'''For RISC-V, please extract all possible information about {INSTRUCTION_NAME} instruction from the text below. The information must be only about {INSTRUCTION_NAME} instruction, not other things. Extract ONLY explicit, testable, normative description about architectural behavior.

Please output in paragraphs. If you want, you can include bullet or numbered points in your output. You can also include tables. However, do not give me any introduction in your output. You just need to output the information that you extract.

If your output includes some sentences that refer to some tables or figures, include those tables or figures too.

Remember, output must be ASCII-only text.

If it happens to be there is no information about {INSTRUCTION_NAME} instruction from the text below, output NONE

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

Your description must be very clear and unambiguous. If you want, you can output in bullet points also.

Use the exact terminology and wording used in the text below; do not substitute synonyms or paraphrase architectural terms.

The text below:

{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE_1 = FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT_1, TEMPERATURE, CONTEXT)
    if STRING_RESPONSE_1.strip() == 'NONE':
        continue
    STRING_RESPONSE_1 = FUNCTION_CLEAN_TEXT(STRING_RESPONSE_1)
    STRING_PROMPT_HISTORY_1 += FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER_1, LOCAL_NUMBER_1, INSTRUCTION_NAME, STRING_PROMPT_1, STRING_RESPONSE_1, SUFFIX)
    # PROMPT 1 - FINISH

    # PROMPT 2 - BEGIN
    STRING_PROMPT_2 = f'''For RISC-V, for {INSTRUCTION_NAME} instruction, rewrite the following description as a list of rules. Each rule must explain about {INSTRUCTION_NAME} instruction. Essentially, for each rule, it is about {INSTRUCTION_NAME} instruction, not other things.

Do not lose any information about {INSTRUCTION_NAME} instruction from the description. Read the text or the description carefully. Understand the context and the sentences. Do not lose any important information.

For each rule, do not use any internal references like "According to Table 7," or "see Figure 3." Each rule must be one sentence only. Each rule must stand fully on its own without cross-references. Do not quote or refer to other rules. Essentially, each rule is one idea. Avoid a rule like this: "The instruction being described is {INSTRUCTION_NAME}."

Each rule should become a single testable statement. Each rule should be a separate atomic rule, each expressing only one fact. Each rule needs to be clear and detailed. Extract ONLY explicit, testable, normative rules about architectural behavior. A rule MUST describe behavioral semantics, not naming, encoding, or syntax.

Remember, rules must describe architectural cause -> effect semantics (event -> architectural outcome).

Remember, if a sentence contains multiple distinct behaviors or enumerated cases, split them into multiple rules so each rule expresses only one case.

Exclude purely structural or descriptive properties (such as register width, naming, or existence) unless they imply observable architectural behavior.

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

OUTPUT FORMAT:
+ Each rule must begin with the word "RULE", followed by its number (e.g., "RULE 1: ", "RULE 2: ", "RULE 3: ", ...).
+ Do not give me any introduction.
+ Each rule must contain the word {INSTRUCTION_NAME}.

Remember, output must be ASCII-only text.

The following description:

{STRING_RESPONSE_1}'''
    STRING_RESPONSE_2 = FUNCTION_CLEAN_TEXT(FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT_2, TEMPERATURE, CONTEXT))
    STRING_PROMPT_HISTORY_2 += FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER_2, LOCAL_NUMBER_2, INSTRUCTION_NAME, STRING_PROMPT_2, STRING_RESPONSE_2, SUFFIX)
    # PROMPT 2 - FINISH

    # FINALIZING - BEGIN
    STRING_RESPONSE_FINISH = STRING_RESPONSE_2
    LIST_LINE_RULE = [EACH_LN.strip() for EACH_LN in STRING_RESPONSE_FINISH.splitlines() if EACH_LN.strip()]
    LIST_RULE = []
    for EACH_LINE_RULE in LIST_LINE_RULE:
        if re.match(r'^RULE\s+\d+:', EACH_LINE_RULE, re.IGNORECASE):
            EACH_LINE_RULE = re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_LINE_RULE)
            LIST_RULE.append(EACH_LINE_RULE)
    LIST_LINE_FIXED = []
    for EACH_LINE_RULE in LIST_RULE:
        MATCH_RULE = re.match(r'^\s*RULE\s+(\d+):\s*(.*)', EACH_LINE_RULE)
        if not MATCH_RULE:
            LIST_LINE_FIXED.append(EACH_LINE_RULE)
            continue
        NUMBER_RULE = MATCH_RULE.group(1)
        STRING_REST = MATCH_RULE.group(2)
        STRING_REST = re.sub(rf'^RULE\s+{NUMBER_RULE}:\s*', '', STRING_REST)
        LIST_LINE_FIXED.append(f'RULE {NUMBER_RULE}: {STRING_REST}')
    STRING_CLEANED = '\n'.join(LIST_LINE_FIXED)
    LIST_OUTPUT_CONTENT.append(f'INSTRUCTION: {INSTRUCTION_NAME}\n{STRING_CLEANED}')
    # FINALIZING - FINISH

    # CLEANING UP PROMPT - BEGIN
    GLOBAL_NUMBER_1 += 1
    LOCAL_NUMBER_1 += 1
    GLOBAL_NUMBER_2 += 1
    LOCAL_NUMBER_2 += 1

STRING_PROMPT_HISTORY_1 = STRING_PROMPT_HISTORY_1.removesuffix(SUFFIX)
STRING_PROMPT_HISTORY_2 = STRING_PROMPT_HISTORY_2.removesuffix(SUFFIX)

with open(FILE_PROMPT_1, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY_1)

with open(FILE_PROMPT_2, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY_2)
# CLEANING UP PROMPT - FINISH

for EACH_STRING_CONTENT in LIST_OUTPUT_CONTENT:
    LIST_LINE = EACH_STRING_CONTENT.strip().splitlines()
    STRING_NAME = LIST_LINE[0].split(':', 1)[1].strip()
    MAP_INSTRUCTION_RULE[STRING_NAME].extend(LIST_LINE[1:])

for EACH_STRING_NAME, EACH_LIST_RULE in MAP_INSTRUCTION_RULE.items():
    LIST_LINE_NEW = [f'INSTRUCTION: {EACH_STRING_NAME}']
    for EACH_INDEX_RULE, EACH_RULE_LINE in enumerate(EACH_LIST_RULE, 1):
        STRING_RULE_TEXT = EACH_RULE_LINE.split(':', 1)[1].strip()
        LIST_LINE_NEW.append(f'RULE {EACH_INDEX_RULE}: {STRING_RULE_TEXT}')
    LIST_OUTPUT_FINAL.append('\n'.join(LIST_LINE_NEW))

for EACH_STRING_EACH in LIST_OUTPUT_FINAL:
    LIST_LINE = EACH_STRING_EACH.splitlines()
    STRING_INSTRUCTION_LINE = LIST_LINE[0].strip()
    LIST_RULE = [EACH_LN.strip() for EACH_LN in LIST_LINE[1:] if EACH_LN.strip() and EACH_LN.strip() != 'RULE:']
    LIST_RULE = [re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_LN) for EACH_LN in LIST_RULE]
    LIST_RENUMBERED = []
    for EACH_INDEX_RULE, EACH_RULE_LINE in enumerate(LIST_RULE, 1):
        LIST_RENUMBERED.append(re.sub(r'RULE\s+\d+:', f'RULE {EACH_INDEX_RULE}:', EACH_RULE_LINE))
    LIST_OUTPUT_SUPER_FINAL.append(f'{STRING_INSTRUCTION_LINE}\n' + '\n'.join(LIST_RENUMBERED))

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT_SUPER_FINAL))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 4 - CREATE - 4_INSTRUCTION_RULE.txt - END

1 minute(s) 24.98 second(s)
====
DONE
====


In [5]:
# 5 - CREATE - 5_INSTRUCTION_FINAL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# PATH - BEGIN
FILE_INPUT_NAME = '4_INSTRUCTION/1_OUTPUT/2_INSTRUCTION_SOME.txt'
FILE_INPUT_RULE = '4_INSTRUCTION/1_OUTPUT/4_INSTRUCTION_RULE.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/5_INSTRUCTION_FINAL.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_INPUT_NAME, encoding='utf-8') as FILE_HANDLE:
    LIST_NAME = FILE_HANDLE.readlines()

with open(FILE_INPUT_RULE, encoding='utf-8') as FILE_HANDLE:
    LIST_OUTPUT_LINE = FILE_HANDLE.readlines()

for EACH_INDEX_LINE, EACH_STRING_LINE in enumerate(LIST_NAME):
    if EACH_STRING_LINE.startswith('INSTRUCTION: '):
        try:
            INDEX_MATCH = LIST_OUTPUT_LINE.index(EACH_STRING_LINE)
        except ValueError:
            continue
        LIST_OUTPUT_LINE.insert(INDEX_MATCH + 1, LIST_NAME[EACH_INDEX_LINE + 1].rstrip() + '\n')

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_OUTPUT_LINE))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 5 - CREATE - 5_INSTRUCTION_FINAL.txt - END

0.00 second(s)
====
DONE
====


In [6]:
# 6 - CREATE - 6_INSTRUCTION_TOTAL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/5_INSTRUCTION_FINAL.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/6_INSTRUCTION_TOTAL.txt'
# PATH - FINISH

# VARIABLE - BEGIN
IS_NEED_INIT = (not os.path.exists(FILE_OUTPUT)) or os.path.getsize(FILE_OUTPUT) == 0
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_INPUT = FILE_HANDLE.read()

if IS_NEED_INIT:
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(TEXT_INPUT)
else:
    with open(FILE_OUTPUT, encoding='utf-8') as FILE_HANDLE:
        TEXT_OUTPUT = FILE_HANDLE.read()
    TEXT_COMBINED = TEXT_OUTPUT.strip() + '\n\n' + TEXT_INPUT.strip()
    REGEX_INSTRUCTION = re.compile(r'(?ms)^\s*INSTRUCTION:\s*([^\n]+)\s*\n.*?(?=^\s*INSTRUCTION:\s*[^\n]+\s*\n|\Z)')
    MAP_BY_NAME = {EACH_MATCH.group(1).strip(): EACH_MATCH.group(0) for EACH_MATCH in REGEX_INSTRUCTION.finditer(TEXT_COMBINED)}
    TEXT_FINAL = '\n\n'.join(MAP_BY_NAME[EACH_KEY].strip() for EACH_KEY in sorted(MAP_BY_NAME, key=str.upper)).strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(TEXT_FINAL)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 6 - CREATE - 6_INSTRUCTION_TOTAL.txt - END

0.00 second(s)
====
DONE
====


In [7]:
# 7 - CREATE - 7_INSTRUCTION_SUMMARY.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
from typing import Dict, List, Optional, Tuple
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/7_INSTRUCTION_SUMMARY.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# ALARM PLAY - BEGIN
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
import pygame
import time
pygame.mixer.init()
pygame.mixer.music.load('alarm.mp3')
for _ in range(3):
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.01)
pygame.mixer.quit()
# ALARM PLAY - FINISH

# 7 - CREATE - 7_INSTRUCTION_SUMMARY.txt - END

0.00 second(s)
====
DONE
====


In [8]:
# 1 - CREATE - 1_REGISTER_ALL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FOLDER_INPUT = '2_INPUT/riscv-unified-db/spec/std/isa/csr'
FOLDER_INPUT_OTHER = '2_INPUT/riscv-opcodes'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_BLOCK_SORT_KEY(STRING_BLOCK: str):
    MATCH = re.search(r'^REGISTER:\s*([^\n]+)', STRING_BLOCK, flags=re.M)
    STRING_NAME = MATCH.group(1).strip() if MATCH else STRING_BLOCK
    return FUNCTION_NATURAL_SORT_KEY(STRING_NAME)

def FUNCTION_GET_PRIV_AND_WIDTH(STRING_REGISTER: str):
    if STRING_REGISTER.startswith(('seed', 'ssp')):
        return 'U-mode', 'XLEN'
    STRING_FIRST_CHAR = STRING_REGISTER[0].lower()
    if STRING_FIRST_CHAR == 'h':
        return 'HS-mode', 'SXLEN'
    if STRING_FIRST_CHAR == 'd':
        return 'D-mode', 'MXLEN'
    if STRING_FIRST_CHAR == 'm':
        return 'M-mode', 'MXLEN'
    if STRING_FIRST_CHAR == 's':
        return 'S-mode', 'SXLEN'
    return 'U-mode', 'XLEN'
# FUNCTION - FINISH

# VARIABLE - BEGIN
LIST_OUTPUT = []
SET_REGISTER_YAML = set()
MAP_ENTRY_MISSING = {}
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_PATH, _, EACH_LIST_FILE in os.walk(FOLDER_INPUT):
    for EACH_FILE_NAME in EACH_LIST_FILE:
        if not EACH_FILE_NAME.endswith('.yaml'):
            continue
        FILE_PATH = os.path.join(EACH_PATH, EACH_FILE_NAME)
        with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            LIST_LINE = FILE_HANDLE.readlines()
        STRING_CURRENT = ''
        IS_FIELD = False
        LIST_EXTENSION = []
        LIST_BIT = []
        LIST_LOCATION = []
        INDENT_FIELDS = None
        IS_DEFINED_BY = False
        IS_ALL_OF = False
        HAD_FIELDS = False
        for EACH_LINE in LIST_LINE:
            STRING_LINE_STRIPPED = EACH_LINE.strip()
            if IS_FIELD:
                INDENT_LINE = len(EACH_LINE) - len(EACH_LINE.lstrip(' '))
                if INDENT_FIELDS is not None and INDENT_LINE <= INDENT_FIELDS and STRING_LINE_STRIPPED:
                    IS_FIELD = False
                if IS_FIELD:
                    if (
                        STRING_LINE_STRIPPED.endswith(':')
                        and not STRING_LINE_STRIPPED.startswith('-')
                        and ':' in STRING_LINE_STRIPPED
                        and INDENT_FIELDS is not None
                        and INDENT_LINE == INDENT_FIELDS + 2
                    ):
                        STRING_FIELD_NAME = STRING_LINE_STRIPPED[:-1].strip()
                        if STRING_FIELD_NAME not in ('fields', 'definedBy', 'allOf'):
                            LIST_BIT.append(STRING_FIELD_NAME)
                        continue
                    if STRING_LINE_STRIPPED.startswith(('location:', 'location_rv64:')):
                        LIST_LOCATION.append(EACH_LINE.split(':', 1)[1].strip())
                        continue
                    if STRING_LINE_STRIPPED.startswith(('affectedBy:', 'definedBy:')):
                        LIST_EXTENSION.append(EACH_LINE.split(':', 1)[1].strip())
                        continue
                    continue
            if STRING_LINE_STRIPPED.startswith('name:') and not IS_DEFINED_BY:
                STRING_REGISTER = EACH_LINE.split(':', 1)[1].strip()
                STRING_CURRENT += f'REGISTER: {STRING_REGISTER}\n'
            elif STRING_LINE_STRIPPED.startswith('address:'):
                STRING_CURRENT += f"NUMBER: {EACH_LINE.split(':', 1)[1].strip().lower()}\n"
            elif STRING_LINE_STRIPPED.startswith('priv_mode:'):
                STRING_PRIV = EACH_LINE.split(':', 1)[1].strip()
                if STRING_CURRENT.startswith('REGISTER: h') and STRING_PRIV == 'S':
                    STRING_PRIV = 'HS'
                STRING_CURRENT += f'MODE: {STRING_PRIV}-mode\n'
            elif STRING_LINE_STRIPPED.startswith('length:'):
                STRING_CURRENT += f"WIDTH: {EACH_LINE.split(':', 1)[1].strip()}\n"
            elif STRING_LINE_STRIPPED.startswith('definedBy:'):
                LIST_EXTENSION.append(EACH_LINE.split(':', 1)[1].strip())
                IS_DEFINED_BY = True
            elif IS_DEFINED_BY and not STRING_LINE_STRIPPED.startswith('fields:'):
                if STRING_LINE_STRIPPED.startswith('allOf:'):
                    IS_ALL_OF = True
                elif IS_ALL_OF and STRING_LINE_STRIPPED.startswith('-'):
                    LIST_EXTENSION.append(EACH_LINE.split('-', 1)[1].strip())
                elif STRING_LINE_STRIPPED.startswith('name:'):
                    LIST_EXTENSION.append(EACH_LINE.split('name:', 1)[1].strip())
            elif STRING_LINE_STRIPPED.startswith('fields:'):
                IS_ALL_OF = False
                IS_DEFINED_BY = False
                if STRING_LINE_STRIPPED == 'fields: {}':
                    LIST_EXTENSION = [EACH_EXTENSION for EACH_EXTENSION in LIST_EXTENSION if EACH_EXTENSION]
                    if not LIST_EXTENSION:
                        LIST_EXTENSION.append('-')
                    STRING_CURRENT += 'FIELDS: -\nEXTENSIONS: -'
                    break
                IS_FIELD = True
                HAD_FIELDS = True
                INDENT_FIELDS = len(EACH_LINE) - len(EACH_LINE.lstrip(' '))
        if HAD_FIELDS:
            LIST_BIT_FINAL = []
            if LIST_LOCATION and LIST_LOCATION[0].startswith('0'):
                ITER_PAIR = zip(reversed(LIST_LOCATION), reversed(LIST_BIT))
                for EACH_LOCATION, EACH_NAME in ITER_PAIR:
                    if '-' in EACH_LOCATION:
                        STRING_A, STRING_B = EACH_LOCATION.split('-')
                        LIST_BIT_FINAL.append(f'{STRING_B}..{STRING_A}={EACH_NAME}')
                    else:
                        LIST_BIT_FINAL.append(f'{EACH_LOCATION}..{EACH_LOCATION}={EACH_NAME}')
            else:
                ITER_PAIR = zip(LIST_LOCATION, LIST_BIT)
                for EACH_LOCATION, EACH_NAME in ITER_PAIR:
                    if '-' in EACH_LOCATION:
                        STRING_A, STRING_B = EACH_LOCATION.split('-')
                        LIST_BIT_FINAL.append(f'{STRING_A}..{STRING_B}={EACH_NAME}')
                    else:
                        LIST_BIT_FINAL.append(f'{EACH_LOCATION}..{EACH_LOCATION}={EACH_NAME}')
            if LIST_BIT_FINAL:
                STRING_CURRENT += f"FIELDS: {', '.join(LIST_BIT_FINAL)}\n"
            else:
                STRING_CURRENT += 'FIELDS: -\n'
            if LIST_EXTENSION:
                LIST_EXTENSION = [EACH_EXTENSION for EACH_EXTENSION in LIST_EXTENSION if EACH_EXTENSION]
                LIST_EXTENSION_FLAT = []
                for EACH_EXTENSION in LIST_EXTENSION:
                    if EACH_EXTENSION.startswith('[') and EACH_EXTENSION.endswith(']'):
                        LIST_EXTENSION_FLAT.extend(EACH_PART.strip() for EACH_PART in EACH_EXTENSION[1:-1].split(','))
                    else:
                        LIST_EXTENSION_FLAT.append(EACH_EXTENSION)
                LIST_EXTENSION = sorted(set(EACH_X.split('#')[0].strip() for EACH_X in LIST_EXTENSION_FLAT))
                STRING_CURRENT += 'EXTENSIONS: -'
            else:
                STRING_CURRENT += 'EXTENSIONS: -'
        if STRING_CURRENT:
            if re.search(r'(?m)^\s*MODE:\s*D-mode\s*$', STRING_CURRENT):
                continue
            LIST_OUTPUT.append(STRING_CURRENT)
            for EACH_LINE in STRING_CURRENT.splitlines():
                if EACH_LINE.startswith('REGISTER:'):
                    SET_REGISTER_YAML.add(EACH_LINE.split('REGISTER:', 1)[1].strip())

LIST_FILE_CSV = [EACH_FILE_NAME for EACH_FILE_NAME in os.listdir(FOLDER_INPUT_OTHER) if EACH_FILE_NAME.startswith('csrs') and EACH_FILE_NAME.endswith('.csv')]

for EACH_FILE_NAME in sorted(LIST_FILE_CSV, key=FUNCTION_NATURAL_SORT_KEY):
    FILE_PATH = os.path.join(FOLDER_INPUT_OTHER, EACH_FILE_NAME)
    with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        for EACH_LINE in FILE_HANDLE:
            STRING_LINE = EACH_LINE.strip()
            if not STRING_LINE or STRING_LINE.startswith('#'):
                continue
            LIST_PART = STRING_LINE.split(',')
            if len(LIST_PART) < 2:
                continue
            STRING_HEX = LIST_PART[0].strip().lower()
            STRING_REGISTER = LIST_PART[1].strip().strip('"')
            if STRING_REGISTER not in SET_REGISTER_YAML:
                STRING_PRIV, STRING_WIDTH = FUNCTION_GET_PRIV_AND_WIDTH(STRING_REGISTER)
                if STRING_PRIV == 'D-mode':
                    continue
                MAP_ENTRY_MISSING[STRING_REGISTER] = {'hex': STRING_HEX, 'priv': STRING_PRIV, 'width': STRING_WIDTH}

for EACH_REGISTER in list(MAP_ENTRY_MISSING):
    if EACH_REGISTER.endswith('h'):
        STRING_BASE = EACH_REGISTER[:-1]
        if STRING_BASE in MAP_ENTRY_MISSING:
            MAP_ENTRY_MISSING[STRING_BASE]['width'] = '64'
            MAP_ENTRY_MISSING[EACH_REGISTER]['width'] = '32'

for EACH_REGISTER, EACH_INFO in sorted(MAP_ENTRY_MISSING.items()):
    LIST_OUTPUT.append(f'REGISTER: {EACH_REGISTER}\n' f"NUMBER: {EACH_INFO['hex']}\n" f"MODE: {EACH_INFO['priv']}\n" f"WIDTH: {EACH_INFO['width']}\n" f'FIELDS: -\n' f'EXTENSIONS: -')

LIST_OUTPUT.sort(key=FUNCTION_BLOCK_SORT_KEY)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 1_REGISTER_ALL.txt - END

0.08 second(s)
====
DONE
====


In [9]:
# 2 - CREATE - 2_REGISTER_SOME.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_FIND_V_ALIAS(TEXT_VALUE: str):
    LIST_NAME = [EACH_MATCH.group(1).strip() for EACH_MATCH in re.finditer(r'^REGISTER:\s*([^\n]+)', TEXT_VALUE, flags=re.M)]
    SET_NAME = set(LIST_NAME)
    SET_SEEN = set()
    LIST_OUTPUT = []
    for EACH_NAME in LIST_NAME:
        if (EACH_NAME.startswith('v') and len(EACH_NAME) > 1 and EACH_NAME[1:] in SET_NAME and EACH_NAME not in SET_SEEN):
            LIST_OUTPUT.append(EACH_NAME)
            SET_SEEN.add(EACH_NAME)
    return LIST_OUTPUT
# FUNCTION - FINISH

# VARIABLE - BEGIN
LIST_NOT_IMPLEMENTED = [
    'sireg', 'sireg2', 'sireg3', 'sireg4', 'sireg5', 'sireg6',
    'vsireg', 'vsireg2', 'vsireg3', 'vsireg4', 'vsireg5', 'vsireg6',
    'jvt', 'mnstatus', 'mncause', 'tcontrol',
]
LIST_NOT_IMPLEMENTED += [
    f'hpmcounter{i}' for i in range(19, 32)
] + [
    f'hpmcounter{i}h' for i in range(19, 32)
]
LIST_NOT_IMPLEMENTED += [
    f'mhpmcounter{i}' for i in range(19, 32)
] + [
    f'mhpmcounter{i}h' for i in range(19, 32)
]
LIST_REGISTER_WANTED = ['hpmcounter18', 'mcause', 'mcounteren', 'mcountinhibit', 'mcycle', 'medeleg', 'menvcfg', 'mepc', 'mhartid', 'mhpmcounter18', 'mhpmevent18', 'mideleg', 'minstret', 'mscratch', 'mtvec', 'pmpaddr0', 'pmpcfg0', 'satp', 'scause', 'scounteren', 'scountinhibit', 'sepc', 'sstatus', 'stvec', 'fflags', 'frm', 'fcsr', 'vstart', 'vl', 'vtype', 'vlenb', 'vxrm', 'vxsat', 'vcsr']
# LIST_REGISTER_WANTED = ['tdata1']
IS_USING_EVERYTHING = False
# VARIABLE - FINISH

# CODE - BEGIN
LIST_REGISTER_WANTED = [
    EACH_REGISTER
    for EACH_REGISTER in LIST_REGISTER_WANTED
    if EACH_REGISTER not in LIST_NOT_IMPLEMENTED
]

if not LIST_REGISTER_WANTED:
    LIST_REGISTER_WANTED.append('tdata1')

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_ALL = FILE_HANDLE.read()

SET_V_ALIAS = set(FUNCTION_FIND_V_ALIAS(TEXT_ALL))

for EACH_V_ALIAS in (SET_V_ALIAS & set(LIST_REGISTER_WANTED)):
    LIST_REGISTER_WANTED.append(EACH_V_ALIAS[1:])

LIST_REGISTER_WANTED.sort(key=FUNCTION_NATURAL_SORT_KEY)

LIST_LINE = TEXT_ALL.splitlines(True)
SET_NOT_IMPLEMENTED = {EACH.lower() for EACH in LIST_NOT_IMPLEMENTED}

if IS_USING_EVERYTHING:
    LIST_OUT = []
    EACH_INDEX_LINE = 0
    while EACH_INDEX_LINE < len(LIST_LINE):
        EACH_LINE = LIST_LINE[EACH_INDEX_LINE]
        STRIPPED = EACH_LINE.strip()
        if STRIPPED.startswith('REGISTER:'):
            NAME = STRIPPED.split(':', 1)[1].strip().lower()
            if NAME in SET_NOT_IMPLEMENTED:
                EACH_INDEX_LINE += 1
                while (
                    EACH_INDEX_LINE < len(LIST_LINE)
                    and not LIST_LINE[EACH_INDEX_LINE].strip().startswith('REGISTER:')
                ):
                    EACH_INDEX_LINE += 1
                continue
        LIST_OUT.append(EACH_LINE)
        EACH_INDEX_LINE += 1
    TEXT_FINAL = ''.join(LIST_OUT)
    TEXT_FINAL = re.sub(r'\n{3,}', '\n\n', TEXT_FINAL).strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(TEXT_FINAL)
else:
    LIST_WANTED = sorted({EACH_WANTED.lower() for EACH_WANTED in LIST_REGISTER_WANTED})
    LIST_SELECTED = []
    for EACH_WANTED in LIST_WANTED:
        STRING_TARGET = f'REGISTER: {EACH_WANTED}'
        for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
            if EACH_LINE.strip() == STRING_TARGET:
                LIST_SELECTED.append(''.join(EACH_L.strip() + '\n' for EACH_L in LIST_LINE[EACH_INDEX_LINE:EACH_INDEX_LINE + 6]).rstrip('\n'))
                break
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write('\n\n'.join(LIST_SELECTED))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 2 - CREATE - 2_REGISTER_SOME.txt - END

0.02 second(s)
====
DONE
====


In [11]:
# 3 - CREATE - 3_REGISTER_BOOLEAN.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FOLDER_INPUT = '3_SPECIFICATION/2_SPLIT'
FILE_INPUT = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_PROMPT = '5_REGISTER/2_HISTORY/1_HISTORY_REGISTER_BOOLEAN.txt'
FILE_ALL = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_LLM_CALL(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 16384
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
STRING_PREFIX_REGISTER_DIGIT = ''
LIST_OUTPUT_LINE = []
LIST_HISTORY = []
GLOBAL_COUNTER = 1
SPECIAL_NAME_FOR_NUMBERS = ''
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_REGISTER = [EACH_LINE.split(': ', 1)[1].strip() for EACH_LINE in FILE_HANDLE if EACH_LINE.startswith('REGISTER: ')]

with open(FILE_ALL, encoding='utf-8') as FILE_HANDLE:
    LIST_ALL_LINE = FILE_HANDLE.readlines()

for EACH_REGISTER in LIST_REGISTER:
    LIST_WORD = [EACH_REGISTER]
    if re.search(r'\d', EACH_REGISTER):
        MATCH_PREFIX = re.match(r'^[a-zA-Z]+', EACH_REGISTER)
        if MATCH_PREFIX:
            SPECIAL_NAME_FOR_NUMBERS = MATCH_PREFIX.group(0)
            LIST_WORD.append(SPECIAL_NAME_FOR_NUMBERS)
    elif EACH_REGISTER.endswith('h'):
        STRING_BASE = EACH_REGISTER[:-1]
        if STRING_BASE and STRING_BASE != EACH_REGISTER:
            LIST_WORD.append(STRING_BASE)
    elif EACH_REGISTER.startswith('v'):
        STRING_BASE = EACH_REGISTER[1:]
        if (STRING_BASE and STRING_BASE != EACH_REGISTER and any(f'REGISTER: {STRING_BASE}' in EACH_LINE for EACH_LINE in LIST_ALL_LINE)):
            LIST_WORD.append(STRING_BASE)
    LIST_WORD = list(dict.fromkeys(LIST_WORD))
    LIST_PATH = []
    for EACH_FILE_NAME in sorted(os.listdir(FOLDER_INPUT), key=FUNCTION_NATURAL_SORT_KEY):
        if not EACH_FILE_NAME.endswith('.txt'):
            continue
        FILE_PATH = os.path.join(FOLDER_INPUT, EACH_FILE_NAME)
        with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            if any(any(EACH_WORD in EACH_LINE for EACH_WORD in LIST_WORD) for EACH_LINE in FILE_HANDLE):
                LIST_PATH.append(FILE_PATH)
    LIST_PATH.sort(key=lambda EACH_PATH: int(EACH_PATH.split('SPECIFICATION_')[1].split('.txt')[0]))
    LOCAL_COUNTER = 1
    for EACH_FILE_PATH in LIST_PATH:
        with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            STRING_SPEC = FILE_HANDLE.read()
        STRING_PREFIX_REGISTER = (EACH_REGISTER if STRING_PREFIX_REGISTER_DIGIT == '' else STRING_PREFIX_REGISTER_DIGIT)
        if SPECIAL_NAME_FOR_NUMBERS == '':
            SPECIAL_NAME_FOR_NUMBERS_PROMPT = ''
        else:
            SPECIAL_NAME_FOR_NUMBERS_PROMPT = f' or {SPECIAL_NAME_FOR_NUMBERS.strip()}'
        STRING_PROMPT = f'''For RISC-V, does this text below contain any information about {STRING_PREFIX_REGISTER}{SPECIAL_NAME_FOR_NUMBERS_PROMPT} register? Output YES or NO only.

The text:

{STRING_SPEC}'''
        STRING_RESPONSE = FUNCTION_CLEAN_TEXT(FUNCTION_LLM_CALL(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT))
        STRING_INTRO_FIRST = f' START - {GLOBAL_COUNTER} '.center(WIDTH, '=')
        STRING_INTRO_END = f' END - {GLOBAL_COUNTER} '.center(WIDTH, '=')
        STRING_INTRO = f'''GLOBAL NUMBER: {GLOBAL_COUNTER}
LOCAL NUMBER: {LOCAL_COUNTER}
NAME: {STRING_PREFIX_REGISTER}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
        STRING_ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
        LIST_HISTORY.append(STRING_INTRO_FIRST + '\n' + STRING_INTRO + STRING_PROMPT + STRING_ENDING + STRING_INTRO_END + SUFFIX)
        if 'YES' in STRING_RESPONSE:
            LIST_OUTPUT_LINE.append(f'{EACH_REGISTER} - {EACH_FILE_PATH}')
        GLOBAL_COUNTER += 1
        LOCAL_COUNTER += 1

STRING_PROMPT_HISTORY = ''.join(LIST_HISTORY).removesuffix(SUFFIX)

with open(FILE_PROMPT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n'.join(LIST_OUTPUT_LINE))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 3 - CREATE - 3_REGISTER_BOOLEAN.txt - END

14 minute(s) 12.93 second(s)
====
DONE
====


In [12]:
# 4 - CREATE - 4_REGISTER_RULE.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from collections import defaultdict
from ollama import chat, Client
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_PROMPT_1 = '5_REGISTER/2_HISTORY/2_HISTORY_REGISTER_RULE_1.txt'
FILE_PROMPT_2 = '5_REGISTER/2_HISTORY/3_HISTORY_REGISTER_RULE_2.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_LLM(STRING_PROMPT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()

def FUNCTION_HISTORY_BLOCK(GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_NORMALIZE_RULE(TEXT_VALUE):
    LIST_LINE = [EACH_LN.strip() for EACH_LN in TEXT_VALUE.splitlines() if EACH_LN.strip()]
    LIST_RULE = []
    for EACH_LN in LIST_LINE:
        if re.match(r'^RULE\s+\d+:', EACH_LN, re.I):
            EACH_LN = re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_LN)
            LIST_RULE.append(EACH_LN)
    LIST_FIXED = []
    for EACH_LN in LIST_RULE:
        MATCH_RULE = re.match(r'^\s*RULE\s+(\d+):\s*(.*)', EACH_LN)
        if not MATCH_RULE:
            LIST_FIXED.append(EACH_LN)
            continue
        NUMBER_RULE = MATCH_RULE.group(1)
        STRING_REST = MATCH_RULE.group(2)
        STRING_REST = re.sub(rf'^RULE\s+{NUMBER_RULE}:\s*', '', STRING_REST)
        LIST_FIXED.append(f'RULE {NUMBER_RULE}: {STRING_REST}')
    return '\n'.join(LIST_FIXED)
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 65536
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
LIST_BLOCK_OUTPUT = []
MAP_REGISTER_RULE = defaultdict(list)
LIST_HISTORY_1 = []
LIST_HISTORY_2 = []
GLOBAL_COUNTER_1 = 1
LOCAL_COUNTER_1 = 1
GLOBAL_COUNTER_2 = 1
LOCAL_COUNTER_2 = 1
REGISTER_INITIAL = None
LIST_BLOCK_FINAL = []
LIST_SUPER_FINAL = []
IS_64_BIT = True
PATTERN_RV32 = re.compile(r'\bRV32\b')
PATTERN_32BIT = re.compile(r'\b32[\-\u2011]bits?\b')
PATTERN_WFI = re.compile(r'\bWFI\b')
PATTERN_INTERRUPT = re.compile(r'\binterrupts\b', re.IGNORECASE)
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
    if EACH_REGISTER not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_REGISTER)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER].append(FILE_HANDLE.read())

for EACH_REGISTER in LIST_REGISTER_ORDER:
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER]).strip()
    LOCAL_COUNTER_1 = 1
    LOCAL_COUNTER_2 = 1
    
    # PROMPT 1 - BEGIN
    STRING_PROMPT_1 = f'''For RISC-V, please extract all possible information about {EACH_REGISTER} register from the text below. The information must be only about {EACH_REGISTER} register, not other things. Extract ONLY explicit, testable, normative description about architectural behavior.

Please output in paragraphs. If you want, you can include bullet or numbered points in your output. You can also include tables. However, do not give me any introduction in your output. You just need to output the information that you extract.

If your output includes some sentences that refer to some tables or figures, include those tables or figures too.

Remember, output must be ASCII-only text.

If it happens to be there is no information about {EACH_REGISTER} register from the text below, output NONE

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

Your description must be very clear and unambiguous. If you want, you can output in bullet points also.

Use the exact terminology and wording used in the text below; do not substitute synonyms or paraphrase architectural terms.

The text below:

{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE_1 = FUNCTION_LLM(STRING_PROMPT_1)
    if STRING_RESPONSE_1.strip() == 'NONE':
        continue
    STRING_RESPONSE_1 = FUNCTION_CLEAN_TEXT(STRING_RESPONSE_1)
    LIST_HISTORY_1.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_1, LOCAL_COUNTER_1, EACH_REGISTER, STRING_PROMPT_1, STRING_RESPONSE_1))
    # PROMPT 1 - FINISH

    # PROMPT 2 - BEGIN
    STRING_PROMPT_2 = f'''For RISC-V, for {EACH_REGISTER} register, rewrite the following description as a list of rules. Each rule must explain about {EACH_REGISTER} register. Essentially, for each rule, it is about {EACH_REGISTER} register, not other things.

Do not lose any information about {EACH_REGISTER} register from the description. Read the text or the description carefully. Understand the context and the sentences. Do not lose any important information.

For each rule, do not use any internal references like "According to Table 7," or "see Figure 3." Each rule must be one sentence only. Each rule must stand fully on its own without cross-references. Do not quote or refer to other rules. Essentially, each rule is one idea. Avoid a rule like this: "The register being described is {EACH_REGISTER}."

Each rule should become a single testable statement. Each rule should be a separate atomic rule, each expressing only one fact. Each rule needs to be clear and detailed. Extract ONLY explicit, testable, normative rules about architectural behavior. A rule MUST describe behavioral semantics, not naming, encoding, or syntax.

Remember, rules must describe architectural cause -> effect semantics (event -> architectural outcome) rather than decoding the meaning of an already-set register value.

Remember, if a sentence contains multiple distinct behaviors or enumerated cases, split them into multiple rules so each rule expresses only one case.

Do not write rules in the form "{EACH_REGISTER} has <bit pattern> means/represents/indicates <cause>"; instead, write them as "If <cause/event> occurs, {EACH_REGISTER} becomes/contains <bit pattern>".

Exclude purely structural or descriptive properties (such as register width, naming, or existence) unless they imply observable architectural behavior.

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

Ignore any statements that describe reset values, power-on defaults, or initial state after reset; do not generate rules from reset-related information.

OUTPUT FORMAT:
+ Each rule must begin with the word "RULE", followed by its number (e.g., "RULE 1: ", "RULE 2: ", "RULE 3: ", ...).
+ Do not give me any introduction.
+ Each rule must contain the word {EACH_REGISTER}.

Remember, output must be ASCII-only text.

The following description:

{STRING_RESPONSE_1}'''
    STRING_RESPONSE_2 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_2))
    LIST_HISTORY_2.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_2, LOCAL_COUNTER_2, EACH_REGISTER, STRING_PROMPT_2, STRING_RESPONSE_2))
    # PROMPT 2 - FINISH

    # FINALIZING - BEGIN
    STRING_RESPONSE_FINISH = STRING_RESPONSE_2
    STRING_CLEANED = FUNCTION_NORMALIZE_RULE(STRING_RESPONSE_FINISH)
    LIST_BLOCK_OUTPUT.append(f'REGISTER: {EACH_REGISTER}\n{STRING_CLEANED}')
    # FINALIZING - FINISH

    # CLEANING UP PROMPT - BEGIN
    GLOBAL_COUNTER_1 += 1
    LOCAL_COUNTER_1 += 1
    GLOBAL_COUNTER_2 += 1
    LOCAL_COUNTER_2 += 1

with open(FILE_PROMPT_1, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY_1).removesuffix(SUFFIX))

with open(FILE_PROMPT_2, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY_2).removesuffix(SUFFIX))
# CLEANING UP PROMPT - FINISH

for EACH_BLOCK in LIST_BLOCK_OUTPUT:
    LIST_LINE = EACH_BLOCK.strip().splitlines()
    
    # BONUS FEATURE - DELETE UNWORTHY RULES - BEGIN
    if IS_64_BIT:
        LIST_LINE = [
            EACH_LINE
            for EACH_LINE in LIST_LINE
            if not (
                PATTERN_RV32.search(EACH_LINE)
                or PATTERN_32BIT.search(EACH_LINE)
                or PATTERN_WFI.search(EACH_LINE)
                or PATTERN_INTERRUPT.search(EACH_LINE)
                or 'XLEN=32' in EACH_LINE
                or 'XLEN = 32' in EACH_LINE
                or 'XLEN is 32' in EACH_LINE
                or 'XLEN equals 32' in EACH_LINE
            )
        ]
    # BONUS FEATURE - DELETE UNWORTHY RULES - FINISH
    
    STRING_NAME = LIST_LINE[0].split(':', 1)[1].strip()
    MAP_REGISTER_RULE[STRING_NAME].extend(LIST_LINE[1:])

for EACH_NAME, EACH_LIST_RULE in MAP_REGISTER_RULE.items():
    LIST_LINE_NEW = [f'REGISTER: {EACH_NAME}']
    for EACH_INDEX_RULE, EACH_RULE_LINE in enumerate(EACH_LIST_RULE, 1):
        LIST_LINE_NEW.append(f"RULE {EACH_INDEX_RULE}: {EACH_RULE_LINE.split(':', 1)[1].strip()}")
    LIST_BLOCK_FINAL.append('\n'.join(LIST_LINE_NEW))

for EACH_BLOCK in LIST_BLOCK_FINAL:
    LIST_LINE = EACH_BLOCK.splitlines()
    STRING_HEADER = LIST_LINE[0].strip()
    LIST_RULE = [EACH_R.strip() for EACH_R in LIST_LINE[1:] if EACH_R.strip() and EACH_R.strip() != 'RULE:']
    LIST_RULE = [re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_R) for EACH_R in LIST_RULE]
    LIST_RULE = [re.sub(r'RULE\s+\d+:', f'RULE {EACH_I}:', EACH_R) for EACH_I, EACH_R in enumerate(LIST_RULE, 1)]
    LIST_SUPER_FINAL.append(STRING_HEADER + '\n' + '\n'.join(LIST_RULE))

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_SUPER_FINAL))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 4 - CREATE - 4_REGISTER_RULE.txt - END

20 minute(s) 55.66 second(s)
====
DONE
====


In [13]:
# 5 - CREATE - 5_REGISTER_EXTENSION.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/5_REGISTER_EXTENSION.txt'
FILE_PROMPT = '5_REGISTER/2_HISTORY/4_HISTORY_REGISTER_EXTENSION.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_LLM(STRING_PROMPT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_HISTORY_BLOCK(GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_SPLIT_BLOCK(TEXT_VALUE: str):
    LIST_BLOCK = []
    LIST_CURRENT = []
    for EACH_LINE in TEXT_VALUE.splitlines(True):
        if EACH_LINE.startswith('REGISTER:') and LIST_CURRENT:
            LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
            LIST_CURRENT = []
        LIST_CURRENT.append(EACH_LINE)
    if LIST_CURRENT:
        LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
    return LIST_BLOCK

def FUNCTION_TOKEN_OK(STRING_TOKEN: str):
    STRING_TOKEN = STRING_TOKEN.strip()
    if not STRING_TOKEN or ' ' in STRING_TOKEN or not STRING_TOKEN.isalpha():
        return False
    return STRING_TOKEN[0].isupper() and (len(STRING_TOKEN) == 1 or STRING_TOKEN[1:].islower())
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 65536
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
LIST_HISTORY = []
GLOBAL_COUNTER = 1
LOCAL_COUNTER = 1
REGISTER_INITIAL = None
MAP_MERGED = {}
LIST_OUTPUT_ENTRY = []
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
    if EACH_REGISTER not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_REGISTER)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER].append(FILE_HANDLE.read())

for EACH_REGISTER in LIST_REGISTER_ORDER:
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER]).strip()
    LOCAL_COUNTER = 1
    STRING_PROMPT = f'''For RISC-V, based on the text below, output all extensions (RISC-V extensions) for {EACH_REGISTER} register. Note: registers, CSRs, fields, modes, interrupts, and instructions are not extensions.

Read the text below carefully and make sure you don't miss any extensions!

HOW TO OUTPUT:
Enter extensions as a comma-separated list on one line. If there are no extensions for {EACH_REGISTER} register in the rules, output NONE.

The text below:

{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT))
    LIST_HISTORY.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER, LOCAL_COUNTER, EACH_REGISTER, STRING_PROMPT, STRING_RESPONSE))
    STRING_RESPONSE_2 = STRING_RESPONSE.replace(EACH_REGISTER, '')
    if 'NONE' not in STRING_RESPONSE_2:
        if EACH_REGISTER in MAP_MERGED:
            MAP_MERGED[EACH_REGISTER] = f'{MAP_MERGED[EACH_REGISTER]}, {STRING_RESPONSE_2}'
        else:
            MAP_MERGED[EACH_REGISTER] = STRING_RESPONSE_2
    GLOBAL_COUNTER += 1
    LOCAL_COUNTER += 1

with open(FILE_PROMPT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY).removesuffix(SUFFIX))

for EACH_REGISTER in sorted(MAP_MERGED, key=str.lower):
    SET_SEEN = set()
    LIST_TOKEN = []
    for EACH_TOKEN in MAP_MERGED[EACH_REGISTER].split(','):
        STRING_TOKEN = EACH_TOKEN.strip()
        if STRING_TOKEN and STRING_TOKEN not in SET_SEEN:
            SET_SEEN.add(STRING_TOKEN)
            LIST_TOKEN.append(STRING_TOKEN)
    LIST_TOKEN = [EACH_TOKEN for EACH_TOKEN in LIST_TOKEN if FUNCTION_TOKEN_OK(EACH_TOKEN)]
    LIST_OUTPUT_ENTRY.append(f"REGISTER: {EACH_REGISTER}\nEXTENSIONS: {', '.join(LIST_TOKEN)}")

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT_ENTRY))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 5 - CREATE - 5_REGISTER_EXTENSION.txt - END

3 minute(s) 25.31 second(s)
====
DONE
====


In [14]:
# 6 - CREATE - 6_REGISTER_FIELD.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/6_REGISTER_FIELD.txt'
FILE_PROMPT_1 = '5_REGISTER/2_HISTORY/5_HISTORY_REGISTER_FIELD_1.txt'
FILE_PROMPT_2 = '5_REGISTER/2_HISTORY/6_HISTORY_REGISTER_FIELD_2.txt'
FILE_OTHER_WIDTH = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_OTHER_INPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_LLM(STRING_PROMPT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_HISTORY_BLOCK(GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_SPLIT_BLOCK(LIST_LINE):
    LIST_BLOCK = []
    LIST_CURRENT = []
    for EACH_LINE in LIST_LINE:
        if EACH_LINE.startswith('REGISTER:') and LIST_CURRENT:
            LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
            LIST_CURRENT = []
        LIST_CURRENT.append(EACH_LINE)
    if LIST_CURRENT:
        LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
    return LIST_BLOCK
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 65536
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
LIST_REGISTER_NAME = []
MAP_WIDTH = {}
LIST_INJECTED = []
LIST_OUTPUT_BLOCK = []
LIST_HISTORY_1 = []
LIST_HISTORY_2 = []
GLOBAL_COUNTER_1 = 1
LOCAL_COUNTER_1 = 1
GLOBAL_COUNTER_2 = 1
LOCAL_COUNTER_2 = 1
REGISTER_INITIAL = None
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_OTHER_WIDTH, encoding='utf-8') as FILE_HANDLE:
    STRING_REGISTER = None
    STRING_WIDTH = None
    for EACH_LINE in FILE_HANDLE:
        if EACH_LINE.startswith('REGISTER: '):
            STRING_REGISTER = EACH_LINE.split(': ', 1)[1].strip()
        elif EACH_LINE.startswith('WIDTH: '):
            STRING_WIDTH = EACH_LINE.split(': ', 1)[1].strip()
        elif (EACH_LINE.startswith('FIELDS: -') and STRING_REGISTER is not None and STRING_WIDTH is not None):
            STRING_PAIR = f'{STRING_REGISTER} - {STRING_WIDTH}'
            LIST_REGISTER_NAME.append(STRING_PAIR)
            MAP_WIDTH[STRING_REGISTER] = STRING_WIDTH

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_RULE_LINE = FILE_HANDLE.readlines()

for EACH_LINE in LIST_RULE_LINE:
    if EACH_LINE.startswith('REGISTER:'):
        LIST_INJECTED.append(EACH_LINE)
        STRING_REGISTER = EACH_LINE.split('REGISTER:', 1)[1].strip()
        if STRING_REGISTER in MAP_WIDTH:
            LIST_INJECTED.append(f'WIDTH: {MAP_WIDTH[STRING_REGISTER]}\n')

LIST_INPUT_2 = FUNCTION_SPLIT_BLOCK(LIST_INJECTED)
MAP_BLOCK_BY_REGISTER = {EACH_BLOCK.splitlines()[0].split('REGISTER:', 1)[1].strip(): EACH_BLOCK for EACH_BLOCK in LIST_INPUT_2}

with open(FILE_OTHER_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
    if EACH_REGISTER not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_REGISTER)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER].append(FILE_HANDLE.read())

for EACH_INDEX_SPECIAL, EACH_SPECIAL in enumerate(LIST_REGISTER_NAME):
    STRING_REGISTER, STRING_WIDTH = (EACH_PART.strip() for EACH_PART in EACH_SPECIAL.split(' - ', 1))
    STRING_BLOCK = MAP_BLOCK_BY_REGISTER.get(STRING_REGISTER, '')
    if EACH_INDEX_SPECIAL == 0:
        REGISTER_INITIAL = STRING_REGISTER
    elif STRING_REGISTER != REGISTER_INITIAL:
        REGISTER_INITIAL = STRING_REGISTER
        LOCAL_COUNTER_1 = 1
        LOCAL_COUNTER_2 = 1
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[STRING_REGISTER]).strip()
    STRING_PROMPT_1 = f'''For RISC-V, based on the text below, output all fields for {STRING_REGISTER} register. Do not give any introduction.

The text below:

{STRING_BLOCK}
TEXT:
{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE_1 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_1))
    LIST_HISTORY_1.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_1, LOCAL_COUNTER_1, STRING_REGISTER, STRING_PROMPT_1, STRING_RESPONSE_1))
    NUMBER_WIDTH = int(STRING_WIDTH) if STRING_WIDTH.isdigit() else 64
    STRING_PROMPT_2 = f'''For RISC-V, from the description below, extract the fields for {STRING_REGISTER} register. The register of {STRING_REGISTER} is a {NUMBER_WIDTH}-bit register.

OUTPUT FORMAT:
FIELDS: <high_bit>..<low_bit>=<FIELD_NAME>, <high_bit>..<low_bit>=<FIELD_NAME>, ...

RULES:
+ Range Delimiter -> Always use two dots (..) for inclusive ranges (e.g., 7..0). Repeat the index for single-bit fields (e.g., 3..3).
+ Coverage -> Define every bit from {NUMBER_WIDTH - 1} down to 0, in contiguous, non-overlapping ranges. No gaps.
+ Reserved Bits -> If the description says something like "unused", "future use", or "reserved", then the <FIELD_NAME> must be WPRI.
+ Unknown Positions -> If a field's bit range is not given in the description, append the field at the end. The <high_bit> and the <low_bit> become "UNKNOWN". For example, "UNKNOWN..UNKNOWN=<FIELD_NAME>".
+ Ordering -> Based on the position or the bit range, list fields strictly from high to low, separated by ", " (comma and space).
+ No Overlaps -> Ensure bit ranges don't overlap.

Do not output any introduction.

The description:

{STRING_RESPONSE_1}'''
    STRING_RESPONSE_2 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_2))
    LIST_HISTORY_2.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_2, LOCAL_COUNTER_2, STRING_REGISTER, STRING_PROMPT_2, STRING_RESPONSE_2))
    LIST_OUTPUT_BLOCK.append(f'REGISTER: {STRING_REGISTER}\n{STRING_RESPONSE_2}')
    GLOBAL_COUNTER_1 += 1
    LOCAL_COUNTER_1 += 1
    GLOBAL_COUNTER_2 += 1
    LOCAL_COUNTER_2 += 1

with open(FILE_PROMPT_1, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY_1).removesuffix(SUFFIX))

with open(FILE_PROMPT_2, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY_2).removesuffix(SUFFIX))

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT_BLOCK))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 6 - CREATE - 6_REGISTER_FIELD.txt - END

9.98 second(s)
====
DONE
====


In [15]:
# 7 - CREATE - 7_REGISTER_FINAL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_LIST = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_RULE = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_EXTENSION = '5_REGISTER/1_OUTPUT/5_REGISTER_EXTENSION.txt'
FILE_BIT = '5_REGISTER/1_OUTPUT/6_REGISTER_FIELD.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/7_REGISTER_FINAL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_READ(FILE_PATH):
    PATH_VALUE = Path(FILE_PATH)
    return PATH_VALUE.read_text(encoding='utf-8') if PATH_VALUE.exists() else ''

def FUNCTION_SPLIT_SECTION(TEXT_VALUE: str):
    MAP_OUTPUT = {}
    for EACH_PART in re.split(r'(?=^REGISTER:\s*)', TEXT_VALUE, flags=re.M):
        if not EACH_PART.strip():
            continue
        MATCH_REGISTER = re.match(r'^REGISTER:\s*(.+?)\s*[\r\n]+', EACH_PART)
        if MATCH_REGISTER:
            MAP_OUTPUT[MATCH_REGISTER.group(1).strip()] = EACH_PART.strip()
    return MAP_OUTPUT
# FUNCTION - FINISH

# VARIABLE - BEGIN
MAP_SECTION_BASE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_LIST))
MAP_SECTION_FIELD = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_BIT))
MAP_SECTION_EXTENSION = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_EXTENSION))
MAP_SECTION_RULE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_RULE))
LIST_BLOCK_FINAL = []
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_REGISTER, EACH_BASE_SECTION in MAP_SECTION_BASE.items():
    LIST_LINE = EACH_BASE_SECTION.splitlines()
    if EACH_REGISTER in MAP_SECTION_FIELD:
        MATCH_FIELD = re.search(r'^\s*FIELDS:\s*(.+)$', MAP_SECTION_FIELD[EACH_REGISTER], flags=re.M)
        if MATCH_FIELD:
            STRING_FIELD = MATCH_FIELD.group(1).strip()
            for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
                if EACH_LINE.strip().startswith('FIELDS:'):
                    LIST_LINE[EACH_INDEX_LINE] = f'FIELDS: {STRING_FIELD}'
                    break
            else:
                LIST_LINE.append(f'FIELDS: {STRING_FIELD}')
    INDEX_BASE_EXTENSION = None
    LIST_EXTENSION_BASE = []
    for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
        if EACH_LINE.strip().startswith('EXTENSIONS:'):
            INDEX_BASE_EXTENSION = EACH_INDEX_LINE
            STRING_AFTER = EACH_LINE.split(':', 1)[1].strip()
            if STRING_AFTER and STRING_AFTER != '-':
                LIST_EXTENSION_BASE = [EACH_EXTENSION.strip() for EACH_EXTENSION in STRING_AFTER.split(',') if EACH_EXTENSION.strip()]
            break
    LIST_EXTENSION_ADD = []
    if EACH_REGISTER in MAP_SECTION_EXTENSION:
        MATCH_EXTENSION = re.search(r'^\s*EXTENSIONS:\s*(.+)$', MAP_SECTION_EXTENSION[EACH_REGISTER], flags=re.M)
        if MATCH_EXTENSION:
            LIST_EXTENSION_ADD = [EACH_EXTENSION.strip() for EACH_EXTENSION in MATCH_EXTENSION.group(1).split(',') if EACH_EXTENSION.strip()]
    LIST_EXTENSION_MERGED = sorted(set(LIST_EXTENSION_BASE + LIST_EXTENSION_ADD))
    STRING_EXTENSION_LINE = 'EXTENSIONS: ' + (', '.join(LIST_EXTENSION_MERGED) if LIST_EXTENSION_MERGED else '-')
    if INDEX_BASE_EXTENSION is None:
        LIST_LINE.append(STRING_EXTENSION_LINE)
    else:
        LIST_LINE[INDEX_BASE_EXTENSION] = STRING_EXTENSION_LINE
    if EACH_REGISTER in MAP_SECTION_RULE:
        LIST_RULE_LINE = []
        for EACH_RAW in MAP_SECTION_RULE[EACH_REGISTER].splitlines():
            STRING_RAW = EACH_RAW.strip()
            if STRING_RAW.upper().startswith('RULE '):
                LIST_RULE_LINE.append(STRING_RAW.replace('“', '"').replace('”', '"'))
        if LIST_RULE_LINE:
            NUMBER_RULE_MAX = 0
            for EACH_LINE in LIST_LINE:
                MATCH_RULE = re.match(r'^\s*RULE\s+(\d+)\s*:', EACH_LINE, flags=re.I)
                if MATCH_RULE:
                    NUMBER_RULE_MAX = max(NUMBER_RULE_MAX, int(MATCH_RULE.group(1)))
            NUMBER_NEXT = NUMBER_RULE_MAX + 1
            for EACH_RULE in LIST_RULE_LINE:
                if re.match(r'^\s*RULE\s+\d+\s*:', EACH_RULE, flags=re.I):
                    LIST_LINE.append(EACH_RULE)
                else:
                    LIST_LINE.append(f'RULE {NUMBER_NEXT}: {EACH_RULE}')
                    NUMBER_NEXT += 1
    if EACH_REGISTER not in MAP_SECTION_RULE:
        continue
    LIST_BLOCK_FINAL.append('\n'.join(LIST_LINE).strip())

Path(FILE_OUTPUT).write_text('\n\n'.join(LIST_BLOCK_FINAL), encoding='utf-8')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 7 - CREATE - 7_REGISTER_FINAL.txt - END

0.01 second(s)
====
DONE
====


In [16]:
# 8 - CREATE - 8_REGISTER_TOTAL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/7_REGISTER_FINAL.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/8_REGISTER_TOTAL.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_INPUT = FILE_HANDLE.read()

if (not os.path.exists(FILE_OUTPUT)) or os.path.getsize(FILE_OUTPUT) == 0:
    TEXT_OUTPUT = TEXT_INPUT
else:
    with open(FILE_OUTPUT, encoding='utf-8') as FILE_HANDLE:
        TEXT_PREVIOUS = FILE_HANDLE.read()
    TEXT_COMBINED = TEXT_PREVIOUS.strip() + '\n\n' + TEXT_INPUT.strip()
    REGEX_BLOCK = re.compile(r'(?ms)^\s*REGISTER:\s*([^\n]+)\s*\n.*?(?=^\s*REGISTER:\s*[^\n]+\s*\n|\Z)')
    MAP_BLOCK = {MATCH.group(1).strip(): MATCH.group(0) for MATCH in REGEX_BLOCK.finditer(TEXT_COMBINED)}
    TEXT_OUTPUT = '\n\n'.join(MAP_BLOCK[EACH_NAME].strip() for EACH_NAME in sorted(MAP_BLOCK, key=str.upper)).strip()

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(TEXT_OUTPUT)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 8 - CREATE - 8_REGISTER_TOTAL.txt - END

0.00 second(s)
====
DONE
====


In [17]:
# 9 - CREATE - 9_REGISTER_SUMMARY.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
from typing import Dict, List, Tuple
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/9_REGISTER_SUMMARY.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# ALARM PLAY - BEGIN
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
import pygame
import time
pygame.mixer.init()
pygame.mixer.music.load('alarm.mp3')
for _ in range(3):
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.01)
pygame.mixer.quit()
# ALARM PLAY - FINISH

# 9 - CREATE - 9_REGISTER_SUMMARY.txt - END

0.00 second(s)
====
DONE
====


In [18]:
# 1 - CREATE - 1_TEST_CASE_INPUT.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT_1 = '4_INSTRUCTION/1_OUTPUT/6_INSTRUCTION_TOTAL.txt'
FILE_INPUT_2 = '5_REGISTER/1_OUTPUT/8_REGISTER_TOTAL.txt'
FILE_OUTPUT = '6_TEST_CASE/1_TEST_CASE_INPUT.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_PARSE_BLOCK(TEXT_VALUE: str):
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n').replace('\r', '\n')
    LIST_HEADER = [(MATCH_HEADER.group(1), MATCH_HEADER.group(2).strip(), MATCH_HEADER.end()) for MATCH_HEADER in REGEX_HEADER.finditer(TEXT_VALUE)]
    LIST_POSITION = [EACH_POS for _, _, EACH_POS in LIST_HEADER] + [len(TEXT_VALUE)]
    MAP_BLOCK = {'INSTRUCTION': {}, 'REGISTER': {}}
    for EACH_INDEX, (EACH_KIND, EACH_NAME, EACH_END) in enumerate(LIST_HEADER):
        STRING_BODY = TEXT_VALUE[EACH_END:LIST_POSITION[EACH_INDEX + 1]].strip()
        if EACH_KIND == 'INSTRUCTION':
            MATCH_FORMAT = re.search(r'^FORMAT:\s*(.+)$', STRING_BODY, re.M)
            MAP_RULE = {int(EACH_RULE_NUMBER): EACH_RULE_TEXT.strip() for EACH_RULE_NUMBER, EACH_RULE_TEXT in re.findall(r'RULE\s+(\d+):\s*(.*)', STRING_BODY)}
            MAP_BLOCK[EACH_KIND][EACH_NAME] = {'format': MATCH_FORMAT.group(1).strip() if MATCH_FORMAT else '', 'rules': MAP_RULE}
        else:
            def FUNCTION_GET_VALUE(STRING_KEY):
                MATCH_VALUE = re.search(rf'^{STRING_KEY}:\s*(.+)$', STRING_BODY, re.M)
                return MATCH_VALUE.group(1).strip() if MATCH_VALUE else ''
            MAP_RULE = {int(EACH_RULE_NUMBER): EACH_RULE_TEXT.strip() for EACH_RULE_NUMBER, EACH_RULE_TEXT in re.findall(r'RULE\s+(\d+):\s*(.*)', STRING_BODY)}
            STRING_EXTENSION = FUNCTION_GET_VALUE('EXTENSIONS')
            LIST_EXTENSION = ([EACH_EXTENSION.strip() for EACH_EXTENSION in STRING_EXTENSION.split(',')] if STRING_EXTENSION else [])
            MAP_BLOCK[EACH_KIND][EACH_NAME] = {'number': FUNCTION_GET_VALUE('NUMBER'), 'mode': FUNCTION_GET_VALUE('MODE'), 'width': FUNCTION_GET_VALUE('WIDTH'), 'fields': FUNCTION_GET_VALUE('FIELDS'), 'extensions': LIST_EXTENSION, 'rules': MAP_RULE}
    return MAP_BLOCK

def FUNCTION_RENDER_SELECTED(TEXT_VALUE: str, VALUE_WANT):
    MAP_BLOCK = FUNCTION_PARSE_BLOCK(TEXT_VALUE)
    MAP_INDEX = {EACH_NAME.lower(): (EACH_KIND, EACH_NAME) for EACH_KIND, EACH_MAP_ITEM in MAP_BLOCK.items() for EACH_NAME in EACH_MAP_ITEM}
    LIST_OUTPUT_LINE = []
    for EACH_RAW_KEY, EACH_SELECTED in (VALUE_WANT.items() if isinstance(VALUE_WANT, dict) else VALUE_WANT):
        STRING_KEY = EACH_RAW_KEY.lower()
        if STRING_KEY not in MAP_INDEX:
            continue
        STRING_KIND, STRING_NAME = MAP_INDEX[STRING_KEY]
        MAP_ITEM = MAP_BLOCK[STRING_KIND][STRING_NAME]
        LIST_WANTED = ([EACH_SELECTED] if isinstance(EACH_SELECTED, int) else list(EACH_SELECTED))
        if STRING_KIND == 'INSTRUCTION':
            LIST_OUTPUT_LINE += [f'INSTRUCTION: {STRING_NAME}']
            if MAP_ITEM['format']:
                LIST_OUTPUT_LINE += [f'FORMAT: {MAP_ITEM['format']}']
        else:
            LIST_OUTPUT_LINE += [f'REGISTER: {STRING_NAME}']
            for EACH_FIELD_KEY in ('number', 'mode', 'width', 'fields'):
                if MAP_ITEM.get(EACH_FIELD_KEY):
                    LIST_OUTPUT_LINE += [f'{EACH_FIELD_KEY.upper()}: {MAP_ITEM[EACH_FIELD_KEY]}']
            if MAP_ITEM.get('extensions'):
                LIST_OUTPUT_LINE += ['EXTENSIONS: ' + ', '.join(MAP_ITEM['extensions'])]
        for EACH_RULE_ID in LIST_WANTED:
            if EACH_RULE_ID in MAP_ITEM['rules']:
                LIST_OUTPUT_LINE += [f'RULE {EACH_RULE_ID}: {MAP_ITEM['rules'][EACH_RULE_ID]}']
        LIST_OUTPUT_LINE += ['']
    while LIST_OUTPUT_LINE and LIST_OUTPUT_LINE[-1] == '':
        LIST_OUTPUT_LINE.pop()
    return '\n'.join(LIST_OUTPUT_LINE)
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_HEADER = re.compile(r'^(INSTRUCTION|REGISTER):\s*(.+?)\s*$', re.M)
IS_USING_EVERYTHING = True
LIST_WANT = [('mcause', [1, 2])]
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT_1, encoding='utf-8') as FILE_HANDLE:
    STRING_CONTENT_1 = FILE_HANDLE.read().strip()

with open(FILE_INPUT_2, encoding='utf-8') as FILE_HANDLE:
    STRING_CONTENT_2 = FILE_HANDLE.read().strip()

STRING_TOTAL = STRING_CONTENT_1 + '\n\n' + STRING_CONTENT_2
STRING_FINAL = STRING_TOTAL if IS_USING_EVERYTHING else FUNCTION_RENDER_SELECTED(STRING_TOTAL, LIST_WANT)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_FINAL)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 1_TEST_CASE_INPUT.txt - END

0.00 second(s)
====
DONE
====


In [19]:
# 2 - CREATE - 2_TEST_CASE_RAW.txt & 3_TEST_CASE_REVISED.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from __future__ import annotations
from dataclasses import dataclass
from ollama import chat, Client
from typing import Dict, List, Optional, Tuple
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '6_TEST_CASE/1_TEST_CASE_INPUT.txt'
FILE_OUTPUT_REVISED = '6_TEST_CASE/3_TEST_CASE_REVISED.txt'
FILE_OUTPUT_RAW = '6_TEST_CASE/2_TEST_CASE_RAW.txt'
FILE_OUTPUT_HISTORY = '6_TEST_CASE/4_HISTORY_TEST_CASE.txt'
FILE_REGISTER_ALL = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
FILE_INSTRUCTION_ALL = '4_INSTRUCTION/1_OUTPUT/1_INSTRUCTION_ALL.txt'
FILE_INSTRUCTION_SUMMARY = '4_INSTRUCTION/1_OUTPUT/7_INSTRUCTION_SUMMARY.txt'
FILE_REGISTER_SUMMARY = '5_REGISTER/1_OUTPUT/9_REGISTER_SUMMARY.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_SANITIZE_TEXT(STRING_VALUE: str) -> str:
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_CODEPOINT in range(0xE000, 0xF900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_CODEPOINT), '')
    STRING_VALUE = (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))
    return STRING_VALUE.strip()

def FUNCTION_CALL_LLM(STRING_PROMPT: str) -> str:
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content

def FUNCTION_WHOLE_WORD_PATTERN(STRING_WORD: str, FLAGS: int = 0) -> re.Pattern:
    return re.compile(
        r'(?<![A-Za-z0-9_])' + re.escape(STRING_WORD) + r'(?![A-Za-z0-9_])',
        FLAGS
    )

def FUNCTION_PARSE_REGISTER_NAME_FILE(TEXT_VALUE: str) -> Dict[str, Tuple[str, str, Optional[str], Optional[str]]]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n').strip()
    LIST_BLOCK = re.split(r'\n\s*\n', TEXT_VALUE)
    MAP_OUTPUT: Dict[str, Tuple[str, str, Optional[str], Optional[str]]] = {}
    for EACH_BLOCK in LIST_BLOCK:
        LIST_LINE = [EACH_LINE.strip() for EACH_LINE in EACH_BLOCK.splitlines() if EACH_LINE.strip()]
        if not LIST_LINE:
            continue
        STRING_REGISTER = None
        STRING_NUMBER = None
        STRING_FIELDS = None
        STRING_MODE = None  # NEW
        for EACH_LINE in LIST_LINE:
            if EACH_LINE.startswith('REGISTER:') and not EACH_LINE.startswith('REGISTER: mcause'):
                STRING_REGISTER = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('NUMBER:'):
                STRING_NUMBER = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('MODE:'):  # NEW
                STRING_MODE = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('FIELDS:'):
                STRING_VALUE = EACH_LINE.split(':', 1)[1].strip()
                if STRING_VALUE != '-':
                    STRING_FIELDS = STRING_VALUE
        if STRING_REGISTER and STRING_NUMBER:
            STRING_HEX = STRING_NUMBER
            try:
                VALUE_INT = int(STRING_NUMBER, 0)
                STRING_HEX = f'0x{VALUE_INT:X}'
            except ValueError:
                pass
            MAP_OUTPUT[STRING_REGISTER.lower()] = (STRING_REGISTER, STRING_HEX, STRING_FIELDS, STRING_MODE)  # CHANGED
    return MAP_OUTPUT

def FUNCTION_PARSE_INSTRUCTION_NAME_FILE(TEXT_VALUE: str) -> Dict[str, Tuple[str, str]]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n').strip()
    LIST_BLOCK = re.split(r'\n\s*\n', TEXT_VALUE)
    MAP_OUTPUT: Dict[str, Tuple[str, str]] = {}
    for EACH_BLOCK in LIST_BLOCK:
        LIST_LINE = [EACH_LINE.strip() for EACH_LINE in EACH_BLOCK.splitlines() if EACH_LINE.strip()]
        STRING_INSTRUCTION = None
        STRING_FORMAT = None
        for EACH_LINE in LIST_LINE:
            if EACH_LINE.startswith('INSTRUCTION:'):
                STRING_INSTRUCTION = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('FORMAT:'):
                STRING_FORMAT = EACH_LINE.split(':', 1)[1].strip()
        if STRING_INSTRUCTION and STRING_FORMAT:
            MAP_OUTPUT[STRING_INSTRUCTION.lower()] = (STRING_INSTRUCTION, STRING_FORMAT)
    return MAP_OUTPUT

def FUNCTION_PARSE_SUMMARY_FILE_INSTRUCTION(TEXT_VALUE: str) -> Dict[str, str]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n')
    MAP_OUTPUT: Dict[str, str] = {}
    INSTRUCTION_CURRENT: Optional[str] = None
    IS_COLLECTING = False
    LIST_BUFFER: List[str] = []
    def FUNCTION_FLUSH():
        nonlocal INSTRUCTION_CURRENT, IS_COLLECTING, LIST_BUFFER
        if INSTRUCTION_CURRENT and LIST_BUFFER:
            MAP_OUTPUT[INSTRUCTION_CURRENT.lower()] = ' '.join(EACH_LINE.strip() for EACH_LINE in LIST_BUFFER if EACH_LINE.strip()).strip()
        INSTRUCTION_CURRENT = None
        IS_COLLECTING = False
        LIST_BUFFER = []
    for EACH_RAW_LINE in TEXT_VALUE.splitlines():
        EACH_LINE = EACH_RAW_LINE.strip()
        if not EACH_LINE:
            if IS_COLLECTING:
                FUNCTION_FLUSH()
            continue
        if EACH_LINE.startswith('INSTRUCTION:'):
            FUNCTION_FLUSH()
            INSTRUCTION_CURRENT = EACH_LINE.split(':', 1)[1].strip()
            continue
        if EACH_LINE.startswith('SUMMARY:'):
            IS_COLLECTING = True
            LIST_BUFFER.append(EACH_LINE.split(':', 1)[1].strip())
            continue
        if IS_COLLECTING:
            LIST_BUFFER.append(EACH_LINE)
    FUNCTION_FLUSH()
    return MAP_OUTPUT

def FUNCTION_PARSE_SUMMARY_FILE_REGISTER(TEXT_VALUE: str) -> Dict[str, str]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n').strip()
    LIST_BLOCK = re.split(r'\n\s*\n', TEXT_VALUE)
    MAP_OUTPUT: Dict[str, str] = {}
    for EACH_BLOCK in LIST_BLOCK:
        LIST_LINE = [EACH_LINE.strip() for EACH_LINE in EACH_BLOCK.splitlines() if EACH_LINE.strip()]
        STRING_REGISTER = None
        STRING_SUMMARY = None
        for EACH_LINE in LIST_LINE:
            if EACH_LINE.startswith('REGISTER:'):
                STRING_REGISTER = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('SUMMARY:'):
                STRING_SUMMARY = EACH_LINE.split(':', 1)[1].strip()
        if STRING_REGISTER and STRING_SUMMARY:
            MAP_OUTPUT[STRING_REGISTER.lower()] = STRING_SUMMARY
    return MAP_OUTPUT

def FUNCTION_PARSE_RULE_ITEM(TEXT_VALUE: str) -> List[RuleItem]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n')
    LIST_LINE = TEXT_VALUE.splitlines()
    KIND_CURRENT: Optional[str] = None
    NAME_CURRENT: Optional[str] = None
    NUMBER_CURRENT: Optional[str] = None
    FIELDS_CURRENT: Optional[str] = None
    FORMAT_CURRENT: Optional[str] = None
    LIST_ITEM: List[RuleItem] = []
    EACH_INDEX = 0
    while EACH_INDEX < len(LIST_LINE):
        STRING_LINE = LIST_LINE[EACH_INDEX].strip()
        if not STRING_LINE:
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('REGISTER:'):
            KIND_CURRENT = 'REGISTER'
            NAME_CURRENT = STRING_LINE.split(':', 1)[1].strip()
            NUMBER_CURRENT = None
            FIELDS_CURRENT = None
            FORMAT_CURRENT = None
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('INSTRUCTION:'):
            KIND_CURRENT = 'INSTRUCTION'
            NAME_CURRENT = STRING_LINE.split(':', 1)[1].strip()
            FORMAT_CURRENT = None
            NUMBER_CURRENT = None
            FIELDS_CURRENT = None
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('NUMBER:') and KIND_CURRENT == 'REGISTER':
            NUMBER_CURRENT = STRING_LINE.split(':', 1)[1].strip()
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('FIELDS:') and KIND_CURRENT == 'REGISTER':
            STRING_VALUE = STRING_LINE.split(':', 1)[1].strip()
            FIELDS_CURRENT = None if STRING_VALUE == '-' else STRING_VALUE
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('FORMAT:') and KIND_CURRENT == 'INSTRUCTION':
            FORMAT_CURRENT = STRING_LINE.split(':', 1)[1].strip()
            EACH_INDEX += 1
            continue
        MATCH_RULE = REGEX_RULE_LINE.match(STRING_LINE)
        if MATCH_RULE and KIND_CURRENT and NAME_CURRENT:
            LIST_PART = [MATCH_RULE.group(1).strip()]
            EACH_NEXT_INDEX = EACH_INDEX + 1
            while EACH_NEXT_INDEX < len(LIST_LINE):
                STRING_NEXT = LIST_LINE[EACH_NEXT_INDEX].strip()
                if not STRING_NEXT:
                    break
                if (STRING_NEXT.startswith('RULE ') or STRING_NEXT.startswith('REGISTER:') or STRING_NEXT.startswith('INSTRUCTION:') or STRING_NEXT.startswith('NUMBER:') or STRING_NEXT.startswith('FIELDS:') or STRING_NEXT.startswith('FORMAT:')):
                    break
                LIST_PART.append(STRING_NEXT)
                EACH_NEXT_INDEX += 1
            STRING_RULE_TEXT = ' '.join([EACH_PART for EACH_PART in LIST_PART if EACH_PART]).strip()
            LIST_ITEM.append(RuleItem(kind=KIND_CURRENT, name=NAME_CURRENT, number=NUMBER_CURRENT, fields=FIELDS_CURRENT, fmt=FORMAT_CURRENT, rule_text=STRING_RULE_TEXT))
            EACH_INDEX = EACH_NEXT_INDEX
            continue
        EACH_INDEX += 1
    return LIST_ITEM

def FUNCTION_BUILD_BACKGROUND(ITEM: RuleItem, MAP_REGISTER: Dict[str, Tuple[str, str, Optional[str], Optional[str]]], MAP_INSTRUCTION: Dict[str, Tuple[str, str]], MAP_REGISTER_SUMMARY: Dict[str, str], MAP_INSTRUCTION_SUMMARY: Dict[str, str], LIST_PATTERN_REGISTER: List[Tuple[str, re.Pattern]], LIST_PATTERN_INSTRUCTION: List[Tuple[str, re.Pattern]]) -> Tuple[str, str]:
    LIST_BACKGROUND_LINE: List[str] = []
    SET_FIELD_NAME = FUNCTION_PARSE_FIELD_NAME_SET(ITEM.fields if ITEM.kind == 'REGISTER' else None)
    if ITEM.kind == 'REGISTER':
        if ITEM.name:
            LIST_BACKGROUND_LINE.append(f'REGISTER: {ITEM.name}')
        if ITEM.number:
            LIST_BACKGROUND_LINE.append(f'CSR ADDRESS: {ITEM.number}')
        if ITEM.fields:
            LIST_BACKGROUND_LINE.append(f'FIELDS: {ITEM.fields}')
        STRING_KEY = ITEM.name.lower()
        if STRING_KEY in MAP_REGISTER:
            _, _, _, STRING_MODE = MAP_REGISTER[STRING_KEY]
            if STRING_MODE:
                LIST_BACKGROUND_LINE.append(f'MINIMUM PRIVILEGE REQUIRED TO ACCESS THIS REGISTER: {STRING_MODE}')
        if STRING_KEY in MAP_REGISTER_SUMMARY:
            LIST_BACKGROUND_LINE.append(f'SUMMARY: {MAP_REGISTER_SUMMARY[STRING_KEY]}')
    else:
        LIST_BACKGROUND_LINE.append(f'INSTRUCTION: {ITEM.name}')
        if ITEM.fmt:
            LIST_BACKGROUND_LINE.append(f'FORMAT: {ITEM.fmt}')

        STRING_KEY = ITEM.name.lower()

        # NEW: immediate range line (only for instructions in the dict)
        STRING_RANGE = FUNCTION_FORMAT_IMMEDIATE_RANGE(STRING_KEY)
        if STRING_RANGE:
            LIST_BACKGROUND_LINE.append(STRING_RANGE)

        if STRING_KEY in MAP_INSTRUCTION_SUMMARY:
            LIST_BACKGROUND_LINE.append(f'SUMMARY: {MAP_INSTRUCTION_SUMMARY[STRING_KEY]}')
    STRING_RULE_LOWER = ITEM.rule_text.lower()
    STRING_MAIN_REGISTER = ITEM.name.lower() if ITEM.kind == 'REGISTER' else None
    for EACH_REGISTER_LOWER, EACH_PATTERN in LIST_PATTERN_REGISTER:
        if STRING_MAIN_REGISTER and EACH_REGISTER_LOWER == STRING_MAIN_REGISTER:
            continue
        if EACH_REGISTER_LOWER in SET_FIELD_NAME:
            continue
        if EACH_PATTERN.search(STRING_RULE_LOWER):
            STRING_OTHER_NAME, STRING_OTHER_HEX, STRING_OTHER_FIELDS, STRING_OTHER_MODE = MAP_REGISTER[EACH_REGISTER_LOWER]
            LIST_BACKGROUND_LINE.append('')
            LIST_BACKGROUND_LINE.append(f'REGISTER: {STRING_OTHER_NAME}')
            LIST_BACKGROUND_LINE.append(f'CSR ADDRESS: {STRING_OTHER_HEX}')
            if STRING_OTHER_FIELDS:
                LIST_BACKGROUND_LINE.append(f'FIELDS: {STRING_OTHER_FIELDS}')
            if STRING_OTHER_MODE:
                LIST_BACKGROUND_LINE.append(
                    f'MINIMUM PRIVILEGE REQUIRED TO ACCESS THIS REGISTER: {STRING_OTHER_MODE}'
                )
    STRING_MAIN_INSTRUCTION = ITEM.name.lower() if ITEM.kind == 'INSTRUCTION' else None
    for EACH_INSTRUCTION_LOWER, EACH_PATTERN in LIST_PATTERN_INSTRUCTION:
        if STRING_MAIN_INSTRUCTION and EACH_INSTRUCTION_LOWER == STRING_MAIN_INSTRUCTION:
            continue
        if EACH_INSTRUCTION_LOWER in SET_FIELD_NAME:
            continue
        if EACH_PATTERN.search(ITEM.rule_text):
            STRING_INSTRUCTION_NAME, STRING_INSTRUCTION_FORMAT = MAP_INSTRUCTION[EACH_INSTRUCTION_LOWER]
            LIST_BACKGROUND_LINE.append('')
            LIST_BACKGROUND_LINE.append(f'INSTRUCTION: {STRING_INSTRUCTION_NAME}')
            LIST_BACKGROUND_LINE.append(f'FORMAT: {STRING_INSTRUCTION_FORMAT}')

            # NEW: range for referenced instruction too
            STRING_SUMMARY_KEY = STRING_INSTRUCTION_NAME.lower()
            STRING_RANGE = FUNCTION_FORMAT_IMMEDIATE_RANGE(STRING_SUMMARY_KEY)
            if STRING_RANGE:
                LIST_BACKGROUND_LINE.append(STRING_RANGE)

            if STRING_SUMMARY_KEY in MAP_INSTRUCTION_SUMMARY:
                LIST_BACKGROUND_LINE.append(f'SUMMARY: {MAP_INSTRUCTION_SUMMARY[STRING_SUMMARY_KEY]}')
    STRING_BACKGROUND = '\n'.join(LIST_BACKGROUND_LINE).strip()
    STRING_EXAMPLE_NAME = ITEM.name if ITEM.name else 'UNKNOWN'
    return STRING_BACKGROUND, STRING_EXAMPLE_NAME

def FUNCTION_REVISED_JOIN_ONLY(STRING_MODEL_TEXT: str, STRING_RULE_TEXT: str) -> str:
    STRING_TEXT = STRING_MODEL_TEXT.replace('\r\n', '\n')

    # helper: normalize inline comment spacing => "code # comment"
    def FUNCTION_NORMALIZE_INLINE_COMMENT(LINE: str) -> str:
        LINE = ' '.join(LINE.strip().split())
        if not LINE:
            return ''
        if LINE.startswith('#'):
            return LINE  # header/result handled elsewhere
        # split at first '#'
        POS = LINE.find('#')
        if POS == -1:
            return LINE
        CODE = LINE[:POS].rstrip()
        CMT = LINE[POS+1:].strip()
        if not CODE:
            return ''  # drop comment-only lines inside body
        if not CMT:
            return CODE
        return f'{CODE} # {CMT}'

    LIST_LINE = STRING_TEXT.splitlines()

    LIST_BLOCK = []  # (header_line, instr_lines, result_line)
    HEADER_CURRENT = None
    INSTR_CURRENT = []
    RESULT_CURRENT = None

    # accept both:
    # "# Extension Must Be Active: X"
    # "# 51 # Extension Must Be Active: X"
    REGEX_EXTENSION_ANY = re.compile(
        r'(?i)^#\s*(?:\d+\s*)?(?:#\s*)?Extension\s+Must\s+Be\s+Active:\s*(.+?)\s*$'
    )
    REGEX_RESULT_ANY = re.compile(
        r'(?i)^#\s*Result\s*:\s*(.+?)\s*$'
    )

    def FUNCTION_FLUSH():
        nonlocal HEADER_CURRENT, INSTR_CURRENT, RESULT_CURRENT
        if HEADER_CURRENT and RESULT_CURRENT:
            # remove empty lines inside body
            BODY = [LN for LN in INSTR_CURRENT if LN.strip()]
            LIST_BLOCK.append((HEADER_CURRENT, BODY, RESULT_CURRENT))
        HEADER_CURRENT = None
        INSTR_CURRENT = []
        RESULT_CURRENT = None

    for RAW in LIST_LINE:
        LINE = RAW.strip()
        if not LINE:
            continue  # IMPORTANT: no empty line anywhere

        M_EXT = REGEX_EXTENSION_ANY.match(LINE)
        if M_EXT:
            FUNCTION_FLUSH()
            EXT = M_EXT.group(1).strip()
            # keep same header style as your canonical output (no double #)
            HEADER_CURRENT = f'# Extension Must Be Active: {EXT}'
            continue

        M_RES = REGEX_RESULT_ANY.match(LINE)
        if M_RES:
            if RESULT_CURRENT is None:
                RESULT_CURRENT = f'# Result: {M_RES.group(1).strip()}'
            continue

        # ignore other comment-only lines (besides extension/result)
        if LINE.startswith('#'):
            continue

        NORMAL = FUNCTION_NORMALIZE_INLINE_COMMENT(LINE)
        if NORMAL:
            INSTR_CURRENT.append(NORMAL)

    FUNCTION_FLUSH()

    # extension correction logic tetap boleh (kayak fungsi lama kamu)
    STRING_RULE_BODY = (STRING_RULE_TEXT or '')
    LIST_FIXED = []
    for HEADER, BODY, RESULT in LIST_BLOCK:
        EXT = HEADER.split(':', 1)[1].strip() if ':' in HEADER else 'NONE'
        if EXT.upper() != 'NONE':
            if not re.search(r'\b' + re.escape(EXT) + r'\b', STRING_RULE_BODY):
                HEADER = '# Extension Must Be Active: NONE'
        LIST_FIXED.append((HEADER, BODY, RESULT))

    # final write: no blank line between header/body/result,
    # and only ONE blank line between test cases? -> kamu minta "jangan ada empty line di antaranya"
    # jadi kita pisahin antar test case pakai 1 newline saja (tanpa baris kosong).
    OUT = []
    for HEADER, BODY, RESULT in LIST_FIXED:
        OUT.append(HEADER)
        OUT.extend(BODY)
        OUT.append(RESULT)

    return '\n'.join(OUT).strip()

def FUNCTION_HISTORY_WRAP(GLOBAL_NUMBER: int, LOCAL_NUMBER: int, STRING_NAME: str, STRING_PROMPT: str, STRING_RESPONSE: str) -> str:
    STRING_INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH_PROMPT, '=')
    STRING_INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH_PROMPT, '=')
    STRING_MID = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * (WIDTH_PROMPT // 2)}
=======
PROMPT:
=======
{'=' * (WIDTH_PROMPT // 2)}
'''
    STRING_TAIL = f'''
{'=' * (WIDTH_PROMPT // 2)}
=========
RESPONSE:
=========
{'=' * (WIDTH_PROMPT // 2)}
{STRING_RESPONSE}
'''
    return STRING_INTRO_FIRST + '\n' + STRING_MID + STRING_PROMPT + STRING_TAIL + STRING_INTRO_END + STRING_SUFFIX

def FUNCTION_RENUMBER_EXTENSION_HEADER_INPLACE(FILE_PATH: str):
    STRING_TEXT = open(FILE_PATH, 'r', encoding='utf-8').read().replace('\r\n', '\n')
    COUNTER_INDEX = 0
    def FUNCTION_REPLACE(MATCH_VALUE: re.Match) -> str:
        nonlocal COUNTER_INDEX
        COUNTER_INDEX += 1
        return f'# {COUNTER_INDEX} {MATCH_VALUE.group(0)}'
    STRING_REVISED = re.sub(r'(?m)^#\s*Extension Must Be Active:.*$', FUNCTION_REPLACE, STRING_TEXT)
    with open(FILE_PATH, 'w', encoding='utf-8', newline='\n') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_REVISED)

def FUNCTION_PARSE_FIELD_NAME_SET(STRING_FIELDS: Optional[str]) -> set[str]:
    if not STRING_FIELDS:
        return set()
    SET_NAME: set[str] = set()
    for EACH_PART in STRING_FIELDS.split(','):
        EACH_PART = EACH_PART.strip()
        if not EACH_PART:
            continue
        if '=' in EACH_PART:
            NAME = EACH_PART.split('=', 1)[1].strip()
            if NAME:
                SET_NAME.add(NAME.lower())
    return SET_NAME

def FUNCTION_GET_IMMEDIATE_RANGE(STRING_MNEMONIC_LOWER: str):
    # 1) scalar explicit map
    TUP = MAP_IMMEDIATE_RANGE.get(STRING_MNEMONIC_LOWER)
    if TUP:
        return TUP

    # 2) vector immediate by suffix
    if STRING_MNEMONIC_LOWER.endswith('.vi'):
        return MAP_VECTOR_IMM_RANGE['.vi']

    return None

def FUNCTION_FORMAT_IMMEDIATE_RANGE(STRING_MNEMONIC_LOWER: str):
    TUP = FUNCTION_GET_IMMEDIATE_RANGE(STRING_MNEMONIC_LOWER)
    if not TUP:
        return None
    MIN_V, MAX_V = TUP
    return f'RANGE FOR IMMEDIATE: {MIN_V} until {MAX_V}'
# FUNCTION - FINISH

# VARIABLE - BEGIN
MAP_IMMEDIATE_RANGE = {
    'addi': (-2048, 2047), 'addiw': (-2048, 2047),
    'xori': (-2048, 2047), 'ori': (-2048, 2047), 'andi': (-2048, 2047),
    'slti': (-2048, 2047), 'sltiu': (-2048, 2047),
    'slli': (0, 63), 'srli': (0, 63), 'srai': (0, 63),
    'slliw': (0, 31), 'srliw': (0, 31), 'sraiw': (0, 31),
    'lui': (0, (1 << 20) - 1), 'auipc': (0, (1 << 20) - 1),
}
MAP_VECTOR_IMM_RANGE = {
    '.vi': (-16, 15),  # OPIVI simm5
}
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 16384
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
IS_LOCAL = False
DOES_SKIP_VIRTUAL = True
DOES_WANT_VIRTUAL = False
WIDTH_PROMPT = 120
STRING_DIVIDER = (('#' * WIDTH_PROMPT) + '\n') * 3
STRING_DIVIDER = STRING_DIVIDER.rstrip('\n')
STRING_SUFFIX = '\n' + STRING_DIVIDER + '\n'
REGEX_REGISTER_HEADER = re.compile(r'(?m)^REGISTER:\s*(.+?)\s*$')
REGEX_NUMBER = re.compile(r'(?m)^NUMBER:\s*(.+?)\s*$')
REGEX_FIELDS = re.compile(r'(?m)^FIELDS:\s*(.+?)\s*$')
REGEX_INSTRUCTION_HEADER = re.compile(r'(?m)^INSTRUCTION:\s*(.+?)\s*$')
REGEX_FORMAT = re.compile(r'(?m)^FORMAT:\s*(.+?)\s*$')
REGEX_RULE_LINE = re.compile(r'^RULE\s+\d+:\s*(.*)$')
REGEX_EXTENSION_HEADER = re.compile(r'(?i)^#\s*Extension\s+Must\s+Be\s+Active:\s*(.+?)\s*$')
REGEX_RESULT_LINE = re.compile(r'(?i)^#\s*Result\s*:\s*(.+?)\s*$')
STRING_INPUT_TEXT = open(FILE_INPUT, 'r', encoding='utf-8').read().strip()
STRING_REGISTER_ALL_TEXT = open(FILE_REGISTER_ALL, 'r', encoding='utf-8').read().strip()
STRING_INSTRUCTION_ALL_TEXT = open(FILE_INSTRUCTION_ALL, 'r', encoding='utf-8').read().strip()
STRING_INSTRUCTION_SUMMARY_TEXT = open(FILE_INSTRUCTION_SUMMARY, 'r', encoding='utf-8').read()
STRING_REGISTER_SUMMARY_TEXT = open(FILE_REGISTER_SUMMARY, 'r', encoding='utf-8').read()
MAP_REGISTER = FUNCTION_PARSE_REGISTER_NAME_FILE(STRING_REGISTER_ALL_TEXT)
MAP_INSTRUCTION = FUNCTION_PARSE_INSTRUCTION_NAME_FILE(STRING_INSTRUCTION_ALL_TEXT)
MAP_INSTRUCTION_SUMMARY = FUNCTION_PARSE_SUMMARY_FILE_INSTRUCTION(STRING_INSTRUCTION_SUMMARY_TEXT)
MAP_REGISTER_SUMMARY = FUNCTION_PARSE_SUMMARY_FILE_REGISTER(STRING_REGISTER_SUMMARY_TEXT)
LIST_PATTERN_REGISTER = [(EACH_KEY, FUNCTION_WHOLE_WORD_PATTERN(EACH_KEY)) for EACH_KEY in MAP_REGISTER.keys()]
LIST_PATTERN_INSTRUCTION = []
for EACH_KEY in MAP_INSTRUCTION.keys():
    if EACH_KEY in ('and', 'or'):
        # hanya match token "AND"/"OR" uppercase (bukan kata and/or biasa)
        LIST_PATTERN_INSTRUCTION.append(
            (EACH_KEY, FUNCTION_WHOLE_WORD_PATTERN(EACH_KEY.upper(), 0))
        )
    else:
        LIST_PATTERN_INSTRUCTION.append(
            (EACH_KEY, FUNCTION_WHOLE_WORD_PATTERN(EACH_KEY, re.IGNORECASE))
        )
LIST_OUTPUT_CLEAN: List[str] = []
LIST_OUTPUT_RAW: List[str] = []
STRING_PROMPT_HISTORY = ''
GLOBAL_COUNTER = 1
LOCAL_COUNTER = 1
STRING_NAME_CURRENT: Optional[str] = None
IS_FIRST_SEEN = True
PATTERN_VIRTUAL = r'''(?:VS|VU)(?:\s*-\s*|\s+)mode|virtual\s*-?\s*mode|virtual(?:\s*-\s*|\s+)(?:supervisor|user)'''
PC_INITIAL = 0x800001e6
STRING_PROMPT_TEMPLATE = '''For RISC-V, based on the rule below, generate me 1 test case in this format:
# Extension Must Be Active: <extension>
<contiguous assembly code>
# Result: <result>

Do not output any introduction and follow the output format above!

####################
THE RULE:
{rule}{bg_block}
####################

Note:
+ <result> must be a single register condition.
+ <result> must be in this format: <register> <operator> <value>.
+ <register> should be general-purpose registers (it can be floating-point registers also, or vector registers).
+ <operator> can either be '=' or '!='.
+ <value> can either be hexadecimal or decimal.
+ There can be multiple <result> (for example, # Result: sstatus != 0xffffffff & sstatus != 0xffffffffffffffff) (another example, # Result: a0 = 0 | a1 = 0).
+ The initial values of all general-purpose registers, or floating, or vector registers, are 0.
+ Since the initial values of many CSR fields are unknown and may contain implementation-defined bits, generate test cases that isolate the target field using a bitwise AND mask on the CSR readback value instead of checking the full CSR value directly.
+ <extension> must be NONE if the rule doesn't explicitly say about any extension required.
+ The kernel always starts in M-mode.
+ If you are from M-mode and need to go to S-mode or HS-mode, do not use MRET instruction; instead, use this assembly code -> 'call machine_to_supervisor'. This function already has MRET instruction. Do not make a label or branch called "machine_to_supervisor:".
+ If you are from M-mode and need to go to U-mode, do not use MRET instruction; instead, use this assembly code -> 'call machine_to_user'. This function already has MRET instruction. Do not make a label or branch called "machine_to_user:".
+ If you are from M-mode and need to go to VS-mode, do not USE MRET instruction; instead, use this assembly code -> 'call machine_to_virtual'. Do not make a label or branch called "machine_to_virtual:".
+ DO NOT USE '.globl' and '_start'; assume all boilerplate is already set up!
+ Do not use assembler directives or pseudo-instruction encodings such as .inst, .word, .dword, .byte, .half, .quad, or relocation expressions like %pcrel_hi, %pcrel_lo, %hi, %lo; use only standard RISC-V assembly instructions and registers.
+ The kernel info: XLEN = 64; RISC-V 64-bit (RV64); PC initial value is {PC_INITIAL}.
+ If a test case triggers a trap, execution stops at the trap. Basically, any other instructions should be performed before the trapping instruction.
+ Read THE BACKGROUND INFORMATION.'''
# VARIABLE - FINISH

# CODE - BEGIN
@dataclass
class RuleItem:
    kind: str
    name: str
    number: Optional[str]
    fields: Optional[str]
    fmt: Optional[str]
    rule_text: str

LIST_ITEM = FUNCTION_PARSE_RULE_ITEM(STRING_INPUT_TEXT)

for EACH_ITEM in LIST_ITEM:
    if DOES_SKIP_VIRTUAL and re.search(PATTERN_VIRTUAL, EACH_ITEM.rule_text, re.IGNORECASE):
        continue
    if DOES_WANT_VIRTUAL and not re.search(PATTERN_VIRTUAL, EACH_ITEM.rule_text, re.IGNORECASE):
        continue
    STRING_BACKGROUND, STRING_EXAMPLE_NAME = FUNCTION_BUILD_BACKGROUND(ITEM=EACH_ITEM, MAP_REGISTER=MAP_REGISTER, MAP_INSTRUCTION=MAP_INSTRUCTION, MAP_REGISTER_SUMMARY=MAP_REGISTER_SUMMARY, MAP_INSTRUCTION_SUMMARY=MAP_INSTRUCTION_SUMMARY, LIST_PATTERN_REGISTER=LIST_PATTERN_REGISTER, LIST_PATTERN_INSTRUCTION=LIST_PATTERN_INSTRUCTION)
    if IS_FIRST_SEEN:
        STRING_NAME_CURRENT = STRING_EXAMPLE_NAME
        LOCAL_COUNTER = 1
        IS_FIRST_SEEN = False
    else:
        if STRING_NAME_CURRENT != STRING_EXAMPLE_NAME:
            STRING_NAME_CURRENT = STRING_EXAMPLE_NAME
            LOCAL_COUNTER = 1
    STRING_BACKGROUND_BLOCK = ''
    if STRING_BACKGROUND:
        STRING_BACKGROUND_BLOCK = (f'\n\nTHE BACKGROUND INFORMATION:\n##########\n{STRING_BACKGROUND}\n##########')
    STRING_PROMPT = STRING_PROMPT_TEMPLATE.format(rule=EACH_ITEM.rule_text, bg_block=STRING_BACKGROUND_BLOCK, PC_INITIAL=PC_INITIAL)
    STRING_RAW = FUNCTION_CALL_LLM(STRING_PROMPT)
    STRING_RAW = FUNCTION_SANITIZE_TEXT(STRING_RAW)
    LIST_OUTPUT_RAW.append('\n'.join(' '.join(EACH_LINE.split()) for EACH_LINE in STRING_RAW.splitlines()))
    STRING_REVISED = FUNCTION_REVISED_JOIN_ONLY(STRING_RAW, EACH_ITEM.rule_text)
    LIST_OUTPUT_CLEAN.append(STRING_REVISED)
    STRING_PROMPT_HISTORY += FUNCTION_HISTORY_WRAP(GLOBAL_COUNTER, LOCAL_COUNTER, STRING_EXAMPLE_NAME, STRING_PROMPT, STRING_RAW)
    GLOBAL_COUNTER += 1
    LOCAL_COUNTER += 1

if STRING_PROMPT_HISTORY.endswith(STRING_SUFFIX):
    STRING_PROMPT_HISTORY = STRING_PROMPT_HISTORY[:-len(STRING_SUFFIX)]

with open(FILE_OUTPUT_HISTORY, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY)

with open(FILE_OUTPUT_REVISED, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT_CLEAN))

with open(FILE_OUTPUT_RAW, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT_RAW))

FUNCTION_RENUMBER_EXTENSION_HEADER_INPLACE(FILE_OUTPUT_RAW)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 2 - CREATE - 2_TEST_CASE_RAW.txt & 3_TEST_CASE_REVISED.txt - END

1 hour(s) 26 minute(s) 18.66 second(s)
====
DONE
====


In [20]:
import shutil
file_a = '6_TEST_CASE/3_TEST_CASE_REVISED.txt'
file_b = '6_TEST_CASE/3_TEST_CASE_REVISED_1.txt'
shutil.copyfile(file_a, file_b)     # content only

import re

# -----------------------------
# CONFIG
# -----------------------------
MAP_IMMEDIATE_RANGE = {
    'addi': (-2048, 2047), 'addiw': (-2048, 2047),
    'xori': (-2048, 2047), 'ori': (-2048, 2047), 'andi': (-2048, 2047),
    'slti': (-2048, 2047), 'sltiu': (-2048, 2047),
    'slli': (0, 63), 'srli': (0, 63), 'srai': (0, 63),
    'slliw': (0, 31), 'srliw': (0, 31), 'sraiw': (0, 31),
    'lui': (0, (1 << 20) - 1), 'auipc': (0, (1 << 20) - 1),
}

CSR_MNEMS = {
    'csrr', 'csrw', 'csrs', 'csrc',
    'csrrw', 'csrrs', 'csrrc',
    'csrrwi', 'csrrsi', 'csrrci',
}

# RISC-V ABI aliases (rv64)
REG_ALIAS = {
    **{f'x{i}': i for i in range(32)},
    'zero': 0, 'ra': 1, 'sp': 2, 'gp': 3, 'tp': 4,
    't0': 5, 't1': 6, 't2': 7,
    's0': 8, 'fp': 8, 's1': 9,
    'a0': 10, 'a1': 11, 'a2': 12, 'a3': 13, 'a4': 14, 'a5': 15, 'a6': 16, 'a7': 17,
    's2': 18, 's3': 19, 's4': 20, 's5': 21, 's6': 22, 's7': 23, 's8': 24, 's9': 25, 's10': 26, 's11': 27,
    't3': 28, 't4': 29, 't5': 30, 't6': 31,
}

# -----------------------------
# BIT HELPERS
# -----------------------------
MASK64 = (1 << 64) - 1
MASK32 = (1 << 32) - 1

def u64(x: int) -> int:
    return x & MASK64

def u32(x: int) -> int:
    return x & MASK32

def s64(x: int) -> int:
    x &= MASK64
    return x - (1 << 64) if (x >> 63) & 1 else x

def s32(x: int) -> int:
    x &= MASK32
    return x - (1 << 32) if (x >> 31) & 1 else x

def sext(x: int, bits: int) -> int:
    mask = (1 << bits) - 1
    x &= mask
    sign = 1 << (bits - 1)
    return (x ^ sign) - sign

def sext32_to_64(x: int) -> int:
    return u64(sext(x, 32))

def mul_low_xlen(a: int, b: int) -> int:
    return u64(u64(a) * u64(b))

def mulh_signed(a: int, b: int) -> int:
    prod = s64(a) * s64(b)
    return u64(prod >> 64)

def mulh_su(a: int, b: int) -> int:
    prod = s64(a) * u64(b)
    return u64(prod >> 64)

def mulh_unsigned(a: int, b: int) -> int:
    prod = u64(a) * u64(b)
    return u64(prod >> 64)

def div_signed(a: int, b: int) -> int:
    a_s = s64(a)
    b_s = s64(b)
    if b_s == 0:
        return u64(-1)
    if a_s == -(1 << 63) and b_s == -1:
        return u64(a_s)  # overflow case
    return u64(int(a_s / b_s))  # trunc toward 0 in Python for ints

def div_unsigned(a: int, b: int) -> int:
    a_u = u64(a)
    b_u = u64(b)
    if b_u == 0:
        return MASK64
    return u64(a_u // b_u)

def rem_signed(a: int, b: int) -> int:
    a_s = s64(a)
    b_s = s64(b)
    if b_s == 0:
        return u64(a_s)
    if a_s == -(1 << 63) and b_s == -1:
        return u64(0)
    q = int(a_s / b_s)
    r = a_s - q * b_s
    return u64(r)

def rem_unsigned(a: int, b: int) -> int:
    a_u = u64(a)
    b_u = u64(b)
    if b_u == 0:
        return u64(a_u)
    return u64(a_u % b_u)

def divw_signed(a: int, b: int) -> int:
    a_s = s32(a)
    b_s = s32(b)
    if b_s == 0:
        return u64(-1)  # -1 sign-extended to 64 => 0xffff...
    if a_s == -(1 << 31) and b_s == -1:
        return sext32_to_64(a_s)
    return sext32_to_64(int(a_s / b_s))

def divuw(a: int, b: int) -> int:
    a_u = u32(a)
    b_u = u32(b)
    if b_u == 0:
        return sext32_to_64(0xFFFFFFFF)
    return sext32_to_64(a_u // b_u)

def remw_signed(a: int, b: int) -> int:
    a_s = s32(a)
    b_s = s32(b)
    if b_s == 0:
        return sext32_to_64(a_s)
    if a_s == -(1 << 31) and b_s == -1:
        return sext32_to_64(0)
    q = int(a_s / b_s)
    r = a_s - q * b_s
    return sext32_to_64(r)

def remuw(a: int, b: int) -> int:
    a_u = u32(a)
    b_u = u32(b)
    if b_u == 0:
        return sext32_to_64(a_u)
    return sext32_to_64(a_u % b_u)

# -----------------------------
# NUMBER PARSE / FORMAT (preserve base)
# -----------------------------
_NUM_HEX_RE = re.compile(r'^\s*([+-]?)0[xX]([0-9a-fA-F_]+)\s*$')
_NUM_DEC_RE = re.compile(r'^\s*[-+]?\d+\s*$')

def parse_int_token(tok: str) -> int:
    tok = tok.strip()
    m = _NUM_HEX_RE.match(tok)
    if m:
        sign = -1 if m.group(1) == '-' else 1
        return sign * int(m.group(2).replace('_', ''), 16)
    # decimal (possibly negative)
    return int(tok, 10)

def is_hex_token(tok: str) -> bool:
    return bool(_NUM_HEX_RE.match(tok.strip()))

def format_int_like(tok_style: str, value: int) -> str:
    if is_hex_token(tok_style):
        return hex(u64(value))
    # decimal style -> print signed 64-bit
    return str(s64(value))

def clamp(v: int, lo: int, hi: int) -> int:
    return lo if v < lo else hi if v > hi else v

# -----------------------------
# IMMEDIATE FIXING RULES
# -----------------------------
def normalize_imm_for_insn(insn: str, imm_tok: str) -> int:
    """
    Convert immediate token to the *semantic* immediate used by RISC-V decoding:
    - For signed immeds (addi/addiw/andi/ori/xori/slti/sltiu): 12-bit sign-extended.
      If token is hex and fits in 12 bits, interpret as bit-pattern (e.g. 0xfff -> -1).
      Else clamp to range.
    - For shift: clamp to allowed range.
    - For lui/auipc: 20-bit unsigned; clamp.
    """
    insn = insn.lower()
    lo, hi = MAP_IMMEDIATE_RANGE[insn]

    # LUI/AUIPC are 20-bit unsigned immeds (already shifted by hardware, but assembly uses the 20-bit field)
    if insn in ('lui', 'auipc'):
        raw = parse_int_token(imm_tok)
        # treat hex/dec same here; just clamp into 20-bit unsigned range
        return clamp(raw, lo, hi)

    # shifts: unsigned immediate
    if insn in ('slli', 'srli', 'srai', 'slliw', 'srliw', 'sraiw'):
        raw = parse_int_token(imm_tok)
        return clamp(raw, lo, hi)

    # 12-bit signed immed group
    raw = parse_int_token(imm_tok)

    if is_hex_token(imm_tok):
        # If the literal fits in 12 bits, treat it as bit-pattern then sign-extend.
        # This makes 0xfff behave like -1 (important for your example addi ..., 0xfff).
        if 0 <= raw <= 0xFFF:
            return sext(raw, 12)
        # If it's bigger than 12 bits, clamp into legal numeric range.
        return clamp(raw, lo, hi)

    # decimal: clamp numeric value to legal range
    return clamp(raw, lo, hi)

def format_imm_preserve_style(insn: str, imm_tok_style: str, imm_value_semantic: int) -> str:
    insn = insn.lower()

    # decimal input -> keep decimal output
    if not is_hex_token(imm_tok_style):
        return str(int(imm_value_semantic))

    # hex input
    if insn in ('addi', 'addiw', 'andi', 'ori', 'xori', 'slti', 'sltiu'):
        # IMPORTANT: assembler-safe for negative immediates
        # -2048..-1 MUST be printed as decimal negative, not 0x800..0xfff
        if imm_value_semantic < 0:
            return str(int(imm_value_semantic))  # e.g., -2048, -1
        # non-negative stays hex (0..2047), assembler-safe
        return hex(int(imm_value_semantic))      # e.g., 0x7ff, 0x0, 0x10

    # shifts and 20-bit immeds: keep hex numeric
    return hex(int(imm_value_semantic))

def is_numeric_token(tok: str) -> bool:
    tok = tok.strip()
    return bool(_NUM_HEX_RE.match(tok) or _NUM_DEC_RE.match(tok))

# -----------------------------
# PARSING
# -----------------------------
def split_blocks(text: str):
    # split at blank lines where next block starts with:
    #   "# Extension Must Be Active:"  OR  "# <n> # Extension Must Be Active:"
    return [
        b for b in re.split(
            r'\n\s*\n(?=\s*#\s*(?:\d+\s*#\s*)?Extension\s+Must\s+Be\s+Active\s*:)',
            text.strip(),
            flags=re.I | re.M
        )
        if b.strip()
    ]

def strip_comment(line: str) -> str:
    return line.split('#', 1)[0].strip()

def tokenize_asm(line: str):
    # handle: "addi x5, x5, 0xfff" and "li a0,0x123"
    line = strip_comment(line)
    if not line:
        return None
    # normalize commas to spaces
    line = re.sub(r'[,\t]+', ' ', line.strip())
    parts = [p for p in line.split(' ') if p]
    if not parts:
        return None
    return parts  # [mnemonic, op1, op2, ...]

def reg_id(r: str) -> int:
    r = r.strip()
    if r not in REG_ALIAS:
        raise ValueError(f'Unknown register: {r}')
    return REG_ALIAS[r]

MEM_MNEMS = {
    'lb','lbu','lh','lhu','lw','lwu','ld',
    'sb','sh','sw','sd',
    # kalau kamu mau sekalian:
    'flw','fsw','fld','fsd',
}

def block_has_mem(lines):
    for ln in lines:
        parts = tokenize_asm(ln)
        if parts and parts[0].lower() in MEM_MNEMS:
            return True
    return False

def block_has_vector(lines) -> bool:
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        m = parts[0]          # jangan pakai .lower()
        if m.startswith('v'): # hanya huruf 'v' kecil
            return True
    return False

# -----------------------------
# VECTOR IMMEDIATE (heuristic clamp)
# -----------------------------
VECTOR_IMM_RULES = [
    # zimm6 (0..63)
    (re.compile(r'^vror\.vi$'), 0, 63),

    # zimm5 (0..31) common shift/permute-ish
    (re.compile(r'^(v(?:sll|srl|sra|ssll|ssrl|ssra|wsll)\.vi)$'), 0, 31),
    (re.compile(r'^(v(?:slidedown|slideup)\.vi)$'), 0, 31),
    (re.compile(r'^(vrgather\.vi)$'), 0, 31),
    (re.compile(r'^(v(?:nclip|nclipu)\.wi)$'), 0, 31),
    (re.compile(r'^(v(?:nsrl|nsra)\.wi)$'), 0, 31),

    # known zimm5 from your list
    (re.compile(r'^vaeskf[12]\.vi$'), 0, 31),
    (re.compile(r'^vsm3c\.vi$'), 0, 31),
    (re.compile(r'^vsm4k\.vi$'), 0, 31),

    # fallback: most *.vi / *.vim are simm5 (-16..15)
    (re.compile(r'^v.+\.vi(m)?$'), -16, 15),
]

def get_vector_imm_range(mnem: str):
    # case-sensitive: mnem harus 'v...' kecil kalau mau masuk sini
    for rx, lo, hi in VECTOR_IMM_RULES:
        if rx.match(mnem):
            return lo, hi
    return None

def format_vector_imm_preserve_style(old_tok: str, new_val: int) -> str:
    # kalau awalnya hex, keep hex untuk non-negative; kalau negative, amanin pakai decimal
    if is_hex_token(old_tok):
        if new_val < 0:
            return str(int(new_val))
        return hex(int(new_val))
    return str(int(new_val))

def fix_vector_immediate_in_line(line: str) -> str:
    # preserve comment
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''

    parts = tokenize_asm(code)
    if not parts:
        return line

    mnem = parts[0]  # NO .lower() (case-sensitive)
    if not mnem.startswith('v'):
        return line

    rng = get_vector_imm_range(mnem)
    if not rng:
        return line

    lo, hi = rng

    # heuristic: immediate biasanya operand ke-3 untuk *.vi/*.vim/*.wi
    # contoh: vxor.vi vd, vs2, simm5, vm  => parts: [vxor.vi, v3, v2, 0x1f]
    # kita cari token numeric terakhir sebelum optional 'vm'/'v0'
    imm_idx = None
    for i in range(len(parts) - 1, 0, -1):
        t = parts[i]
        if t in ('vm', 'v0'):
            continue
        if is_numeric_token(t):
            imm_idx = i
            break
        # kalau ketemu token non-numeric yang bukan vm/v0, stop
        # (biar gak salah ngerubah register)
        # tapi kalau mau lebih agresif, hapus break ini.
        # break

    if imm_idx is None:
        return line

    old_imm_tok = parts[imm_idx]
    try:
        raw = parse_int_token(old_imm_tok)
    except Exception:
        return line

    new_val = clamp(raw, lo, hi)
    if new_val == raw:
        return line

    parts[imm_idx] = format_vector_imm_preserve_style(old_imm_tok, new_val)

    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

# -----------------------------
# COMPRESSED (RVC) IMMEDIATE (heuristic clamp)
# -----------------------------
COMPRESSED_IMM_RULES = [
    # imm6 signed (-32..31)
    (re.compile(r'^c\.(?:addi|addiw|andi|li)$'), -32, 31),

    # shifts: treat as unsigned shamt (RV64 => 0..63)
    (re.compile(r'^c\.(?:slli|srli|srai)$'), 0, 63),

    # c.lui operand is the "imm" before <<12; real encoding has nzimm18.
    # For safety, clamp to signed 18-bit range (you don't need to enforce non-zero).
    (re.compile(r'^c\.lui$'), -(1 << 17), (1 << 17) - 1),
]

def get_compressed_imm_range(mnem: str):
    # case-insensitive is fine here; but mnemonic in your pipeline is lower anyway
    m = mnem.lower()
    for rx, lo, hi in COMPRESSED_IMM_RULES:
        if rx.match(m):
            return lo, hi
    return None

def fix_compressed_immediate_in_line(line: str) -> str:
    # preserve comment
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''

    parts = tokenize_asm(code)
    if not parts:
        return line

    mnem = parts[0]
    if not mnem.lower().startswith('c.'):
        return line

    rng = get_compressed_imm_range(mnem)
    if not rng:
        return line

    lo, hi = rng

    # heuristic: immediate biasanya operand terakhir untuk c.addi/c.li/c.andi/c.slli/c.srli/c.srai/c.lui/c.addiw
    imm_idx = None
    for i in range(len(parts) - 1, 0, -1):
        if is_numeric_token(parts[i]):
            imm_idx = i
            break
    if imm_idx is None:
        return line

    old = parts[imm_idx]
    try:
        raw = parse_int_token(old)
    except Exception:
        return line

    new_val = clamp(raw, lo, hi)
    if new_val == raw:
        return line

    # preserve style: hex stays hex if non-negative; negative -> decimal (assembler-safe)
    if is_hex_token(old):
        parts[imm_idx] = str(int(new_val)) if new_val < 0 else hex(int(new_val))
    else:
        parts[imm_idx] = str(int(new_val))

    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

def block_has_compressed(lines) -> bool:
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        # case-insensitive, tapi in practice mnemonic kamu lowercase
        if parts[0].lower().startswith('c.'):
            return True
    return False

# -----------------------------
# MINI EXECUTOR (subset you used)
# -----------------------------
def exec_block(lines, PC_START: int = 0):
    regs = [0] * 32
    pc = u64(PC_START)

    def r(i):
        return 0 if i == 0 else regs[i]

    def w(i, v):
        if i != 0:
            regs[i] = u64(v)

    for raw_line in lines:
        parts = tokenize_asm(raw_line)
        if not parts:
            continue

        mnem = parts[0].lower()

        if mnem == 'li':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
        
            # If value fits signed 32-bit, sign-extend like typical RV64 usage
            if -(1 << 31) <= imm <= (1 << 31) - 1:
                w(rd, sext32_to_64(imm))
            else:
                w(rd, imm)
        
            pc = u64(pc + 4)
            continue

        if mnem == 'lui':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            w(rd, u64(imm << 12))
            pc = u64(pc + 4)
            continue

        if mnem == 'auipc':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            val32 = u32(pc + (imm << 12))
            w(rd, val32)          # zero-extend
            pc = u64(pc + 4)
            continue

        if mnem in ('addi', 'andi', 'ori', 'xori', 'slti', 'sltiu'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            imm_tok = parts[3]
            imm = normalize_imm_for_insn(mnem, imm_tok)

            if mnem == 'addi':
                w(rd, r(rs1) + imm)
            elif mnem == 'andi':
                w(rd, r(rs1) & u64(imm))
            elif mnem == 'ori':
                w(rd, r(rs1) | u64(imm))
            elif mnem == 'xori':
                w(rd, r(rs1) ^ u64(imm))
            elif mnem == 'slti':
                w(rd, 1 if s64(r(rs1)) < int(imm) else 0)
            elif mnem == 'sltiu':
                w(rd, 1 if u64(r(rs1)) < u64(imm) else 0)

            pc = u64(pc + 4)
            continue

        if mnem == 'addiw':
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            imm_tok = parts[3]
            imm = normalize_imm_for_insn(mnem, imm_tok)
            res32 = (r(rs1) + imm) & MASK32
            w(rd, sext32_to_64(res32))
            pc = u64(pc + 4)
            continue

        if mnem in ('slli', 'srli', 'srai', 'slliw', 'srliw', 'sraiw'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            sh_tok = parts[3]
            sh = normalize_imm_for_insn(mnem, sh_tok)

            if mnem == 'slli':
                w(rd, r(rs1) << sh)
            elif mnem == 'srli':
                w(rd, u64(r(rs1)) >> sh)
            elif mnem == 'srai':
                w(rd, u64(s64(r(rs1)) >> sh))
            elif mnem == 'slliw':
                res32 = (u32(r(rs1)) << sh) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'srliw':
                res32 = (u32(r(rs1)) >> sh) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'sraiw':
                res32 = s32(r(rs1)) >> sh
                w(rd, sext32_to_64(res32))

            pc = u64(pc + 4)
            continue

        # pseudo: sll/srl/sra rd, rs1, shamt  (numeric operand3) -> treat like slli/srli/srai
        if mnem in ('sll', 'srl', 'sra') and len(parts) >= 4 and is_numeric_token(parts[3]):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            if rd is None or rs1 is None:
                pc = u64(pc + 4)
                continue
        
            sh = parse_int_token(parts[3]) & 0x3F  # RV64
            if mnem == 'sll':
                w(rd, u64(r(rs1)) << sh)
            elif mnem == 'srl':
                w(rd, u64(r(rs1)) >> sh)
            else:  # sra
                w(rd, u64(s64(r(rs1)) >> sh))
        
            pc = u64(pc + 4)
            continue
            
        # -----------------------------
        # RV64I R-type (register-register)
        # -----------------------------
        if mnem in ('add', 'sub', 'and', 'or', 'xor', 'sll', 'srl', 'sra', 'slt', 'sltu'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            rs2 = reg_id(parts[3])
            a = r(rs1)
            b = r(rs2)

            if mnem == 'add':
                w(rd, a + b)
            elif mnem == 'sub':
                w(rd, a - b)
            elif mnem == 'and':
                w(rd, a & b)
            elif mnem == 'or':
                w(rd, a | b)
            elif mnem == 'xor':
                w(rd, a ^ b)
            elif mnem == 'sll':
                w(rd, u64(a) << (u64(b) & 0x3F))
            elif mnem == 'srl':
                w(rd, u64(a) >> (u64(b) & 0x3F))
            elif mnem == 'sra':
                w(rd, u64(s64(a) >> (u64(b) & 0x3F)))
            elif mnem == 'slt':
                w(rd, 1 if s64(a) < s64(b) else 0)
            elif mnem == 'sltu':
                w(rd, 1 if u64(a) < u64(b) else 0)

            pc = u64(pc + 4)
            continue
        
        # -----------------------------
        # Bitmanip Zbb (subset): andn / orn / xnor (optional)
        # -----------------------------
        if mnem in ('andn', 'orn', 'xnor'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            rs2 = reg_id(parts[3])
            a = r(rs1)
            b = r(rs2)

            if mnem == 'andn':
                w(rd, a & u64(~b))
            elif mnem == 'orn':
                w(rd, a | u64(~b))
            elif mnem == 'xnor':
                w(rd, u64(~(a ^ b)))

            pc = u64(pc + 4)
            continue
        
        # -----------------------------
        # RV64I R-type word ops (W)
        # -----------------------------
        if mnem in ('addw', 'subw', 'sllw', 'srlw', 'sraw'):
            if len(parts) < 4:
                pc = u64(pc + 4)
                continue

            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            rs2 = reg_id(parts[3])
            a = r(rs1)
            b = r(rs2)
            sh = u64(b) & 0x1F

            if mnem == 'addw':
                res32 = (u32(a) + u32(b)) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'subw':
                res32 = (u32(a) - u32(b)) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'sllw':
                res32 = (u32(a) << sh) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'srlw':
                res32 = (u32(a) >> sh) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'sraw':
                res32 = s32(a) >> sh
                w(rd, sext32_to_64(res32))

            pc = u64(pc + 4)
            continue
        
        # -----------------------------
        # RVC (compressed) aliases / semantics
        # Treat as the corresponding RV64I/RV64M ops on registers.
        # PC increments by 2 for all C.* instructions.
        # -----------------------------
        if mnem == 'c.add':
            # c.add rd, rs2  ==> rd = rd + rs2
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) + r(rs2))
            pc = u64(pc + 2); continue

        if mnem == 'c.addi':
            # c.addi rd, imm6 ==> rd = rd + imm
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])  # you may clamp/sign-extend if you want strictness
            w(rd, r(rd) + imm)
            pc = u64(pc + 2); continue

        if mnem == 'c.addiw':
            # c.addiw rd, imm6 ==> rd = sext32((rd + imm)[31:0])
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            res32 = (u32(r(rd)) + imm) & MASK32
            w(rd, sext32_to_64(res32))
            pc = u64(pc + 2); continue

        if mnem == 'c.addw':
            # c.addw rd, rs2 ==> rd = sext32((rd + rs2)[31:0])
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            res32 = (u32(r(rd)) + u32(r(rs2))) & MASK32
            w(rd, sext32_to_64(res32))
            pc = u64(pc + 2); continue

        if mnem == 'c.sub':
            # c.sub rd, rs2 ==> rd = rd - rs2
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) - r(rs2))
            pc = u64(pc + 2); continue

        if mnem == 'c.subw':
            # c.subw rd, rs2 ==> rd = sext32((rd - rs2)[31:0])
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            res32 = (u32(r(rd)) - u32(r(rs2))) & MASK32
            w(rd, sext32_to_64(res32))
            pc = u64(pc + 2); continue

        if mnem == 'c.and':
            # c.and rd, rs2 ==> rd = rd & rs2
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) & r(rs2))
            pc = u64(pc + 2); continue

        if mnem == 'c.andi':
            # c.andi rd, imm6 ==> rd = rd & imm
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            w(rd, r(rd) & u64(imm))
            pc = u64(pc + 2); continue

        if mnem == 'c.or':
            # c.or rd, rs2 ==> rd = rd | rs2
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) | r(rs2))
            pc = u64(pc + 2); continue

        if mnem == 'c.xor':
            # c.xor rd, rs2 ==> rd = rd ^ rs2
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) ^ r(rs2))
            pc = u64(pc + 2); continue

        if mnem == 'c.slli':
            # c.slli rd, shamt ==> rd = rd << shamt (mask shamt for RV64)
            rd = reg_id(parts[1])
            sh = parse_int_token(parts[2]) & 0x3F
            w(rd, u64(r(rd)) << sh)
            pc = u64(pc + 2); continue

        if mnem == 'c.srli':
            # c.srli rd, shamt ==> rd = rd >>u shamt
            rd = reg_id(parts[1])
            sh = parse_int_token(parts[2]) & 0x3F
            w(rd, u64(r(rd)) >> sh)
            pc = u64(pc + 2); continue

        if mnem == 'c.srai':
            # c.srai rd, shamt ==> rd = rd >>s shamt
            rd = reg_id(parts[1])
            sh = parse_int_token(parts[2]) & 0x3F
            w(rd, u64(s64(r(rd)) >> sh))
            pc = u64(pc + 2); continue

        if mnem == 'c.lui':
            # c.lui rd, nzimm18 ==> rd = sext(nzimm18 << 12) (XLEN)
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            # strict RVC uses 18-bit signed non-zero; for your fixer, just sign-extend 18 then << 12
            w(rd, u64(sext(imm, 18) << 12))
            pc = u64(pc + 2); continue

        if mnem == 'c.not':
            # c.not rd ==> rd = ~rd
            rd = reg_id(parts[1])
            w(rd, ~u64(r(rd)))
            pc = u64(pc + 2); continue

        if mnem == 'c.mul':
            # c.mul rd, rs2 ==> rd = (rd * rs2) low XLEN
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, mul_low_xlen(r(rd), r(rs2)))
            pc = u64(pc + 2); continue
        
        # -----------------------------
        # M-extension (RV64)
        # -----------------------------
        if mnem in ('mul', 'mulh', 'mulhsu', 'mulhu',
                    'div', 'divu', 'rem', 'remu',
                    'mulw', 'divw', 'divuw', 'remw', 'remuw'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            rs2 = reg_id(parts[3])
            a = r(rs1)
            b = r(rs2)

            if mnem == 'mul':
                w(rd, mul_low_xlen(a, b))
            elif mnem == 'mulh':
                w(rd, mulh_signed(a, b))
            elif mnem == 'mulhsu':
                w(rd, mulh_su(a, b))
            elif mnem == 'mulhu':
                w(rd, mulh_unsigned(a, b))
            elif mnem == 'div':
                w(rd, div_signed(a, b))
            elif mnem == 'divu':
                w(rd, div_unsigned(a, b))
            elif mnem == 'rem':
                w(rd, rem_signed(a, b))
            elif mnem == 'remu':
                w(rd, rem_unsigned(a, b))
            elif mnem == 'mulw':
                res32 = (s32(a) * s32(b)) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'divw':
                w(rd, divw_signed(a, b))
            elif mnem == 'divuw':
                w(rd, divuw(a, b))
            elif mnem == 'remw':
                w(rd, remw_signed(a, b))
            elif mnem == 'remuw':
                w(rd, remuw(a, b))

            pc = u64(pc + 4)
            continue

        pc = u64(pc + 4)

    return regs

def is_mcause_token(tok: str) -> bool:
    t = tok.strip().lower()
    if t == 'mcause':
        return True
    # allow numeric csr address (mcause = 0x342)
    if is_numeric_token(t):
        try:
            return parse_int_token(t) == 0x342
        except Exception:
            return False
    return False

def try_rewrite_mcause_readback_block(lines):
    """
    If the last asm instruction is a mcause readback into rd, and the Result
    is '# Result: rd = <val>', rewrite into '# Result: mcause = <val>' and
    delete the csrr line.
    Return (did_rewrite, new_lines).
    """
    # Find last "asm" line index (non-empty, not pure comment)
    last_asm_idx = None
    last_asm_parts = None
    for i in range(len(lines) - 1, -1, -1):
        parts = tokenize_asm(lines[i])
        if parts:
            last_asm_idx = i
            last_asm_parts = parts
            break

    if last_asm_idx is None:
        return False, lines

    mnem = last_asm_parts[0].lower()

    # Accept: csrr rd, mcause  OR csrr rd, 0x342
    # tokenize_asm() makes commas spaces, so format is: [csrr, rd, csr]
    if mnem != 'csrr':
        return False, lines
    if len(last_asm_parts) < 3:
        return False, lines

    rd = last_asm_parts[1].strip()
    csr_tok = last_asm_parts[2].strip()

    if not is_mcause_token(csr_tok):
        return False, lines

    # Find result line: "# Result: <reg> = <val>"
    # Use your existing RESULT_RE2
    res_idx = None
    res_reg = None
    res_val = None
    for i, ln in enumerate(lines):
        if not ln.strip().lower().startswith('# result:'):
            continue
        m = RESULT_RE2.match(ln)
        if not m:
            continue
        res_idx = i
        res_reg = m.group(1).strip()
        res_val = m.group(2).strip()
        break

    if res_idx is None:
        return False, lines

    # Only rewrite if Result is for the same destination register
    if res_reg != rd:
        return False, lines

    new_lines = lines[:]
    # remove the csrr line
    new_lines.pop(last_asm_idx)

    # rewrite result (preserve value token exactly as written)
    new_lines[res_idx if res_idx < last_asm_idx else res_idx - 1] = f'# Result: mcause = {res_val}'
    return True, new_lines

# -----------------------------
# REWRITE ONE BLOCK
# -----------------------------
RESULT_RE = re.compile(
    r'^\s*#\s*Result\s*:\s*([a-zA-Z0-9]+)\s*=\s*([^\s#]+)',
    re.I
)

def block_has_csr(lines):
    for ln in lines:
        parts = tokenize_asm(ln)
        if parts and parts[0].lower() in CSR_MNEMS:
            return True
    return False

def fix_immediates_in_line(line: str) -> str:
    """
    If line is one of MAP_IMMEDIATE_RANGE instructions, clamp/fix its immediate and preserve base.
    Keeps everything else unchanged (including spacing/comments as best-effort).
    """
    # preserve comment
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''

    parts = tokenize_asm(code)
    if not parts:
        return line

    mnem = parts[0].lower()
    if mnem not in MAP_IMMEDIATE_RANGE:
        return line

    # find immediate position by mnemonic
    # - lui/auipc: mnem rd imm
    # - others above: mnem rd rs1 imm  (your current set)
    if mnem in ('lui', 'auipc'):
        if len(parts) < 3:
            return line
        imm_idx = 2
    else:
        if len(parts) < 4:
            return line
        imm_idx = 3

    old_imm_tok = parts[imm_idx]
    if not is_numeric_token(old_imm_tok):
        return line
    imm_sem = normalize_imm_for_insn(mnem, old_imm_tok)
    new_imm_tok = format_imm_preserve_style(mnem, old_imm_tok, imm_sem)

    parts[imm_idx] = new_imm_tok

    # Rebuild with comma style similar to common asm
    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    # keep original trailing spaces before comment minimal
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

RESULT_RE2 = re.compile(r'^\s*#\s*Result\s*:\s*([a-zA-Z0-9]+)\s*=\s*(.+?)\s*$', re.I)

def is_reg_name(name: str) -> bool:
    name = name.strip()
    return name in REG_ALIAS

def rewrite_block(block_text: str, PC_START: int = 0) -> str:
    lines = block_text.splitlines()
    fixed_lines = []
    asm_lines = []

    for ln in lines:
        if ln.strip().lower().startswith('# result:'):
            fixed_lines.append(ln)
        else:
            fixed = fix_immediates_in_line(ln)
            fixed = fix_compressed_immediate_in_line(fixed)  # NEW (RVC imm clamp)
            fixed = fix_vector_immediate_in_line(fixed)      # existing (vector imm clamp)
            fixed_lines.append(fixed)
            if tokenize_asm(fixed):
                asm_lines.append(fixed)

    # optional: special-case mcause readback BEFORE skipping
    did, fixed_lines2 = try_rewrite_mcause_readback_block(fixed_lines)
    if did:
        return '\n'.join(fixed_lines2).rstrip()

    # NEW: skip rewriting Result if block contains CSR or memory ops
    if (
        block_has_csr(asm_lines)
        or block_has_mem(asm_lines)
        or block_has_vector(asm_lines)
        or block_has_compressed(asm_lines)  # NEW
    ):
        return '\n'.join(fixed_lines).rstrip()

    regs = exec_block(asm_lines, PC_START=PC_START)

    out_lines = []
    for ln in fixed_lines:
        if not ln.strip().lower().startswith('# result:'):
            out_lines.append(ln)
            continue

        m = RESULT_RE2.match(ln)
        if not m:
            out_lines.append(ln)
            continue

        reg_name = m.group(1).strip()
        old_val_tok = m.group(2).strip()

        if ('&' in old_val_tok) or ('|' in old_val_tok):
            out_lines.append(ln)
            continue

        if not is_reg_name(reg_name):
            out_lines.append(ln)
            continue

        rid = reg_id(reg_name)
        new_val_tok = format_int_like(old_val_tok, regs[rid])
        out_lines.append(f'# Result: {reg_name} = {new_val_tok}')

    return '\n'.join(out_lines).rstrip()

# -----------------------------
# MAIN FUNCTION
# -----------------------------
def FUNCTION_FIX_TEST_TEXT(TEXT_INPUT: str, PC_START: int = 0) -> str:
    blocks = split_blocks(TEXT_INPUT)
    fixed = [rewrite_block(b, PC_START=PC_START) for b in blocks]
    return '\n\n'.join(fixed).rstrip() + '\n'

EXT_HEADER_RE = re.compile(
    r'^\s*#\s*(?:\d+\s*#\s*)?Extension\s+Must\s+Be\s+Active\s*:\s*([^\s#]+)\s*$',
    re.I
)

def get_block_extension(block_text: str):
    for ln in block_text.splitlines():
        m = EXT_HEADER_RE.match(ln)
        if m:
            return m.group(1).strip()
    return None

def is_non_none_extension(ext: str) -> bool:
    return ext is not None and ext.strip().lower() != 'none'

# -----------------------------
# Example usage:
# -----------------------------
if __name__ == '__main__':
    the_input = '6_TEST_CASE/3_TEST_CASE_REVISED_1.txt'
    the_output = '6_TEST_CASE/3_TEST_CASE_REVISED_2.txt'
    #the_other  = '6_TEST_CASE/3_TEST_CASE_REVISED_2_EXT.txt'

    with open(the_input, 'r', encoding='utf-8') as io:
        TEXT = io.read().strip()

    blocks = split_blocks(TEXT)

    fixed_blocks = []
    fixed_blocks_ext = []

    for b in blocks:
        fixed_b = rewrite_block(b, PC_START=0x800001e6)
        fixed_blocks.append(fixed_b)

        ext = get_block_extension(b)  # cek dari block asli (atau fixed_b juga boleh)
        if is_non_none_extension(ext):
            fixed_blocks_ext.append(fixed_b)

    NEW_ALL = '\n\n'.join(fixed_blocks).rstrip() + '\n'
    NEW_EXT = '\n\n'.join(fixed_blocks_ext).rstrip() + '\n'

    with open(the_output, 'w', encoding='utf-8') as io:
        io.write(NEW_ALL)

    #with open(the_other, 'w', encoding='utf-8') as io:
        #io.write(NEW_EXT)

    print('====\nDONE\n====')

====
DONE
====


In [21]:
# 3 - UPDATE - 3_TEST_CASE_REVISED.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from itertools import count
from pathlib import Path
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_TEST_CASE = Path('6_TEST_CASE/3_TEST_CASE_REVISED_2.txt')
FILE_TEST_CASE_2 = Path('6_TEST_CASE/3_TEST_CASE_REVISED_3.txt')
FOLDER_REPAIR = Path('TEXT')
# PATH - FINISH

# VARIABLE - BEGIN
REGEX_HEADER = re.compile(r'(?m)^(#\s*Extension Must Be Active:.*)$')
STRING_TEXT = FILE_TEST_CASE.read_text(encoding='utf-8').replace('\r\n', '\n')
COUNTER = count(1)
STRING_TEXT = REGEX_HEADER.sub(lambda MATCH_VALUE: f'# {next(COUNTER)} {MATCH_VALUE.group(1)}', STRING_TEXT)
# VARIABLE - FINISH

# CODE - BEGIN
FILE_TEST_CASE_2.write_text(STRING_TEXT, encoding='utf-8', newline='\n')
(FOLDER_REPAIR / '1_REPAIR_TEST_CASE_INPUT.txt').write_text(STRING_TEXT, encoding='utf-8')
(FOLDER_REPAIR / 'REPAIR_TEST_CASE_RAW.txt').write_text(STRING_TEXT, encoding='utf-8')
STRING_REGISTER_TEXT = Path('5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt').read_text(encoding='utf-8').strip()
(FOLDER_REPAIR / 'REPAIR_TEST_CASE_REGISTER.txt').write_text(STRING_REGISTER_TEXT, encoding='utf-8')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# ALARM PLAY - BEGIN
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
import pygame
import time
pygame.mixer.init()
pygame.mixer.music.load('alarm.mp3')
for _ in range(3):
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.01)
pygame.mixer.quit()
# ALARM PLAY - FINISH

# 3 - UPDATE - 3_TEST_CASE_REVISED.txt - END

0.02 second(s)
====
DONE
====


In [ ]:
# 1 - CREATE - 4_REPAIR_TEST_CASE_OUTPUT.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
from pathlib import Path
from pwn import *
context.log_level = 'critical'
from typing import Tuple
import subprocess, sys, operator, re, time
# IMPORT - FINISH

# PATH - BEGIN
FILE_PARTIAL = 'TEXT/REPAIR_TEST_CASE_RAW.txt'
FILE_WORKING_INPUT = 'TEXT/REPAIR_TEST_CASE_RAW_BATCH.txt'
FILE_FINAL_OUTPUT = 'TEXT/4_REPAIR_TEST_CASE_OUTPUT.txt'
FILE_PROMPT = 'TEXT/5_HISTORY_REPAIR_TEST_CASE.txt'
FILE_INPUT_TEST_CASE = 'TEXT/1_REPAIR_TEST_CASE_INPUT.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_PARSE_BLOCK_GLOBAL(STRING_TEXT: str):
    REGEX_PATTERN = re.compile(r'(#\s*(\d+)\s*# Extension Must Be Active:.*?)(?=\n\s*\n#\s*\d+\s*# Extension Must Be Active:|\Z)', re.DOTALL)
    MAP_BLOCK = {}
    for EACH_FULL, EACH_NUM in REGEX_PATTERN.findall(STRING_TEXT):
        MAP_BLOCK[int(EACH_NUM)] = EACH_FULL.strip()
    return MAP_BLOCK

def FUNCTION_WRITE_BLOCK_TO_FILE(MAP_BLOCK, LIST_NUMBER, FILE_OUTPUT_PATH):
    STRING_TEXT = '\n\n'.join(MAP_BLOCK[EACH_NUMBER] for EACH_NUMBER in LIST_NUMBER)
    with open(FILE_OUTPUT_PATH, 'w', encoding='utf-8', newline='\n') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_TEXT.strip() + '\n')

def FUNCTION_READ_BLOCK_FROM_FILE(FILE_PATH):
    with open(FILE_PATH, 'r', encoding='utf-8') as FILE_HANDLE:
        return FUNCTION_PARSE_BLOCK_GLOBAL(FILE_HANDLE.read())

def FUNCTION_WRITE_ALL_BLOCK_SORTED(MAP_BLOCK, FILE_OUTPUT_PATH):
    STRING_TEXT = '\n\n'.join(MAP_BLOCK[EACH_KEY] for EACH_KEY in sorted(MAP_BLOCK))
    with open(FILE_OUTPUT_PATH, 'w', encoding='utf-8', newline='\n') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_TEXT.strip() + '\n')

def FUNCTION_PLAY_QEMU():
    global ERROR_COUNT
    global FIRST_TIME
    def FUNCTION_PARSE_REGISTER(FILE_PATH):
        MAP_REGISTER = {}
        with open(FILE_PATH, 'r', encoding='utf-8') as FILE_HANDLE:
            for EACH_LINE in FILE_HANDLE:
                LIST_PART = EACH_LINE.split()
                if len(LIST_PART) >= SKIPPING_LINE_COUNT:
                    MAP_REGISTER[LIST_PART[0]] = LIST_PART[1]
        return MAP_REGISTER
    def FUNCTION_CLEAN_REGISTER_OUTPUT(STRING_OUTPUT):
        LIST_LINE = STRING_OUTPUT.splitlines()
        STRING_TRIMMED = '\n'.join(LIST_LINE[SKIPPING_LINE_COUNT:-1])
        return REGEX_ANSI_ESCAPE.sub('', STRING_TRIMMED)
    def FUNCTION_HANDLE_QEMU_ERROR(LIST_CMDLINE):
        RESULT_ERROR = subprocess.run(LIST_CMDLINE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print(RESULT_ERROR.stderr.strip())
    def FUNCTION_GET_CHANGED_REGISTER_INFO(IO_HANDLE, LIST_CHANGED_REGISTER):
        STRING_REGISTER = ' '.join(LIST_CHANGED_REGISTER)
        IO_HANDLE.sendline(f'info register {STRING_REGISTER}'.encode())
        STRING_OUTPUT = IO_HANDLE.clean().decode()
        LIST_LINE = STRING_OUTPUT.splitlines()
        if LIST_LINE and '(gdb)' in LIST_LINE[-1]:
            LIST_LINE.pop()
        return '\n'.join(LIST_LINE).strip()
    def FUNCTION_TEST_TEST_CASE(IO_HANDLE):
        MAP_REGISTER_OLD = FUNCTION_PARSE_REGISTER(FILE_REGISTER_DEFAULT)
        MAP_REGISTER_NEW = FUNCTION_PARSE_REGISTER(FILE_REGISTER_CHANGED)
        if ONLY_SHOW_CHANGED_REGISTER:
            LIST_CHANGED_REGISTER = []
            SET_EXCLUDED_REGISTER = {'sp', 'fp', 'pc', 'mcycle', 'minstret', 'cycle', 'time', 'instret'}
            for EACH_REGISTER, EACH_NEW_VALUE in MAP_REGISTER_NEW.items():
                STRING_OLD_VALUE = MAP_REGISTER_OLD.get(EACH_REGISTER)
                if STRING_OLD_VALUE and EACH_REGISTER not in SET_EXCLUDED_REGISTER and EACH_NEW_VALUE != STRING_OLD_VALUE:
                    LIST_CHANGED_REGISTER.append(EACH_REGISTER)
        else:
            LIST_CHANGED_REGISTER = LIST_REGISTER_CHOSEN
        if not LIST_CHANGED_REGISTER:
            STRING_OUTPUT = 'No registers changed!'
        else:
            STRING_OUTPUT = FUNCTION_GET_CHANGED_REGISTER_INFO(IO_HANDLE, LIST_CHANGED_REGISTER)
        print(STRING_OUTPUT)
    def FUNCTION_EDIT_MAIN_S(STRING_CONTENT):
        with open(FILE_CODE_BOOTLOADER, 'w', encoding='utf-8') as FILE_HANDLE:
            FILE_HANDLE.write(STRING_CONTENT)
    def FUNCTION_TIDY_OUTPUT_FILE(FILE_NAME):
        with open(FILE_NAME, 'r', encoding='utf-8') as FILE_HANDLE:
            LIST_LINE = FILE_HANDLE.readlines()
        VALUE_START = 0
        while VALUE_START < len(LIST_LINE) and not (LIST_LINE[VALUE_START].startswith('============') or 'START ITER' in LIST_LINE[VALUE_START]):
            VALUE_START += 1
        LIST_KEPT = LIST_LINE[VALUE_START:]
        LIST_COLLAPSED = []
        VALUE_BLANK_STREAK = 0
        for EACH_LINE in LIST_KEPT:
            if EACH_LINE.strip() == '':
                VALUE_BLANK_STREAK += 1
                if VALUE_BLANK_STREAK <= 1:
                    LIST_COLLAPSED.append(EACH_LINE)
            else:
                VALUE_BLANK_STREAK = 0
                LIST_COLLAPSED.append(EACH_LINE)
        while LIST_COLLAPSED and LIST_COLLAPSED[-1].strip() == '':
            LIST_COLLAPSED.pop()
        if LIST_COLLAPSED and LIST_COLLAPSED[-1].endswith('\n'):
            LIST_COLLAPSED[-1] = LIST_COLLAPSED[-1].rstrip('\n')
        with open(FILE_NAME, 'w', encoding='utf-8') as FILE_HANDLE:
            FILE_HANDLE.writelines(LIST_COLLAPSED)
    def FUNCTION_RUN_QEMU(FUNCTION_HOOK, FILE_REGISTER, MODE_VALUE):
        LIST_CMDLINE = ['./run', str(MODE_VALUE)]
        QEMU_IO = process(LIST_CMDLINE)
        try:
            QEMU_IO.recvuntil(b'*** Now run')
            IO_GDB = process('riscv64-unknown-elf-gdb')
            IO_GDB.sendline(b'b *test_end')
            IO_GDB.sendline(b'b *trap_vector')
            IO_GDB.sendline(b'c')
            STRING_OUTPUT = IO_GDB.recvuntil(b'Breakpoint 1, ', timeout=TIMEOUT_SECOND).decode()
            if not STRING_OUTPUT:
                IO_GDB.recvuntil(b'Breakpoint 2, ', timeout=TIMEOUT_SECOND).decode()
            IO_GDB.sendline(b'info all-registers')
            STRING_OUTPUT = IO_GDB.clean().decode()
            STRING_REGISTER_CLEANED = FUNCTION_CLEAN_REGISTER_OUTPUT(STRING_OUTPUT)
            with open(FILE_REGISTER, 'w', encoding='utf-8') as FILE_HANDLE:
                FILE_HANDLE.write(STRING_REGISTER_CLEANED)
            if FUNCTION_HOOK:
                FUNCTION_HOOK(IO_GDB)
            IO_GDB.kill()
        except EOFError:
            FUNCTION_HANDLE_QEMU_ERROR(LIST_CMDLINE)
        finally:
            QEMU_IO.kill()
    FILE_REGISTER_DEFAULT = 'TEXT/REPAIR_TEST_CASE_REGISTER_DEFAULT.txt'
    FILE_REGISTER_CHANGED = 'TEXT/REPAIR_TEST_CASE_REGISTER_CHANGE.txt'
    FILE_CODE_BOOTLOADER = 'bootloader/code.S'
    FILE_INPUT = 'TEXT/1_REPAIR_TEST_CASE_INPUT.txt'
    FILE_OUTPUT = 'TEXT/2_REPAIR_TEST_CASE_RUN.txt'
    MODE_DEFAULT = 3
    SKIPPING_LINE_COUNT = 3
    TIMEOUT_SECOND = 1
    STRING_DEFAULT = '''li x0, 0
li x1, 0
li x2, 0
li x3, 0
li x4, 0
li x5, 0
li x6, 0
li x7, 0
li x8, 0
li x9, 0
li x10, 0
li x11, 0
li x12, 0
li x13, 0
li x14, 0
li x15, 0
li x16, 0
li x17, 0
li x18, 0
li x19, 0
li x20, 0
li x21, 0
li x22, 0
li x23, 0
li x24, 0
li x25, 0
li x26, 0
li x27, 0
li x28, 0
li x29, 0
li x30, 0
li x31, 0
fmv.d.x f0, x0
fmv.d.x f1, x0
fmv.d.x f2, x0
fmv.d.x f3, x0
fmv.d.x f4, x0
fmv.d.x f5, x0
fmv.d.x f6, x0
fmv.d.x f7, x0
fmv.d.x f8, x0
fmv.d.x f9, x0
fmv.d.x f10, x0
fmv.d.x f11, x0
fmv.d.x f12, x0
fmv.d.x f13, x0
fmv.d.x f14, x0
fmv.d.x f15, x0
fmv.d.x f16, x0
fmv.d.x f17, x0
fmv.d.x f18, x0
fmv.d.x f19, x0
fmv.d.x f20, x0
fmv.d.x f21, x0
fmv.d.x f22, x0
fmv.d.x f23, x0
fmv.d.x f24, x0
fmv.d.x f25, x0
fmv.d.x f26, x0
fmv.d.x f27, x0
fmv.d.x f28, x0
fmv.d.x f29, x0
fmv.d.x f30, x0
fmv.d.x f31, x0
vsetvli x0, x0, e32, m1, ta, ma
vmv.v.i v0, 0
vmv.v.i v1, 0
vmv.v.i v2, 0
vmv.v.i v3, 0
vmv.v.i v4, 0
vmv.v.i v5, 0
vmv.v.i v6, 0
vmv.v.i v7, 0
vmv.v.i v8, 0
vmv.v.i v9, 0
vmv.v.i v10, 0
vmv.v.i v11, 0
vmv.v.i v12, 0
vmv.v.i v13, 0
vmv.v.i v14, 0
vmv.v.i v15, 0
vmv.v.i v16, 0
vmv.v.i v17, 0
vmv.v.i v18, 0
vmv.v.i v19, 0
vmv.v.i v20, 0
vmv.v.i v21, 0
vmv.v.i v22, 0
vmv.v.i v23, 0
vmv.v.i v24, 0
vmv.v.i v25, 0
vmv.v.i v26, 0
vmv.v.i v27, 0
vmv.v.i v28, 0
vmv.v.i v29, 0
vmv.v.i v30, 0
vmv.v.i v31, 0

'''
    REGEX_ANSI_ESCAPE = re.compile(r'\x1B\[[0-?]*[ -/]*[@-~]')
    ONLY_SHOW_CHANGED_REGISTER = False
    
    # IMPORTANT - BEGIN
    LIST_REGISTER_CHOSEN = ['priv', 'pc', 'a0', 'a1', 'a2', 'a3', 't0', 't1', 't2', 't3', 'f0', 'f1', 'f2', 'f3', 'ft0', 'ft1', 'ft2', 'ft3', 'v0', 'v1', 'v2', 'v3', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x19', 'x20', 'x21', 'x22', 'x23', 'x24', 'x25', 'x26', 'x27', 'x28', 'x29', 'x30', 'x31', 'ra', 'sp', 'gp', 'tp', 'mstatus', 'mcause']
    IS_OUTPUT = False
    # IMPORTANT - FINISH
    
    STRING_CODE_TEMPLATE = '''
.globl test_start

test_start:
{test_case}

test_end:
j test_end

.global trap_vector
.align 4

trap_vector:
j trap_vector
    '''
    if IS_OUTPUT:
        class CLASS_STDOUT_TO_FILE:
            def __init__(self, FILE_NAME):
                self.file = open(FILE_NAME, 'w', encoding='utf-8')
                self.stdout = sys.__stdout__
            def write(self, STRING_MESSAGE):
                self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
                self.stdout.write(STRING_MESSAGE)
            def flush(self):
                self.file.flush()
                self.stdout.flush()
            def close(self):
                self.file.close()
    else:
        class CLASS_STDOUT_TO_FILE:
            def __init__(self, FILE_NAME):
                self.file = open(FILE_NAME, 'w', encoding='utf-8')
            def write(self, STRING_MESSAGE):
                self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
            def flush(self):
                self.file.flush()
            def close(self):
                self.file.close()
    OBJECT_STDOUT_FILE = CLASS_STDOUT_TO_FILE(FILE_OUTPUT)
    OBJECT_STDOUT_OLD = sys.stdout
    sys.stdout = OBJECT_STDOUT_FILE
    with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_TEST_CASE = FILE_HANDLE.read()
    LIST_BLOCK = [EACH_BLOCK.strip().split('\n') for EACH_BLOCK in STRING_TEST_CASE.strip().split('\n\n') if EACH_BLOCK.strip()]
    LIST_TEST_CASE = [[EACH_LINE.strip() for EACH_LINE in EACH_BLOCK if EACH_LINE.strip()] for EACH_BLOCK in LIST_BLOCK]
    LIST_DEFAULT_TEST_CASE = [STRING_CODE_TEMPLATE.format(test_case=STRING_DEFAULT)]
    LIST_CHANGED_TEST_CASE = [STRING_CODE_TEMPLATE.format(test_case='\n'.join(STRING_DEFAULT.splitlines() + EACH_TEST_CASE) if EACH_TEST_CASE else STRING_DEFAULT) for EACH_TEST_CASE in LIST_TEST_CASE]
    try:
        FUNCTION_EDIT_MAIN_S(LIST_DEFAULT_TEST_CASE[0])
        FUNCTION_RUN_QEMU(None, FILE_REGISTER_DEFAULT, MODE_DEFAULT)
        for EACH_INDEX, (EACH_NEW_CONTENT, EACH_TEST_CASE) in enumerate(zip(LIST_CHANGED_TEST_CASE, LIST_TEST_CASE)):
            FUNCTION_EDIT_MAIN_S(EACH_NEW_CONTENT.strip())
            print(f'============\nSTART ITER {EACH_INDEX + 1}\n')
            print('\n'.join(EACH_TEST_CASE) + '\n')
            MODE_VALUE = MODE_DEFAULT
            FUNCTION_RUN_QEMU(FUNCTION_TEST_TEST_CASE, FILE_REGISTER_CHANGED, MODE_VALUE)
            if EACH_INDEX < len(LIST_CHANGED_TEST_CASE) - 1:
                print(f'\nEND ITER {EACH_INDEX + 1}\n==========\n')
            else:
                print(f'\nEND ITER {EACH_INDEX + 1}\n==========')
    finally:
        sys.stdout = OBJECT_STDOUT_OLD
        OBJECT_STDOUT_FILE.close()
        FUNCTION_TIDY_OUTPUT_FILE(FILE_OUTPUT)

def FUNCTION_CHECK_QEMU():
    global ERROR_COUNT
    global FIRST_TIME
    FILE_INPUT = 'TEXT/2_REPAIR_TEST_CASE_RUN.txt'
    FILE_OUTPUT = 'TEXT/3_REPAIR_TEST_CASE_CHECK.txt'
    def FUNCTION_CANONICAL_REGISTER(STRING_NAME: str) -> str:
        STRING_KEY = STRING_NAME.strip()
        MATCH_V = re.fullmatch(r'(v\d+)\[(\d+)\]', STRING_KEY)
        if MATCH_V:
            return f'{MATCH_V.group(1)}.w[{MATCH_V.group(2)}]'
        MATCH_V2 = re.fullmatch(r'(v\d+)\.(b|s|w|l|q)\[(\d+)\]', STRING_KEY)
        if MATCH_V2:
            return f'{MATCH_V2.group(1)}.{MATCH_V2.group(2)}[{MATCH_V2.group(3)}]'
        return MAP_REGISTER_ALIAS.get(STRING_KEY, STRING_KEY)
    def FUNCTION_PARSE_VECTOR_LIST(STRING_BODY: str, TOTAL_LEN: int):
        LIST_OUT = []
        TOKENS = [x.strip() for x in STRING_BODY.split(',') if x.strip()]
        for TOK in TOKENS:
            MREP = re.fullmatch(r'(0x[0-9a-fA-F]+)\s*<repeats\s+(\d+)\s+times>', TOK)
            if MREP:
                VAL = int(MREP.group(1), 16)
                CNT = int(MREP.group(2))
                LIST_OUT.extend([VAL] * CNT)
                continue
            MHEX = re.fullmatch(r'0x[0-9a-fA-F]+', TOK)
            if MHEX:
                LIST_OUT.append(int(TOK, 16))
                continue
            continue
        if len(LIST_OUT) < TOTAL_LEN:
            LIST_OUT.extend([0] * (TOTAL_LEN - len(LIST_OUT)))
        return LIST_OUT[:TOTAL_LEN]
    def FUNCTION_PARSE_INT(STRING_VALUE: str) -> int:
        STRING_LOWER = STRING_VALUE.strip().lower()
        if STRING_LOWER.startswith(('+0x', '-0x', '0x')):
            return int(STRING_VALUE, 16)
        return int(STRING_VALUE, 10)
    def FUNCTION_PARSE_ITERATION(STRING_CONTENT):
        REGEX_ITERATION = re.compile(r'START ITER (\d+)(.*?)END ITER \1', re.DOTALL)
        LIST_ITERATION = REGEX_ITERATION.findall(STRING_CONTENT)
        return LIST_ITERATION
    def FUNCTION_HAS_ERROR(STRING_ITERATION: str) -> bool:
        return re.search(r'\bpriv\b', STRING_ITERATION) is None
    def FUNCTION_PARSE_EXPECTED_RESULT(STRING_ITERATION: str):
        STRING_VALUE_PATTERN = r'[+-]?(?:0x[0-9a-fA-F]+|\d+)'
        STRING_REGISTER_PATTERN = r'(?:[A-Za-z_]\w+|v\d+(?:\.(?:b|s|w|l|q))?\[\d+\])'
        STRING_RHS_PATTERN = rf'(?:{STRING_VALUE_PATTERN}|{STRING_REGISTER_PATTERN})'
        REGEX_RESULT_LINE = re.compile(r'#\s*Result:\s*(.*)')
        REGEX_CONDITION = re.compile(
            rf'\s*({STRING_REGISTER_PATTERN})\s*(==|=|!=|<=|>=|<|>)\s*({STRING_RHS_PATTERN})\s*'
        )
        LIST_OR_GROUPS = []
        for EACH_LINE in STRING_ITERATION.splitlines():
            MATCH_RESULT = REGEX_RESULT_LINE.search(EACH_LINE)
            if not MATCH_RESULT:
                continue
            STRING_TAIL = MATCH_RESULT.group(1)
            LIST_OR_PART = [x.strip() for x in STRING_TAIL.split('|') if x.strip()]
            for EACH_OR in LIST_OR_PART:
                LIST_AND = []
                LIST_AND_PART = [x.strip() for x in EACH_OR.split('&') if x.strip()]
                for EACH_PART in LIST_AND_PART:
                    MATCH_COND = REGEX_CONDITION.fullmatch(EACH_PART)
                    if not MATCH_COND:
                        continue
                    STRING_REG, STRING_OP, STRING_RHS_TOKEN = MATCH_COND.groups()
                    STRING_REG = FUNCTION_CANONICAL_REGISTER(STRING_REG)
                    if STRING_OP == '=':
                        STRING_OP = '=='
                    LIST_AND.append((STRING_REG, STRING_OP, STRING_RHS_TOKEN))
                if LIST_AND:
                    LIST_OR_GROUPS.append(LIST_AND)
        return LIST_OR_GROUPS
    def FUNCTION_PARSE_ACTUAL_REGISTER(STRING_ITERATION: str):
        STRING_VALUE_PATTERN = r'[+-]?(?:0x[0-9a-fA-F]+|\d+)'
        REGEX_GPR = re.compile(rf'(\w+)\s+0x[0-9a-fA-F]+\s+({STRING_VALUE_PATTERN})')
        REGEX_FP_RAW = re.compile(
            r'^(f(?:t|s|a)?\d+)\s+[\s\S]*?\(raw\s+0x([0-9a-fA-F]+)\)',
            re.MULTILINE
        )
        REGEX_F_DOUBLE = re.compile(
            r'^(f\d+)\s+\{float\s*=\s*0x[0-9a-fA-F]+,\s*double\s*=\s*0x([0-9a-fA-F]+)\}',
            re.MULTILINE
        )
        MAP_ACTUAL = {}
        for EACH_REG, EACH_VALUE in REGEX_GPR.findall(STRING_ITERATION):
            try:
                VALUE_PARSED = FUNCTION_PARSE_INT(EACH_VALUE)
            except ValueError:
                continue
            STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
            MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
            MAP_ACTUAL[EACH_REG] = VALUE_PARSED
        for EACH_REG, EACH_HEX in REGEX_FP_RAW.findall(STRING_ITERATION):
            VALUE_PARSED = int(EACH_HEX, 16)
            STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
            MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
            MAP_ACTUAL[EACH_REG] = VALUE_PARSED
        for EACH_REG, EACH_HEX_DOUBLE in REGEX_F_DOUBLE.findall(STRING_ITERATION):
            VALUE_PARSED = int(EACH_HEX_DOUBLE, 16)
            STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
            MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
            MAP_ACTUAL[EACH_REG] = VALUE_PARSED
        REGEX_VBLOCK = re.compile(r'^(v\d+)\s+(.*?)(?=^v\d+\s+|\Z)', re.MULTILINE | re.DOTALL)
        REGEX_VIEW = re.compile(r'\b([qlwsb])\s*=\s*\{(.*?)\}', re.DOTALL)
        MAP_LEN = {'b': 16, 's': 8, 'w': 4, 'l': 2, 'q': 1}
        MAP_MASK = {'b': (1 << 8) - 1, 's': (1 << 16) - 1, 'w': (1 << 32) - 1, 'l': (1 << 64) - 1, 'q': MASK64}
        for VREG, VBLOCK in REGEX_VBLOCK.findall(STRING_ITERATION):
            VREG = VREG.strip()
            for VIEW, BODY in REGEX_VIEW.findall(VBLOCK):
                VIEW = VIEW.strip()
                if VIEW not in MAP_LEN:
                    continue
                LIST_VAL = FUNCTION_PARSE_VECTOR_LIST(BODY, MAP_LEN[VIEW])
                for IDX, VAL in enumerate(LIST_VAL):
                    KEY = f'{VREG}.{VIEW}[{IDX}]'
                    MAP_ACTUAL[KEY] = VAL & MAP_MASK[VIEW]
        return MAP_ACTUAL
    def FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY: str):
        M = re.fullmatch(r'v\d+\.(b|s|w|l|q)\[\d+\]', STRING_REG_KEY)
        if not M:
            return None
        return {'b': 8, 's': 16, 'w': 32, 'l': 64, 'q': 64}[M.group(1)]
    def FUNCTION_HAS_RESULT_ONLY_ERROR(STRING_ITERATION: str) -> bool:
        return REGEX_RESULT_ONLY_ERROR.search(STRING_ITERATION) is not None
    def FUNCTION_U64(VALUE_X: int) -> int:
        return VALUE_X & MASK64
    def FUNCTION_CHECK_ITERATION(LIST_ITERATION):
        STRING_OUTPUT = ''
        for EACH_ITER_NUMBER, EACH_ITER_CONTENT in LIST_ITERATION:
            VALUE_ITER_NUMBER = int(EACH_ITER_NUMBER)
            if FUNCTION_HAS_RESULT_ONLY_ERROR(EACH_ITER_CONTENT):
                if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
                    continue
                STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
                continue
            if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
                STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is error\n'
                continue
            LIST_EXPECTED_GROUP = FUNCTION_PARSE_EXPECTED_RESULT(EACH_ITER_CONTENT)
            MAP_ACTUAL = FUNCTION_PARSE_ACTUAL_REGISTER(EACH_ITER_CONTENT)
            FLAG_MATCH = False
            for EACH_GROUP in LIST_EXPECTED_GROUP:
                GROUP_OK = True
                for EACH_REG, EACH_OP, EACH_RHS_TOKEN in EACH_GROUP:
                    STRING_REG_KEY = FUNCTION_CANONICAL_REGISTER(EACH_REG)
                    VALUE_ACTUAL = MAP_ACTUAL.get(STRING_REG_KEY)
                    if VALUE_ACTUAL is None:
                        GROUP_OK = False
                        break
                    try:
                        VALUE_EXPECTED = FUNCTION_PARSE_INT(EACH_RHS_TOKEN)
                    except ValueError:
                        STRING_RHS_KEY = FUNCTION_CANONICAL_REGISTER(EACH_RHS_TOKEN)
                        VALUE_RHS = MAP_ACTUAL.get(STRING_RHS_KEY)
                        if VALUE_RHS is None:
                            GROUP_OK = False
                            break
                        VALUE_EXPECTED = VALUE_RHS
                    if EACH_OP in ('==', '!='):
                        WIDTH_BITS = None
                        if FUNCTION_IS_FLOAT_REG(STRING_REG_KEY):
                            WIDTH_BITS = FUNCTION_HEX_WIDTH_BITS(EACH_RHS_TOKEN)
                        V_BITS = FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY)
                        if V_BITS is not None:
                            WIDTH_BITS = V_BITS
                        if WIDTH_BITS == 8:
                            VALUE_A = VALUE_ACTUAL & ((1 << 8) - 1)
                            VALUE_E = VALUE_EXPECTED & ((1 << 8) - 1)
                        elif WIDTH_BITS == 16:
                            VALUE_A = VALUE_ACTUAL & ((1 << 16) - 1)
                            VALUE_E = VALUE_EXPECTED & ((1 << 16) - 1)
                        elif WIDTH_BITS == 32:
                            VALUE_A = VALUE_ACTUAL & MASK32
                            VALUE_E = VALUE_EXPECTED & MASK32
                        else:
                            VALUE_A = FUNCTION_U64(VALUE_ACTUAL)
                            VALUE_E = FUNCTION_U64(VALUE_EXPECTED)
                        if not MAP_OPERATOR[EACH_OP](VALUE_A, VALUE_E):
                            GROUP_OK = False
                            break
                if GROUP_OK:
                    FLAG_MATCH = True
                    break
            if not FLAG_MATCH:
                STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
        STRING_OUTPUT_FINAL = STRING_OUTPUT.strip()
        with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
            FILE_HANDLE.write(STRING_OUTPUT_FINAL)
    def FUNCTION_IS_FLOAT_REG(STRING_REG_CANON: str) -> bool:
        return re.fullmatch(r'f(?:[0-9]|[12][0-9]|3[01])', STRING_REG_CANON) is not None
    def FUNCTION_HEX_WIDTH_BITS(STRING_TOKEN: str):
        S = STRING_TOKEN.strip().lower()
        if S.startswith(('+0x', '-0x')):
            S = S[1:]
        if not S.startswith('0x'):
            return None
        HEXDIG = S[2:]
        if 1 <= len(HEXDIG) <= 8:
            return 32
        if 9 <= len(HEXDIG) <= 16:
            return 64
        return None
    REGEX_RESULT_ONLY_ERROR = re.compile(r'^\s*#\s*Result\s*:\s*error\s*$', re.IGNORECASE | re.MULTILINE)
    MAP_OPERATOR = {'=': operator.eq, '==': operator.eq, '!=': operator.ne, '<': operator.lt, '<=': operator.le, '>': operator.gt, '>=': operator.ge}
    MAP_REGISTER_ALIAS = {'x0': 'x0', 'zero': 'x0', 'x1': 'x1', 'ra': 'x1', 'x2': 'x2', 'sp': 'x2', 'x3': 'x3', 'gp': 'x3', 'x4': 'x4', 'tp': 'x4', 'x5': 'x5', 't0': 'x5', 'x6': 'x6', 't1': 'x6', 'x7': 'x7', 't2': 'x7', 'x8': 'x8', 's0': 'x8', 'fp': 'x8', 'x9': 'x9', 's1': 'x9', 'x10': 'x10', 'a0': 'x10', 'x11': 'x11', 'a1': 'x11', 'x12': 'x12', 'a2': 'x12', 'x13': 'x13', 'a3': 'x13', 'x14': 'x14', 'a4': 'x14', 'x15': 'x15', 'a5': 'x15', 'x16': 'x16', 'a6': 'x16', 'x17': 'x17', 'a7': 'x17', 'x18': 'x18', 's2': 'x18', 'x19': 'x19', 's3': 'x19', 'x20': 'x20', 's4': 'x20', 'x21': 'x21', 's5': 'x21', 'x22': 'x22', 's6': 'x22', 'x23': 'x23', 's7': 'x23', 'x24': 'x24', 's8': 'x24', 'x25': 'x25', 's9': 'x25', 'x26': 'x26', 's10': 'x26', 'x27': 'x27', 's11': 'x27', 'x28': 'x28', 't3': 'x28', 'x29': 'x29', 't4': 'x29', 'x30': 'x30', 't5': 'x30', 'x31': 'x31', 't6': 'x31'}
    MASK64 = (1 << 64) - 1
    MASK32 = (1 << 32) - 1
    MAP_REGISTER_ALIAS.update({**{f'f{i}': f'f{i}' for i in range(32)}, 'ft0': 'f0', 'ft1': 'f1', 'ft2': 'f2', 'ft3': 'f3', 'ft4': 'f4', 'ft5': 'f5', 'ft6': 'f6', 'ft7': 'f7', 'ft8': 'f28', 'ft9': 'f29', 'ft10': 'f30', 'ft11': 'f31', 'fs0': 'f8', 'fs1': 'f9', 'fs2': 'f18', 'fs3': 'f19', 'fs4': 'f20', 'fs5': 'f21', 'fs6': 'f22', 'fs7': 'f23', 'fs8': 'f24', 'fs9': 'f25', 'fs10': 'f26', 'fs11': 'f27', 'fa0': 'f10', 'fa1': 'f11', 'fa2': 'f12', 'fa3': 'f13', 'fa4': 'f14', 'fa5': 'f15', 'fa6': 'f16', 'fa7': 'f17'})
    with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_CONTENT = FILE_HANDLE.read()
    LIST_ITERATION = FUNCTION_PARSE_ITERATION(STRING_CONTENT)
    FUNCTION_CHECK_ITERATION(LIST_ITERATION)

def FUNCTION_FIX_THEM():
    global ERROR_COUNT
    global FIRST_TIME
    def FUNCTION_SPLIT_HEADER_BODY_FOOTER_TAIL(STRING_TEXT: str) -> Tuple[str, str, str, str]:
        STRING_TEXT = STRING_TEXT.replace('\r\n', '\n').strip('\n')
        LIST_LINE = STRING_TEXT.split('\n')
        STRING_HEADER = ''
        STRING_FOOTER = ''
        LIST_BODY_LINE = []
        LIST_TAIL_LINE = []
        VALUE_EXT_INDEX = None
        for EACH_INDEX, EACH_LINE in enumerate(LIST_LINE):
            if (re.match(r'^\s*#\s*\d+\s*#\s*Extension\b', EACH_LINE, re.IGNORECASE) or re.match(r'^\s*#\s*Extension\b', EACH_LINE, re.IGNORECASE)):
                VALUE_EXT_INDEX = EACH_INDEX
                break
        VALUE_RES_INDEX = None
        VALUE_START_SEARCH = (VALUE_EXT_INDEX + 1) if VALUE_EXT_INDEX is not None else 0
        for EACH_INDEX in range(VALUE_START_SEARCH, len(LIST_LINE)):
            if re.match(r'^\s*#\s*Result\b', LIST_LINE[EACH_INDEX], re.IGNORECASE):
                VALUE_RES_INDEX = EACH_INDEX
                break
        if VALUE_EXT_INDEX is None and VALUE_RES_INDEX is None:
            return '', STRING_TEXT.strip(), '', ''
        if VALUE_EXT_INDEX is not None:
            STRING_HEADER = LIST_LINE[VALUE_EXT_INDEX].strip()
        VALUE_BODY_START = (VALUE_EXT_INDEX + 1) if VALUE_EXT_INDEX is not None else 0
        VALUE_BODY_END = VALUE_RES_INDEX if VALUE_RES_INDEX is not None else len(LIST_LINE)
        LIST_BODY_LINE = LIST_LINE[VALUE_BODY_START:VALUE_BODY_END]
        if VALUE_RES_INDEX is not None:
            STRING_FOOTER = LIST_LINE[VALUE_RES_INDEX].strip()
            LIST_TAIL_LINE = LIST_LINE[VALUE_RES_INDEX + 1:]
        STRING_BODY = '\n'.join(LIST_BODY_LINE).strip('\n')
        LIST_TAIL_LINE = [EACH_LINE for EACH_LINE in LIST_TAIL_LINE if not EACH_LINE.startswith('None')]
        STRING_TAIL = '\n'.join(LIST_TAIL_LINE).strip('\n')
        return STRING_HEADER, STRING_BODY, STRING_FOOTER, STRING_TAIL
    def FUNCTION_CLEAN_ASSEMBLY(STRING_TEXT: str) -> str:
        LIST_LINE = STRING_TEXT.splitlines()
        LIST_CLEANED = [EACH_LINE.lstrip() for EACH_LINE in LIST_LINE]
        return '\n'.join(LIST_CLEANED)
    def FUNCTION_REMOVE_CODE_FENCE(STRING_TEXT: str) -> str:
        return re.sub(r'^\s*```.*\n?', '', STRING_TEXT, flags=re.MULTILINE)
    def FUNCTION_PARSE_BLOCK(STRING_TEXT: str):
        REGEX_PATTERN = re.compile(r'(#\s*(\d+)\s*# Extension Must Be Active:.*?)(?=\n\s*\n#\s*\d+\s*# Extension Must Be Active:|\Z)', re.DOTALL)
        MAP_BLOCK = {}
        for EACH_FULL, EACH_NUM in REGEX_PATTERN.findall(STRING_TEXT):
            MAP_BLOCK[int(EACH_NUM)] = EACH_FULL.strip()
        return MAP_BLOCK
    def FUNCTION_EXTRACT_NUMBER(STRING_BLOCK: str) -> int:
        MATCH_VALUE = re.search(r'#\s*(\d+)\s*# Extension Must Be Active:', STRING_BLOCK)
        if not MATCH_VALUE:
            raise ValueError('Cannot find block number')
        return int(MATCH_VALUE.group(1))
    def FUNCTION_NORMALIZE_NEW_BLOCK(STRING_BLOCK: str) -> str:
        STRING_BLOCK = re.sub(r'```[a-zA-Z]*\n?', '', STRING_BLOCK)
        STRING_BLOCK = STRING_BLOCK.replace('```', '')
        return STRING_BLOCK.strip()
    FILE_CHECK = 'TEXT/3_REPAIR_TEST_CASE_CHECK.txt'
    FILE_RUN = 'TEXT/2_REPAIR_TEST_CASE_RUN.txt'
    FILE_ORIGINAL = 'TEXT/REPAIR_TEST_CASE_RAW.txt'
    FILE_OUTPUT = 'TEXT/4_REPAIR_TEST_CASE_OUTPUT.txt'
    FILE_FIXING = 'TEXT/REPAIR_TEST_CASE_ATTEMPT.txt'
    FILE_BASE = FILE_OUTPUT if FIRST_TIME > 0 else FILE_ORIGINAL
    with open(FILE_BASE, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_OLD_TEXT = FILE_HANDLE.read()
    MAP_OLD_BLOCK = FUNCTION_PARSE_BLOCK(STRING_OLD_TEXT)
    SET_ERROR_ITER = set()
    VALUE_TEMP_ERROR = 0
    with open(FILE_CHECK, 'r', encoding='utf-8') as FILE_HANDLE:
        for EACH_LINE in FILE_HANDLE:
            EACH_LINE = EACH_LINE.strip()
            MATCH_VALUE = re.match(r'START ITER (\d+) is error', EACH_LINE)
            if MATCH_VALUE:
                SET_ERROR_ITER.add(int(MATCH_VALUE.group(1)))
                VALUE_TEMP_ERROR += 1
    if VALUE_TEMP_ERROR == 0:
        ERROR_COUNT = 0
        if FIRST_TIME == 0:
            with open(FILE_ORIGINAL, 'r', encoding='utf-8') as FILE_HANDLE:
                STRING_CONTENT = FILE_HANDLE.read().strip()
            with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
                FILE_HANDLE.write(STRING_CONTENT)
        return
    with open(FILE_RUN, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_TEXT = FILE_HANDLE.read()
    REGEX_ITER_PATTERN = re.compile(r'START ITER (\d+)\n(.*?)\nEND ITER \1', re.DOTALL)
    LIST_ERROR = []
    for MATCH_VALUE in REGEX_ITER_PATTERN.finditer(STRING_TEXT):
        VALUE_ITER = int(MATCH_VALUE.group(1))
        STRING_BODY = MATCH_VALUE.group(2).strip()
        if VALUE_ITER in SET_ERROR_ITER:
            STRING_BODY = re.sub(r'^=+\n?', '', STRING_BODY)
            STRING_BODY = re.sub(r'\n?=+$', '', STRING_BODY)
            LIST_ERROR.append(STRING_BODY.strip())
    MODEL = 'gpt-oss:120b'
    TEMPERATURE = 1
    CONTEXT = 16384
    API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
    CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
    LIST_TOTAL_OUTPUT = []
    global GLOBAL_COUNTER
    global STRING_PROMPT_HISTORY
    for EACH_ERROR in LIST_ERROR:
        STRING_HEADER, STRING_BODY, STRING_FOOTER, STRING_TAIL = FUNCTION_SPLIT_HEADER_BODY_FOOTER_TAIL(EACH_ERROR)
        MATCH_NUMBER = re.search(r'#\s*(\d+)', STRING_BODY)
        MATCH_EXTENSION = re.search(r'#\s*Extension Must Be Active:.*', STRING_BODY)
        MATCH_RESULT = re.search(r'#\s*Result:.*', STRING_BODY)
        STRING_NUMBER = f'# {MATCH_NUMBER.group(1)}' if MATCH_NUMBER else None
        STRING_EXTENSION = MATCH_EXTENSION.group(0) if MATCH_EXTENSION else None
        STRING_RESULT = MATCH_RESULT.group(0) if MATCH_RESULT else None
        LIST_LINE = STRING_BODY.strip().splitlines()
        if LIST_LINE and LIST_LINE[0].startswith('#') and 'Extension Must Be Active:' in LIST_LINE[0]:
            LIST_LINE.pop(0)
        if LIST_LINE and LIST_LINE[-1].startswith('# Result:'):
            LIST_LINE.pop()
        STRING_BODY_CLEAN = '\n'.join(LIST_LINE)
        STRING_ERROR_MODIFIED = STRING_BODY_CLEAN + '\n\n' + 'THE ERROR:\n##########\n\n' + STRING_TAIL + '\n\n##########'
        STRING_PROMPT = f'''For RISC-V, minimally fix the assembly test case below while PRESERVING the ORIGINAL INSTRUCTION and OPERANDS whenever possible:

{STRING_ERROR_MODIFIED}

Do not output any introduction. Follow the format above.

Notes:
+ Remember, output must be ASCII-only text.
+ DO NOT USE .globl, _start, labels, directives, or PC-relative expressions; assume all boilerplate is already set up!
+ Do not use ```assembly or ```asm just write the assembly code
+ If the error message is empty, it usually means the program is hanging (e.g., infinite loop), so modify the test case to ensure it terminates.
+ Write the full assembly code, DO NOT write something like this:
/* ... other code ... */
# ... other existing code ...'''
        if IS_LOCAL:
            STRING_RESPONSE = chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
        else:
            STRING_RESPONSE = CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
        for EACH_CHARACTER in ['\u200b', '\u200c', '\u200d', '\ufeff']:
            STRING_RESPONSE = STRING_RESPONSE.replace(EACH_CHARACTER, '')
        for EACH_CHARACTER in ['\u00a0', '\u202f']:
            STRING_RESPONSE = STRING_RESPONSE.replace(EACH_CHARACTER, ' ')
        for EACH_RANGE in range(0xe000, 0xf900):
            STRING_RESPONSE = STRING_RESPONSE.replace(chr(EACH_RANGE), '')
        STRING_RESPONSE = STRING_RESPONSE.replace('“', '\'')
        STRING_RESPONSE = STRING_RESPONSE.replace('”', '\'')
        STRING_RESPONSE = STRING_RESPONSE.replace('←', '<-')
        STRING_RESPONSE = STRING_RESPONSE.replace('→', '->')
        STRING_RESPONSE = STRING_RESPONSE.replace('■', '+')
        PROMPT_INTRO_FIRST = f' START - {GLOBAL_COUNTER} '.center(WIDTH_PROMPT, '=')
        PROMPT_INTRO_END = f' END - {GLOBAL_COUNTER} '.center(WIDTH_PROMPT, '=')
        PROMPT_INTRO = f'''=======
PROMPT:
=======
{'=' * int(WIDTH_PROMPT / 2)}
'''
        PROMPT_ENDING = f'''
{'=' * int(WIDTH_PROMPT / 2)}
=========
RESPONSE:
=========
{'=' * int(WIDTH_PROMPT / 2)}
{STRING_RESPONSE}
'''
        STRING_PROMPT_HISTORY += (PROMPT_INTRO_FIRST + '\n' + PROMPT_INTRO + STRING_PROMPT + PROMPT_ENDING + PROMPT_INTRO_END + SUFFIX_FOR_PROMPT)
        GLOBAL_COUNTER += 1
        if not STRING_HEADER:
            MATCH_HEADER = re.search(r'^\s*#\s*\d+\s*#\s*Extension Must Be Active:.*$', EACH_ERROR, re.MULTILINE)
            if MATCH_HEADER:
                STRING_HEADER = MATCH_HEADER.group(0).strip()
        if not STRING_FOOTER:
            MATCH_FOOTER = re.search(r'^\s*#\s*Result\s*:.*$', EACH_ERROR, re.MULTILINE)
            if MATCH_FOOTER:
                STRING_FOOTER = MATCH_FOOTER.group(0).strip()
        STRING_NEW_RESULT = STRING_HEADER + '\n' + STRING_RESPONSE + '\n' + STRING_FOOTER
        STRING_FINAL = FUNCTION_REMOVE_CODE_FENCE(FUNCTION_CLEAN_ASSEMBLY(STRING_NEW_RESULT)).strip()
        STRING_FINAL = '\n'.join(EACH_LINE for EACH_LINE in STRING_FINAL.splitlines() if EACH_LINE.strip())
        LIST_TOTAL_OUTPUT.append(STRING_FINAL)
    STRING_FIXING_TEXT = '\n\n'.join(LIST_TOTAL_OUTPUT)
    with open(FILE_FIXING, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_FIXING_TEXT)
    for EACH_NEW_BLOCK in LIST_TOTAL_OUTPUT:
        VALUE_NUMBER = FUNCTION_EXTRACT_NUMBER(EACH_NEW_BLOCK)
        MAP_OLD_BLOCK[VALUE_NUMBER] = FUNCTION_NORMALIZE_NEW_BLOCK(EACH_NEW_BLOCK)
    STRING_FINAL_TEXT = '\n\n'.join(MAP_OLD_BLOCK[EACH_KEY] for EACH_KEY in sorted(MAP_OLD_BLOCK))
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_FINAL_TEXT)
    with open('TEXT/1_REPAIR_TEST_CASE_INPUT.txt', 'w', encoding='utf-8', newline='\n') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_FINAL_TEXT.strip() + '\n')
# FUNCTION - FINISH

# VARIABLE - BEGIN
GLOBAL_COUNTER = 1
IS_LOCAL = False
STRING_PROMPT_HISTORY = ''
WIDTH_PROMPT = 120
STRING_DIVIDE_FOR_PROMPT = ('#' * WIDTH_PROMPT + '\n') * 3
STRING_DIVIDE_FOR_PROMPT = STRING_DIVIDE_FOR_PROMPT.rstrip('\n')
SUFFIX_FOR_PROMPT = '\n' + STRING_DIVIDE_FOR_PROMPT + '\n'
MAP_ALL_BLOCK = FUNCTION_READ_BLOCK_FROM_FILE(FILE_PARTIAL)
MAP_FINAL_BLOCK = dict(MAP_ALL_BLOCK)
LIST_NUMBER = sorted(MAP_ALL_BLOCK.keys())
BATCH_SIZE = len(LIST_NUMBER)
FILE_A = FILE_INPUT_TEST_CASE
FILE_B = FILE_FINAL_OUTPUT
FILE_C = FILE_PROMPT
FILE_D = FILE_WORKING_INPUT
PATH_A = Path(FILE_A)
PATH_B = Path(FILE_B)
PATH_C = Path(FILE_C)
PATH_D = Path(FILE_D)
MAXIMUM_LIMIT = 10
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_PROMPT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('')

for EACH_BATCH_START in range(0, len(LIST_NUMBER), BATCH_SIZE):
    LIST_BATCH_NUMBER = LIST_NUMBER[EACH_BATCH_START:EACH_BATCH_START + BATCH_SIZE]
    FUNCTION_WRITE_BLOCK_TO_FILE(MAP_ALL_BLOCK, LIST_BATCH_NUMBER, FILE_WORKING_INPUT)
    FILE_INPUT = FILE_WORKING_INPUT
    FILE_OUTPUT = 'TEXT/1_REPAIR_TEST_CASE_INPUT.txt'
    with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
        STRING_CONTENT = FILE_HANDLE.read()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_CONTENT)
    ERROR_COUNT = -1
    FIRST_TIME = 0
    while ERROR_COUNT != 0 and FIRST_TIME < MAXIMUM_LIMIT:
        FUNCTION_PLAY_QEMU()
        FUNCTION_CHECK_QEMU()
        FUNCTION_FIX_THEM()
        FIRST_TIME += 1
    STRING_PROMPT_HISTORY = STRING_PROMPT_HISTORY.removesuffix(SUFFIX_FOR_PROMPT)
    with open(FILE_PROMPT, 'a', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_PROMPT_HISTORY + '\n\n')
    MAP_FIXED_BATCH_BLOCK = FUNCTION_READ_BLOCK_FROM_FILE('TEXT/4_REPAIR_TEST_CASE_OUTPUT.txt')
    for EACH_NUMBER in LIST_BATCH_NUMBER:
        if EACH_NUMBER in MAP_FIXED_BATCH_BLOCK:
            MAP_FINAL_BLOCK[EACH_NUMBER] = MAP_FIXED_BATCH_BLOCK[EACH_NUMBER]
        else:
            pass
    FUNCTION_WRITE_ALL_BLOCK_SORTED(MAP_FINAL_BLOCK, FILE_FINAL_OUTPUT)

LIST_LINE = PATH_A.read_text(encoding='utf-8').splitlines()

while LIST_LINE and not LIST_LINE[-1].strip():
    LIST_LINE.pop()

PATH_A.write_text('\n'.join(LIST_LINE), encoding='utf-8')
LIST_LINE = PATH_B.read_text(encoding='utf-8').splitlines()

while LIST_LINE and not LIST_LINE[-1].strip():
    LIST_LINE.pop()

PATH_B.write_text('\n'.join(LIST_LINE), encoding='utf-8')
LIST_LINE = PATH_C.read_text(encoding='utf-8').splitlines()

while LIST_LINE and not LIST_LINE[-1].strip():
    LIST_LINE.pop()

PATH_C.write_text('\n'.join(LIST_LINE), encoding='utf-8')
LIST_LINE = PATH_D.read_text(encoding='utf-8').splitlines()

while LIST_LINE and not LIST_LINE[-1].strip():
    LIST_LINE.pop()

PATH_D.write_text('\n'.join(LIST_LINE), encoding='utf-8')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 4_REPAIR_TEST_CASE_OUTPUT.txt - END

In [ ]:
# 2 - UPDATE - 4_REPAIR_TEST_CASE_OUTPUT.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import re
# IMPORT - FINISH

import shutil
file_a = 'TEXT/4_REPAIR_TEST_CASE_OUTPUT.txt'
file_b = 'TEXT/4_REPAIR_TEST_CASE_OUTPUT_1.txt'
shutil.copyfile(file_a, file_b)

# PATH - BEGIN
FILE_INPUT = Path('TEXT/4_REPAIR_TEST_CASE_OUTPUT_1.txt')
FILE_OUTPUT = Path('TEXT/4_REPAIR_TEST_CASE_OUTPUT_2.txt')
FILE_REGISTER = Path('TEXT/REPAIR_TEST_CASE_REGISTER.txt')
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_TWOS_SIGNED(VALUE_INT: int, BIT_COUNT: int) -> int:
    VALUE_MASK = (1 << BIT_COUNT) - 1
    VALUE_INT &= VALUE_MASK
    VALUE_SIGN = 1 << (BIT_COUNT - 1)
    return VALUE_INT - (1 << BIT_COUNT) if (VALUE_INT & VALUE_SIGN) else VALUE_INT

def FUNCTION_CONVERT_HEX_IN_LINE(STRING_LINE: str) -> str:
    def FUNCTION_REPLACE(MATCH_VALUE: re.Match) -> str:
        STRING_HEX = MATCH_VALUE.group(1)
        IS_NEG = STRING_HEX.startswith('-')
        STRING_HEX_BODY = STRING_HEX[1:] if IS_NEG else STRING_HEX
        VALUE_INT = int(STRING_HEX_BODY, 16)
        if IS_NEG:
            VALUE_INT = -VALUE_INT
        if REGEX_RESULT_HEADER.match(STRING_LINE):
            return str(FUNCTION_TWOS_SIGNED(VALUE_INT, RESULT_BIT))
        return str(VALUE_INT)
    return REGEX_HEX.sub(FUNCTION_REPLACE, STRING_LINE)

def FUNCTION_REPLACE_REGISTER(STRING_LINE: str) -> str:
    STRING_LINE = REGEX_X.sub(lambda MATCH_VALUE: MAP_X_ABI.get(MATCH_VALUE.group(0), MATCH_VALUE.group(0)), STRING_LINE)
    STRING_LINE = REGEX_F.sub(lambda MATCH_VALUE: MAP_F_ABI.get(MATCH_VALUE.group(0), MATCH_VALUE.group(0)), STRING_LINE)
    return STRING_LINE

def FUNCTION_REPLACE_CSR_NUMBER(STRING_LINE: str) -> str:
    STRING_STRIPPED = STRING_LINE.lstrip()
    if not STRING_STRIPPED or STRING_STRIPPED.startswith('#'):
        return STRING_LINE
    MATCH_OP = re.match(r'^(\s*)([A-Za-z.][A-Za-z0-9_.]*)\s+(.*)$', STRING_LINE)
    if not MATCH_OP:
        return STRING_LINE
    STRING_INDENT = MATCH_OP.group(1)
    STRING_OP = MATCH_OP.group(2).lower()
    STRING_REST = MATCH_OP.group(3)
    if STRING_OP not in SET_CSR_OP:
        return STRING_LINE
    NL = '\n' if STRING_LINE.endswith('\n') else ''
    STRING_BEFORE_HASH, STRING_COMMENT_TEXT = (STRING_REST.split('#', 1) + [''])[:2]
    STRING_BEFORE_HASH = STRING_BEFORE_HASH.strip()
    STRING_COMMENT_TEXT = STRING_COMMENT_TEXT.strip()
    LIST_OPERAND = [x.strip() for x in STRING_BEFORE_HASH.split(',') if x.strip()]
    INDEX_CSR = None
    if STRING_OP in SET_CSR_PRR or STRING_OP in SET_CSR_PI:
        INDEX_CSR = 0 if len(LIST_OPERAND) >= 1 else None
    elif STRING_OP in SET_CSR_RRR or STRING_OP in SET_CSR_RRI:
        INDEX_CSR = 1 if len(LIST_OPERAND) >= 2 else None
    elif STRING_OP == 'csrr':
        INDEX_CSR = 1 if len(LIST_OPERAND) >= 2 else None
    if INDEX_CSR is not None:
        STRING_TOKEN = LIST_OPERAND[INDEX_CSR]
        if STRING_TOKEN.lower().startswith('0x') or re.fullmatch(r'\d+', STRING_TOKEN):
            VALUE_NUM = int(STRING_TOKEN, 16) if STRING_TOKEN.lower().startswith('0x') else int(STRING_TOKEN, 10)
            STRING_NAME = MAP_CSR_NUM_TO_NAME.get(VALUE_NUM)
            if STRING_NAME:
                LIST_OPERAND[INDEX_CSR] = STRING_NAME
    STRING_NEW = STRING_INDENT + STRING_OP + ' ' + ', '.join(LIST_OPERAND)
    if '#' in STRING_REST:
        if STRING_COMMENT_TEXT:
            return STRING_NEW + ' # ' + STRING_COMMENT_TEXT + NL
        return STRING_NEW + ' #' + NL
    return STRING_NEW + NL

def FUNCTION_COLLAPSE_MCAUSE_BLOCK(STRING_TEXT: str) -> str:
    LIST_PART = re.split(r'(?m)(?=^#\s*\d+\s*#\s*Extension Must Be Active:)', STRING_TEXT)
    LIST_OUTPUT = []
    REGEX_CSRR_MCAUSE = re.compile(r'^\s*csrr\s+([A-Za-z0-9_]+)\s*,\s*mcause\s*(?:#.*)?\s*$', re.I)
    REGEX_CSRRS_MCAUSE = re.compile(r'^\s*csrrs\s+([A-Za-z0-9_]+)\s*,\s*mcause\s*,\s*(x0|zero)\s*(?:#.*)?\s*$', re.I)
    REGEX_RESULT_REG = re.compile(r'^\s*#\s*Result:\s*([A-Za-z0-9_]+)\s*=\s*(.+?)\s*$', re.I)
    for EACH_PART in LIST_PART:
        if not EACH_PART.strip():
            LIST_OUTPUT.append(EACH_PART)
            continue
        LIST_LINE = EACH_PART.splitlines(True)
        REGISTER_DEST = None
        for EACH_INDEX in range(len(LIST_LINE) - 1, -1, -1):
            MATCH_MCAUSE = REGEX_CSRR_MCAUSE.match(LIST_LINE[EACH_INDEX]) or REGEX_CSRRS_MCAUSE.match(LIST_LINE[EACH_INDEX])
            if MATCH_MCAUSE:
                REGISTER_DEST = MATCH_MCAUSE.group(1)
                break
        if REGISTER_DEST is None:
            LIST_OUTPUT.append(EACH_PART)
            continue
        VALUE_REMOVED = None
        LIST_NEW_LINE = []
        for EACH_LINE in LIST_LINE:
            MATCH_RESULT = REGEX_RESULT_REG.match(EACH_LINE)
            if MATCH_RESULT and MATCH_RESULT.group(1) == REGISTER_DEST:
                if VALUE_REMOVED is None:
                    VALUE_REMOVED = MATCH_RESULT.group(2).strip()
                continue
            LIST_NEW_LINE.append(EACH_LINE)
        if VALUE_REMOVED is None:
            LIST_OUTPUT.append(EACH_PART)
            continue
        LIST_FILTERED = []
        REMOVED_CSRR = False
        for EACH_LINE in LIST_NEW_LINE:
            if (not REMOVED_CSRR) and (REGEX_CSRR_MCAUSE.match(EACH_LINE) or REGEX_CSRRS_MCAUSE.match(EACH_LINE)):
                REMOVED_CSRR = True
                continue
            LIST_FILTERED.append(EACH_LINE)
        STRING_INSERT = f'# Result: mcause = {VALUE_REMOVED}\n'
        VALUE_J = len(LIST_FILTERED)
        while VALUE_J > 0 and LIST_FILTERED[VALUE_J - 1].strip() == '':
            VALUE_J -= 1
        LIST_FILTERED.insert(VALUE_J, STRING_INSERT)
        LIST_OUTPUT.append(''.join(LIST_FILTERED))
    return ''.join(LIST_OUTPUT)
# FUNCTION - FINISH

# VARIABLE - BEGIN
RESULT_BIT = 32
MAP_X_ABI = {'x0': 'zero','x1': 'ra','x2': 'sp','x3': 'gp','x4': 'tp', 'x5': 't0','x6': 't1','x7': 't2','x8': 's0','x9': 's1', 'x10': 'a0','x11': 'a1','x12': 'a2','x13': 'a3','x14': 'a4','x15': 'a5','x16': 'a6','x17': 'a7', 'x18': 's2','x19': 's3','x20': 's4','x21': 's5','x22': 's6','x23': 's7','x24': 's8','x25': 's9','x26': 's10','x27': 's11', 'x28': 't3','x29': 't4','x30': 't5','x31': 't6'}
MAP_F_ABI = {'f0':'ft0','f1':'ft1','f2':'ft2','f3':'ft3','f4':'ft4','f5':'ft5','f6':'ft6','f7':'ft7', 'f8':'fs0','f9':'fs1', 'f10':'fa0','f11':'fa1','f12':'fa2','f13':'fa3','f14':'fa4','f15':'fa5','f16':'fa6','f17':'fa7', 'f18':'fs2','f19':'fs3','f20':'fs4','f21':'fs5','f22':'fs6','f23':'fs7','f24':'fs8','f25':'fs9','f26':'fs10','f27':'fs11', 'f28':'ft8','f29':'ft9','f30':'ft10','f31':'ft11'}
REGEX_X = re.compile(r'\bx(3[01]|[12]\d|[0-9])\b')
REGEX_F = re.compile(r'\bf(3[01]|[12]\d|[0-9])\b')
REGEX_HEX = re.compile(r'(?<![A-Za-z0-9_])(-?0x[0-9a-fA-F]+)(?![A-Za-z0-9_])')
REGEX_RESULT_HEADER = re.compile(r'^\s*#\s*Result\s*:', re.IGNORECASE)
SET_CSR_RRR = {'csrrw', 'csrrs', 'csrrc'}
SET_CSR_RRI = {'csrrwi', 'csrrsi', 'csrrci'}
SET_CSR_PRR = {'csrw', 'csrs', 'csrc'}
SET_CSR_PI = {'csrwi', 'csrsi', 'csrci'}
SET_CSR_PSEUDO = {'csrr'}
SET_CSR_OP = SET_CSR_RRR | SET_CSR_RRI | SET_CSR_PRR | SET_CSR_PI | SET_CSR_PSEUDO
MAP_CSR_NUM_TO_NAME = {}
REGISTER_CURRENT = None
STRING_TEXT = FILE_INPUT.read_text(encoding='utf-8', errors='replace').replace('\r\n', '\n')
LIST_LINE = STRING_TEXT.splitlines(True)
LIST_OUTPUT_LINE = []
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_LINE in FILE_REGISTER.read_text(encoding='utf-8', errors='replace').splitlines():
    MATCH_REGISTER = re.match(r'^\s*REGISTER:\s*([A-Za-z0-9_]+)\s*$', EACH_LINE)
    if MATCH_REGISTER:
        REGISTER_CURRENT = MATCH_REGISTER.group(1)
        continue
    MATCH_NUMBER = re.match(r'^\s*NUMBER:\s*(0x[0-9a-fA-F]+|\d+)\s*$', EACH_LINE)
    if MATCH_NUMBER and REGISTER_CURRENT:
        STRING_NUMBER = MATCH_NUMBER.group(1)
        VALUE_NUMBER = int(STRING_NUMBER, 16) if STRING_NUMBER.lower().startswith('0x') else int(STRING_NUMBER, 10)
        MAP_CSR_NUM_TO_NAME.setdefault(VALUE_NUMBER, REGISTER_CURRENT)
        REGISTER_CURRENT = None

for EACH_LINE in LIST_LINE:
    EACH_LINE = FUNCTION_REPLACE_CSR_NUMBER(EACH_LINE)
    EACH_LINE = FUNCTION_REPLACE_REGISTER(EACH_LINE)
    LIST_OUTPUT_LINE.append(EACH_LINE)

STRING_FINAL = ''.join(LIST_OUTPUT_LINE)
STRING_FINAL = FUNCTION_COLLAPSE_MCAUSE_BLOCK(STRING_FINAL)
FILE_OUTPUT.write_text(STRING_FINAL, encoding='utf-8', newline='\n')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 2 - UPDATE - 4_REPAIR_TEST_CASE_OUTPUT.txt - END

In [ ]:
# 3 - CREATE - REPAIR_TEST_CASE_OUTPUT_EXTENSION.txt - START

import shutil
file_a = 'TEXT/4_REPAIR_TEST_CASE_OUTPUT_2.txt'
file_b = 'TEXT/1_QEMU_XV6_INPUT.txt'
shutil.copyfile(file_a, file_b)

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = Path('TEXT/4_REPAIR_TEST_CASE_OUTPUT_2.txt')
FILE_OUTPUT = Path('TEXT/REPAIR_TEST_CASE_OUTPUT_EXTENSION.txt')
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_FILTER_EXTENSION_BLOCK(LIST_LINE):
    LIST_OUTPUT = []
    LIST_CURRENT = []
    CURRENT_EXTENSION = None
    for EACH_LINE in LIST_LINE:
        MATCH = REGEX_EXTENSION_HEADER.match(EACH_LINE)
        if MATCH:
            if (
                LIST_CURRENT
                and CURRENT_EXTENSION
                and CURRENT_EXTENSION.lower() != 'none'
            ):
                while LIST_CURRENT and LIST_CURRENT[-1].strip() == '':
                    LIST_CURRENT.pop()
                if LIST_OUTPUT:
                    LIST_OUTPUT.append('\n')
                LIST_OUTPUT.extend(LIST_CURRENT)
            LIST_CURRENT = [EACH_LINE]
            CURRENT_EXTENSION = MATCH.group('ext')
        else:
            LIST_CURRENT.append(EACH_LINE)
    if (
        LIST_CURRENT
        and CURRENT_EXTENSION
        and CURRENT_EXTENSION.lower() != 'none'
    ):
        while LIST_CURRENT and LIST_CURRENT[-1].strip() == '':
            LIST_CURRENT.pop()
        if LIST_OUTPUT:
            LIST_OUTPUT.append('\n')
        LIST_OUTPUT.extend(LIST_CURRENT)
    return LIST_OUTPUT
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_EXTENSION_HEADER = re.compile(
    r'^\s*#\s*\d+\s*#\s*Extension Must Be Active:\s*(?P<ext>\S+)',
    re.IGNORECASE
)
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, 'r', encoding='utf-8') as f:
    LIST_LINE = f.readlines()

LIST_OUTPUT = FUNCTION_FILTER_EXTENSION_BLOCK(LIST_LINE)
TEXT_OUTPUT = ''.join(LIST_OUTPUT).rstrip('\n').rstrip()

with open(FILE_OUTPUT, 'w', encoding='utf-8', newline='\n') as f:
    f.writelines(TEXT_OUTPUT)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# ALARM PLAY - BEGIN
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
import pygame
import time
pygame.mixer.init()
pygame.mixer.music.load('alarm.mp3')
for _ in range(3):
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.01)
pygame.mixer.quit()
# ALARM PLAY - FINISH

# 3 - CREATE - REPAIR_TEST_CASE_OUTPUT_EXTENSION.txt - END

In [ ]:
# 1 - CREATE - 2_QEMU_XV6_RUN.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pwn import *
context.log_level = 'critical'
import re, subprocess, sys
# IMPORT - FINISH

# PATH - BEGIN
FILE_REGISTER_DEFAULT = 'TEXT/QEMU_XV6_REGISTER_DEFAULT.txt'
FILE_REGISTER_CHANGE = 'TEXT/QEMU_XV6_REGISTER_CHANGE.txt'
FILE_BOOTLOADER_CODE = 'bootloader/code.S'
FILE_INPUT = 'TEXT/1_QEMU_XV6_INPUT.txt'
FILE_OUTPUT = 'TEXT/2_QEMU_XV6_RUN.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_PARSE_REGISTER(FILE_PATH):
    MAP_REGISTER = {}
    with open(FILE_PATH, 'r', encoding='utf-8') as FILE_HANDLE:
        for EACH_LINE in FILE_HANDLE:
            LIST_PART = EACH_LINE.split()
            if len(LIST_PART) >= SKIPPING_LINE_COUNT:
                MAP_REGISTER[LIST_PART[0]] = LIST_PART[1]
    return MAP_REGISTER

def FUNCTION_CLEAN_REGISTER_OUTPUT(STRING_OUTPUT):
    LIST_LINE = STRING_OUTPUT.splitlines()
    STRING_TRIMMED = '\n'.join(LIST_LINE[SKIPPING_LINE_COUNT:-1])
    return REGEX_ANSI_ESCAPE.sub('', STRING_TRIMMED)

def FUNCTION_HANDLE_QEMU_ERROR(LIST_CMDLINE):
    OBJECT_ERROR = subprocess.run(LIST_CMDLINE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print(OBJECT_ERROR.stderr.strip())

def FUNCTION_GET_CHANGED_REGISTER_INFO(OBJECT_IO, LIST_CHANGED_REGISTER):
    STRING_REGISTER = ' '.join(LIST_CHANGED_REGISTER)
    OBJECT_IO.sendline(f'info register {STRING_REGISTER}'.encode())
    STRING_OUTPUT = OBJECT_IO.clean().decode()
    LIST_LINE = STRING_OUTPUT.splitlines()
    if LIST_LINE and '(gdb)' in LIST_LINE[-1]:
        LIST_LINE.pop()
    return '\n'.join(LIST_LINE).strip()

def FUNCTION_TEST_TEST_CASE(OBJECT_IO):
    MAP_REGISTER_OLD = FUNCTION_PARSE_REGISTER(FILE_REGISTER_DEFAULT)
    MAP_REGISTER_NEW = FUNCTION_PARSE_REGISTER(FILE_REGISTER_CHANGE)
    if ONLY_SHOW_CHANGED_REGISTER:
        LIST_CHANGED_REGISTER = []
        SET_REGISTER_EXCLUDED = {'sp', 'fp', 'pc', 'mcycle', 'minstret', 'cycle', 'time', 'instret'}
        for EACH_REGISTER, EACH_VALUE_NEW in MAP_REGISTER_NEW.items():
            VALUE_OLD = MAP_REGISTER_OLD.get(EACH_REGISTER)
            if VALUE_OLD and EACH_REGISTER not in SET_REGISTER_EXCLUDED and EACH_VALUE_NEW != VALUE_OLD:
                LIST_CHANGED_REGISTER.append(EACH_REGISTER)
    else:
        LIST_CHANGED_REGISTER = LIST_REGISTER_CHOSEN
    if not LIST_CHANGED_REGISTER:
        STRING_OUTPUT_CLEAN = 'No registers changed!'
    else:
        STRING_OUTPUT_CLEAN = FUNCTION_GET_CHANGED_REGISTER_INFO(OBJECT_IO, LIST_CHANGED_REGISTER)
    print(STRING_OUTPUT_CLEAN)

def FUNCTION_EDIT_MAIN_S(STRING_CONTENT):
    with open(FILE_BOOTLOADER_CODE, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_CONTENT)

def FUNCTION_TIDY_OUTPUT_FILE(FILE_NAME):
    with open(FILE_NAME, 'r', encoding='utf-8') as FILE_HANDLE:
        LIST_LINE = FILE_HANDLE.readlines()
    INDEX_START = 0
    while INDEX_START < len(LIST_LINE) and not (LIST_LINE[INDEX_START].startswith('============') or 'START ITER' in LIST_LINE[INDEX_START]):
        INDEX_START += 1
    LIST_KEEP = LIST_LINE[INDEX_START:]
    LIST_COLLAPSED = []
    VALUE_BLANK_STREAK = 0
    for EACH_LINE in LIST_KEEP:
        if EACH_LINE.strip() == '':
            VALUE_BLANK_STREAK += 1
            if VALUE_BLANK_STREAK <= 1:
                LIST_COLLAPSED.append(EACH_LINE)
        else:
            VALUE_BLANK_STREAK = 0
            LIST_COLLAPSED.append(EACH_LINE)
    while LIST_COLLAPSED and LIST_COLLAPSED[-1].strip() == '':
        LIST_COLLAPSED.pop()
    if LIST_COLLAPSED and LIST_COLLAPSED[-1].endswith('\n'):
        LIST_COLLAPSED[-1] = LIST_COLLAPSED[-1].rstrip('\n')
    with open(FILE_NAME, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.writelines(LIST_COLLAPSED)

def FUNCTION_RUN_QEMU(FUNCTION_CALLBACK, FILE_REGISTER, MODE_VALUE):
    LIST_CMDLINE = ['./run', str(MODE_VALUE)]
    OBJECT_QEMU_IO = process(LIST_CMDLINE)
    try:
        OBJECT_QEMU_IO.recvuntil(b'*** Now run')
        OBJECT_GDB_IO = process('riscv64-unknown-elf-gdb')
        OBJECT_GDB_IO.sendline(b'b *test_end')
        OBJECT_GDB_IO.sendline(b'b *trap_vector')
        OBJECT_GDB_IO.sendline(b'c')
        STRING_BREAK = OBJECT_GDB_IO.recvuntil(b'Breakpoint 1, ', timeout=TIMEOUT_SECOND).decode()
        if not STRING_BREAK:
            OBJECT_GDB_IO.recvuntil(b'Breakpoint 2, ', timeout=TIMEOUT_SECOND).decode()
        OBJECT_GDB_IO.sendline(b'info all-registers')
        STRING_OUTPUT = OBJECT_GDB_IO.clean().decode()
        STRING_REGISTER_CLEANED = FUNCTION_CLEAN_REGISTER_OUTPUT(STRING_OUTPUT)
        with open(FILE_REGISTER, 'w', encoding='utf-8') as FILE_HANDLE:
            FILE_HANDLE.write(STRING_REGISTER_CLEANED)
        if FUNCTION_CALLBACK:
            FUNCTION_CALLBACK(OBJECT_GDB_IO)
        if not STRING_OUTPUT:
            print('ERROR OCCURRED WHILE EXECUTING TEST CASE!')
        OBJECT_GDB_IO.kill()
    except EOFError:
        FUNCTION_HANDLE_QEMU_ERROR(LIST_CMDLINE)
    finally:
        OBJECT_QEMU_IO.kill()
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODE_DEFAULT = 3
SKIPPING_LINE_COUNT = 3
TIMEOUT_SECOND = 1
STRING_DEFAULT_TEST_CASE = '''li x0, 0
li x1, 0
li x2, 0
li x3, 0
li x4, 0
li x5, 0
li x6, 0
li x7, 0
li x8, 0
li x9, 0
li x10, 0
li x11, 0
li x12, 0
li x13, 0
li x14, 0
li x15, 0
li x16, 0
li x17, 0
li x18, 0
li x19, 0
li x20, 0
li x21, 0
li x22, 0
li x23, 0
li x24, 0
li x25, 0
li x26, 0
li x27, 0
li x28, 0
li x29, 0
li x30, 0
li x31, 0
fmv.d.x f0, x0
fmv.d.x f1, x0
fmv.d.x f2, x0
fmv.d.x f3, x0
fmv.d.x f4, x0
fmv.d.x f5, x0
fmv.d.x f6, x0
fmv.d.x f7, x0
fmv.d.x f8, x0
fmv.d.x f9, x0
fmv.d.x f10, x0
fmv.d.x f11, x0
fmv.d.x f12, x0
fmv.d.x f13, x0
fmv.d.x f14, x0
fmv.d.x f15, x0
fmv.d.x f16, x0
fmv.d.x f17, x0
fmv.d.x f18, x0
fmv.d.x f19, x0
fmv.d.x f20, x0
fmv.d.x f21, x0
fmv.d.x f22, x0
fmv.d.x f23, x0
fmv.d.x f24, x0
fmv.d.x f25, x0
fmv.d.x f26, x0
fmv.d.x f27, x0
fmv.d.x f28, x0
fmv.d.x f29, x0
fmv.d.x f30, x0
fmv.d.x f31, x0
vsetvli x0, x0, e32, m1, ta, ma
vmv.v.i v0, 0
vmv.v.i v1, 0
vmv.v.i v2, 0
vmv.v.i v3, 0
vmv.v.i v4, 0
vmv.v.i v5, 0
vmv.v.i v6, 0
vmv.v.i v7, 0
vmv.v.i v8, 0
vmv.v.i v9, 0
vmv.v.i v10, 0
vmv.v.i v11, 0
vmv.v.i v12, 0
vmv.v.i v13, 0
vmv.v.i v14, 0
vmv.v.i v15, 0
vmv.v.i v16, 0
vmv.v.i v17, 0
vmv.v.i v18, 0
vmv.v.i v19, 0
vmv.v.i v20, 0
vmv.v.i v21, 0
vmv.v.i v22, 0
vmv.v.i v23, 0
vmv.v.i v24, 0
vmv.v.i v25, 0
vmv.v.i v26, 0
vmv.v.i v27, 0
vmv.v.i v28, 0
vmv.v.i v29, 0
vmv.v.i v30, 0
vmv.v.i v31, 0

'''
REGEX_ANSI_ESCAPE = re.compile(r'\x1B\[[0-?]*[ -/]*[@-~]')
ONLY_SHOW_CHANGED_REGISTER = False

# IMPORTANT - BEGIN
LIST_REGISTER_CHOSEN = ['priv', 'pc', 'a0', 'a1', 'a2', 'a3', 't0', 't1', 't2', 't3', 'f0', 'f1', 'f2', 'f3', 'ft0', 'ft1', 'ft2', 'ft3', 'v0', 'v1', 'v2', 'v3', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x19', 'x20', 'x21', 'x22', 'x23', 'x24', 'x25', 'x26', 'x27', 'x28', 'x29', 'x30', 'x31', 'ra', 'sp', 'gp', 'tp', 'mstatus', 'mcause']
IS_OUTPUT = False
# IMPORTANT - FINISH

STRING_CODE_TEMPLATE = '''
.globl test_start

test_start:
{test_case}

test_end:
j test_end

.global trap_vector
.align 4

trap_vector:
j trap_vector
'''
# VARIABLE - FINISH

# CODE - BEGIN
if IS_OUTPUT:
    class CLASS_STDOUT_TO_FILE:
        def __init__(self, FILE_NAME):
            self.file = open(FILE_NAME, 'w', encoding='utf-8')
            self.stdout = sys.__stdout__
        def write(self, STRING_MESSAGE):
            self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
            self.stdout.write(STRING_MESSAGE)
        def flush(self):
            self.file.flush()
            self.stdout.flush()
        def close(self):
            self.file.close()
else:
    class CLASS_STDOUT_TO_FILE:
        def __init__(self, FILE_NAME):
            self.file = open(FILE_NAME, 'w', encoding='utf-8')
        def write(self, STRING_MESSAGE):
            self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
        def flush(self):
            self.file.flush()
        def close(self):
            self.file.close()

OBJECT_STDOUT_FILE = CLASS_STDOUT_TO_FILE(FILE_OUTPUT)
OBJECT_STDOUT_OLD = sys.stdout
sys.stdout = OBJECT_STDOUT_FILE

with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
    STRING_TEST_CASE = FILE_HANDLE.read()

LIST_BLOCK = [EACH_BLOCK.strip().split('\n') for EACH_BLOCK in STRING_TEST_CASE.strip().split('\n\n') if EACH_BLOCK.strip()]
LIST_TEST_CASE = [[EACH_LINE.strip() for EACH_LINE in EACH_BLOCK if EACH_LINE.strip()] for EACH_BLOCK in LIST_BLOCK]
STRING_LIST_TEST_CASE = f'LIST_TEST_CASE = {repr(LIST_TEST_CASE)}'

exec(STRING_LIST_TEST_CASE)

LIST_DEFAULT_TEST_CASE = [STRING_CODE_TEMPLATE.format(test_case=STRING_DEFAULT_TEST_CASE)]
LIST_CHANGED_TEST_CASE = [STRING_CODE_TEMPLATE.format(test_case='\n'.join(STRING_DEFAULT_TEST_CASE.splitlines() + EACH_TEST_CASE) if EACH_TEST_CASE else STRING_DEFAULT_TEST_CASE) for EACH_TEST_CASE in LIST_TEST_CASE]

try:
    FUNCTION_EDIT_MAIN_S(LIST_DEFAULT_TEST_CASE[0])
    FUNCTION_RUN_QEMU(None, FILE_REGISTER_DEFAULT, MODE_DEFAULT)
    for EACH_INDEX, (EACH_CONTENT_NEW, EACH_TEST_CASE) in enumerate(zip(LIST_CHANGED_TEST_CASE, LIST_TEST_CASE)):
        FUNCTION_EDIT_MAIN_S(EACH_CONTENT_NEW.strip())
        print(f'============\nSTART ITER {EACH_INDEX + 1}\n')
        print('\n'.join(EACH_TEST_CASE) + '\n')
        MODE_VALUE = MODE_DEFAULT
        FUNCTION_RUN_QEMU(FUNCTION_TEST_TEST_CASE, FILE_REGISTER_CHANGE, MODE_VALUE)
        print(f'\nEND ITER {EACH_INDEX + 1}\n==========\n')
finally:
    sys.stdout = OBJECT_STDOUT_OLD
    OBJECT_STDOUT_FILE.close()
    FUNCTION_TIDY_OUTPUT_FILE(FILE_OUTPUT)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 2_QEMU_XV6_RUN.txt - END

In [ ]:
# 2 - CREATE - 3_QEMU_XV6_CHECK.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import operator, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = 'TEXT/2_QEMU_XV6_RUN.txt'
FILE_OUTPUT = 'TEXT/3_QEMU_XV6_CHECK.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_CANONICAL_REGISTER(STRING_NAME: str) -> str:
    STRING_KEY = STRING_NAME.strip()
    MATCH_V = re.fullmatch(r'(v\d+)\[(\d+)\]', STRING_KEY)
    if MATCH_V:
        return f'{MATCH_V.group(1)}.w[{MATCH_V.group(2)}]'
    MATCH_V2 = re.fullmatch(r'(v\d+)\.(b|s|w|l|q)\[(\d+)\]', STRING_KEY)
    if MATCH_V2:
        return f'{MATCH_V2.group(1)}.{MATCH_V2.group(2)}[{MATCH_V2.group(3)}]'
    return MAP_REGISTER_ALIAS.get(STRING_KEY, STRING_KEY)

def FUNCTION_PARSE_VECTOR_LIST(STRING_BODY: str, TOTAL_LEN: int):
    LIST_OUT = []
    TOKENS = [x.strip() for x in STRING_BODY.split(',') if x.strip()]
    for TOK in TOKENS:
        MREP = re.fullmatch(r'(0x[0-9a-fA-F]+)\s*<repeats\s+(\d+)\s+times>', TOK)
        if MREP:
            VAL = int(MREP.group(1), 16)
            CNT = int(MREP.group(2))
            LIST_OUT.extend([VAL] * CNT)
            continue
        MHEX = re.fullmatch(r'0x[0-9a-fA-F]+', TOK)
        if MHEX:
            LIST_OUT.append(int(TOK, 16))
            continue
        continue
    if len(LIST_OUT) < TOTAL_LEN:
        LIST_OUT.extend([0] * (TOTAL_LEN - len(LIST_OUT)))
    return LIST_OUT[:TOTAL_LEN]

def FUNCTION_PARSE_INT(STRING_VALUE: str) -> int:
    STRING_LOWER = STRING_VALUE.strip().lower()
    if STRING_LOWER.startswith(('+0x', '-0x', '0x')):
        return int(STRING_VALUE, 16)
    return int(STRING_VALUE, 10)

def FUNCTION_PARSE_ITERATION(STRING_CONTENT):
    REGEX_ITERATION = re.compile(r'START ITER (\d+)(.*?)END ITER \1', re.DOTALL)
    LIST_ITERATION = REGEX_ITERATION.findall(STRING_CONTENT)
    return LIST_ITERATION

def FUNCTION_HAS_ERROR(STRING_ITERATION: str) -> bool:
    return re.search(r'\bpriv\b', STRING_ITERATION) is None

def FUNCTION_PARSE_EXPECTED_RESULT(STRING_ITERATION: str):
    STRING_VALUE_PATTERN = r'[+-]?(?:0x[0-9a-fA-F]+|\d+)'
    STRING_REGISTER_PATTERN = r'(?:[A-Za-z_]\w+|v\d+(?:\.(?:b|s|w|l|q))?\[\d+\])'
    STRING_RHS_PATTERN = rf'(?:{STRING_VALUE_PATTERN}|{STRING_REGISTER_PATTERN})'
    REGEX_RESULT_LINE = re.compile(r'#\s*Result:\s*(.*)')
    REGEX_CONDITION = re.compile(
        rf'\s*({STRING_REGISTER_PATTERN})\s*(==|=|!=|<=|>=|<|>)\s*({STRING_RHS_PATTERN})\s*'
    )
    LIST_OR_GROUPS = []
    for EACH_LINE in STRING_ITERATION.splitlines():
        MATCH_RESULT = REGEX_RESULT_LINE.search(EACH_LINE)
        if not MATCH_RESULT:
            continue
        STRING_TAIL = MATCH_RESULT.group(1)
        LIST_OR_PART = [x.strip() for x in STRING_TAIL.split('|') if x.strip()]
        for EACH_OR in LIST_OR_PART:
            LIST_AND = []
            LIST_AND_PART = [x.strip() for x in EACH_OR.split('&') if x.strip()]
            for EACH_PART in LIST_AND_PART:
                MATCH_COND = REGEX_CONDITION.fullmatch(EACH_PART)
                if not MATCH_COND:
                    continue
                STRING_REG, STRING_OP, STRING_RHS_TOKEN = MATCH_COND.groups()
                STRING_REG = FUNCTION_CANONICAL_REGISTER(STRING_REG)
                if STRING_OP == '=':
                    STRING_OP = '=='
                LIST_AND.append((STRING_REG, STRING_OP, STRING_RHS_TOKEN))
            if LIST_AND:
                LIST_OR_GROUPS.append(LIST_AND)
    return LIST_OR_GROUPS

def FUNCTION_PARSE_ACTUAL_REGISTER(STRING_ITERATION: str):
    STRING_VALUE_PATTERN = r'[+-]?(?:0x[0-9a-fA-F]+|\d+)'
    REGEX_GPR = re.compile(rf'(\w+)\s+0x[0-9a-fA-F]+\s+({STRING_VALUE_PATTERN})')
    REGEX_FP_RAW = re.compile(
        r'^(f(?:t|s|a)?\d+)\s+[\s\S]*?\(raw\s+0x([0-9a-fA-F]+)\)',
        re.MULTILINE
    )
    REGEX_F_DOUBLE = re.compile(
        r'^(f\d+)\s+\{float\s*=\s*0x[0-9a-fA-F]+,\s*double\s*=\s*0x([0-9a-fA-F]+)\}',
        re.MULTILINE
    )
    MAP_ACTUAL = {}
    for EACH_REG, EACH_VALUE in REGEX_GPR.findall(STRING_ITERATION):
        try:
            VALUE_PARSED = FUNCTION_PARSE_INT(EACH_VALUE)
        except ValueError:
            continue
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    for EACH_REG, EACH_HEX in REGEX_FP_RAW.findall(STRING_ITERATION):
        VALUE_PARSED = int(EACH_HEX, 16)
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    for EACH_REG, EACH_HEX_DOUBLE in REGEX_F_DOUBLE.findall(STRING_ITERATION):
        VALUE_PARSED = int(EACH_HEX_DOUBLE, 16)
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    REGEX_VBLOCK = re.compile(r'^(v\d+)\s+(.*?)(?=^v\d+\s+|\Z)', re.MULTILINE | re.DOTALL)
    REGEX_VIEW = re.compile(r'\b([qlwsb])\s*=\s*\{(.*?)\}', re.DOTALL)
    MAP_LEN = {'b': 16, 's': 8, 'w': 4, 'l': 2, 'q': 1}
    MAP_MASK = {'b': (1 << 8) - 1, 's': (1 << 16) - 1, 'w': (1 << 32) - 1, 'l': (1 << 64) - 1, 'q': MASK64}
    for VREG, VBLOCK in REGEX_VBLOCK.findall(STRING_ITERATION):
        VREG = VREG.strip()
        for VIEW, BODY in REGEX_VIEW.findall(VBLOCK):
            VIEW = VIEW.strip()
            if VIEW not in MAP_LEN:
                continue
            LIST_VAL = FUNCTION_PARSE_VECTOR_LIST(BODY, MAP_LEN[VIEW])
            for IDX, VAL in enumerate(LIST_VAL):
                KEY = f'{VREG}.{VIEW}[{IDX}]'
                MAP_ACTUAL[KEY] = VAL & MAP_MASK[VIEW]
    return MAP_ACTUAL

def FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY: str):
    M = re.fullmatch(r'v\d+\.(b|s|w|l|q)\[\d+\]', STRING_REG_KEY)
    if not M:
        return None
    return {'b': 8, 's': 16, 'w': 32, 'l': 64, 'q': 64}[M.group(1)]

def FUNCTION_HAS_RESULT_ONLY_ERROR(STRING_ITERATION: str) -> bool:
    return REGEX_RESULT_ONLY_ERROR.search(STRING_ITERATION) is not None

def FUNCTION_U64(VALUE_X: int) -> int:
    return VALUE_X & MASK64

def FUNCTION_CHECK_ITERATION(LIST_ITERATION):
    STRING_OUTPUT = ''
    for EACH_ITER_NUMBER, EACH_ITER_CONTENT in LIST_ITERATION:
        VALUE_ITER_NUMBER = int(EACH_ITER_NUMBER)
        if FUNCTION_HAS_RESULT_ONLY_ERROR(EACH_ITER_CONTENT):
            if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
                continue
            STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
            continue
        if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
            STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is error\n'
            continue
        LIST_EXPECTED_GROUP = FUNCTION_PARSE_EXPECTED_RESULT(EACH_ITER_CONTENT)
        MAP_ACTUAL = FUNCTION_PARSE_ACTUAL_REGISTER(EACH_ITER_CONTENT)
        FLAG_MATCH = False
        for EACH_GROUP in LIST_EXPECTED_GROUP:
            GROUP_OK = True
            for EACH_REG, EACH_OP, EACH_RHS_TOKEN in EACH_GROUP:
                STRING_REG_KEY = FUNCTION_CANONICAL_REGISTER(EACH_REG)
                VALUE_ACTUAL = MAP_ACTUAL.get(STRING_REG_KEY)
                if VALUE_ACTUAL is None:
                    GROUP_OK = False
                    break
                try:
                    VALUE_EXPECTED = FUNCTION_PARSE_INT(EACH_RHS_TOKEN)
                except ValueError:
                    STRING_RHS_KEY = FUNCTION_CANONICAL_REGISTER(EACH_RHS_TOKEN)
                    VALUE_RHS = MAP_ACTUAL.get(STRING_RHS_KEY)
                    if VALUE_RHS is None:
                        GROUP_OK = False
                        break
                    VALUE_EXPECTED = VALUE_RHS
                if EACH_OP in ('==', '!='):
                    WIDTH_BITS = None
                
                    # NEW: kalau RHS literal hex 32/64-bit, pakai itu untuk semua register (GPR/FPR juga boleh)
                    WIDTH_BITS = FUNCTION_HEX_WIDTH_BITS(EACH_RHS_TOKEN)
                
                    # vector view tetap override
                    V_BITS = FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY)
                    if V_BITS is not None:
                        WIDTH_BITS = V_BITS
                
                    if WIDTH_BITS == 8:
                        VALUE_A = VALUE_ACTUAL & ((1 << 8) - 1)
                        VALUE_E = VALUE_EXPECTED & ((1 << 8) - 1)
                    elif WIDTH_BITS == 16:
                        VALUE_A = VALUE_ACTUAL & ((1 << 16) - 1)
                        VALUE_E = VALUE_EXPECTED & ((1 << 16) - 1)
                    elif WIDTH_BITS == 32:
                        VALUE_A = VALUE_ACTUAL & MASK32
                        VALUE_E = VALUE_EXPECTED & MASK32
                    else:
                        VALUE_A = FUNCTION_U64(VALUE_ACTUAL)
                        VALUE_E = FUNCTION_U64(VALUE_EXPECTED)
                
                    if not MAP_OPERATOR[EACH_OP](VALUE_A, VALUE_E):
                        GROUP_OK = False
                        break
            if GROUP_OK:
                FLAG_MATCH = True
                break
        if not FLAG_MATCH:
            STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
    STRING_OUTPUT_FINAL = STRING_OUTPUT.strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_OUTPUT_FINAL)

def FUNCTION_IS_FLOAT_REG(STRING_REG_CANON: str) -> bool:
    return re.fullmatch(r'f(?:[0-9]|[12][0-9]|3[01])', STRING_REG_CANON) is not None

def FUNCTION_HEX_WIDTH_BITS(STRING_TOKEN: str):
    S = STRING_TOKEN.strip().lower()
    if S.startswith(('+0x', '-0x')):
        S = S[1:]
    if not S.startswith('0x'):
        return None
    HEXDIG = S[2:]
    if 1 <= len(HEXDIG) <= 8:
        return 32
    if 9 <= len(HEXDIG) <= 16:
        return 64
    return None
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_RESULT_ONLY_ERROR = re.compile(r'^\s*#\s*Result\s*:\s*error\s*$', re.IGNORECASE | re.MULTILINE)
MAP_OPERATOR = {'=': operator.eq, '==': operator.eq, '!=': operator.ne, '<': operator.lt, '<=': operator.le, '>': operator.gt, '>=': operator.ge}
MAP_REGISTER_ALIAS = {'x0': 'x0', 'zero': 'x0', 'x1': 'x1', 'ra': 'x1', 'x2': 'x2', 'sp': 'x2', 'x3': 'x3', 'gp': 'x3', 'x4': 'x4', 'tp': 'x4', 'x5': 'x5', 't0': 'x5', 'x6': 'x6', 't1': 'x6', 'x7': 'x7', 't2': 'x7', 'x8': 'x8', 's0': 'x8', 'fp': 'x8', 'x9': 'x9', 's1': 'x9', 'x10': 'x10', 'a0': 'x10', 'x11': 'x11', 'a1': 'x11', 'x12': 'x12', 'a2': 'x12', 'x13': 'x13', 'a3': 'x13', 'x14': 'x14', 'a4': 'x14', 'x15': 'x15', 'a5': 'x15', 'x16': 'x16', 'a6': 'x16', 'x17': 'x17', 'a7': 'x17', 'x18': 'x18', 's2': 'x18', 'x19': 'x19', 's3': 'x19', 'x20': 'x20', 's4': 'x20', 'x21': 'x21', 's5': 'x21', 'x22': 'x22', 's6': 'x22', 'x23': 'x23', 's7': 'x23', 'x24': 'x24', 's8': 'x24', 'x25': 'x25', 's9': 'x25', 'x26': 'x26', 's10': 'x26', 'x27': 'x27', 's11': 'x27', 'x28': 'x28', 't3': 'x28', 'x29': 'x29', 't4': 'x29', 'x30': 'x30', 't5': 'x30', 'x31': 'x31', 't6': 'x31'}
MASK64 = (1 << 64) - 1
MASK32 = (1 << 32) - 1
MAP_REGISTER_ALIAS.update({**{f'f{i}': f'f{i}' for i in range(32)}, 'ft0': 'f0', 'ft1': 'f1', 'ft2': 'f2', 'ft3': 'f3', 'ft4': 'f4', 'ft5': 'f5', 'ft6': 'f6', 'ft7': 'f7', 'ft8': 'f28', 'ft9': 'f29', 'ft10': 'f30', 'ft11': 'f31', 'fs0': 'f8', 'fs1': 'f9', 'fs2': 'f18', 'fs3': 'f19', 'fs4': 'f20', 'fs5': 'f21', 'fs6': 'f22', 'fs7': 'f23', 'fs8': 'f24', 'fs9': 'f25', 'fs10': 'f26', 'fs11': 'f27', 'fa0': 'f10', 'fa1': 'f11', 'fa2': 'f12', 'fa3': 'f13', 'fa4': 'f14', 'fa5': 'f15', 'fa6': 'f16', 'fa7': 'f17'})
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
    STRING_CONTENT = FILE_HANDLE.read()

LIST_ITERATION = FUNCTION_PARSE_ITERATION(STRING_CONTENT)
FUNCTION_CHECK_ITERATION(LIST_ITERATION)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# ALARM PLAY - BEGIN
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
import pygame
import time
pygame.mixer.init()
pygame.mixer.music.load('alarm.mp3')
for _ in range(3):
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.01)
pygame.mixer.quit()
# ALARM PLAY - FINISH

# 2 - CREATE - 3_QEMU_XV6_CHECK.txt - END